## First try to reproduce pipeline in Notebook

In [61]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

from evaluate import run_evaluation, print_summary

In [62]:
df_features = pd.read_parquet("./artifacts/features_dataset.parquet")

feature_cols = [c for c in df_features.columns if c.startswith("f_")]

X = df_features[feature_cols].to_numpy(dtype=np.float32)
y = df_features["label"].astype(int).to_numpy()

X.shape, y.shape, y.mean()

((689, 896), (689,), np.float64(0.7010159651669086))

In [63]:
def split_data_notebook(y, n_splits=5, val_size=0.15, random_state=42):
    y = np.asarray(y)
    idx = np.arange(len(y))

    outer_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    splits = []

    for fold_id, (idx_train_val, idx_test) in enumerate(outer_cv.split(idx, y)):
        inner_cv = StratifiedShuffleSplit(
            n_splits=1,
            test_size=val_size,
            random_state=random_state + fold_id,
        )

        inner_train_pos, inner_val_pos = next(
            inner_cv.split(idx_train_val, y[idx_train_val])
        )

        idx_train = idx_train_val[inner_train_pos]
        idx_val = idx_train_val[inner_val_pos]

        splits.append((idx_train, idx_val, idx_test))

    return splits


splits = split_data_notebook(y, n_splits=5, val_size=0.15, random_state=42)

for i, (tr, va, te) in enumerate(splits, start=1):
    print(
        f"Fold {i}: train={len(tr)}, val={len(va)}, test={len(te)}, "
        f"test_pos_rate={y[te].mean():.3f}"
    )

Fold 1: train=468, val=83, test=138, test_pos_rate=0.703
Fold 2: train=468, val=83, test=138, test_pos_rate=0.703
Fold 3: train=468, val=83, test=138, test_pos_rate=0.703
Fold 4: train=468, val=83, test=138, test_pos_rate=0.696
Fold 5: train=469, val=83, test=137, test_pos_rate=0.701


In [64]:
class HallucinationProbe(BaseEstimator, ClassifierMixin):
    def __init__(self, C=1.0, max_iter=5000, class_weight="balanced"):
        self.C = C
        self.max_iter = max_iter
        self.class_weight = class_weight
        self.threshold_ = 0.5
        self.model_ = None

    def fit(self, X, y):
        self.model_ = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=self.C,
                max_iter=self.max_iter,
                class_weight=self.class_weight,
                solver="lbfgs",
            ),
        )
        self.model_.fit(X, y)
        return self

    def fit_hyperparameters(self, X_val, y_val):
        probs = self.predict_proba(X_val)[:, 1]

        candidates = np.unique(
            np.concatenate([probs, np.linspace(0.05, 0.95, 181)])
        )

        best_threshold = 0.5
        best_f1 = -1.0

        for threshold in candidates:
            preds = (probs >= threshold).astype(int)
            score = f1_score(y_val, preds, zero_division=0)

            if score > best_f1:
                best_f1 = score
                best_threshold = float(threshold)

        self.threshold_ = best_threshold
        return self

    def predict(self, X):
        probs = self.predict_proba(X)[:, 1]
        return (probs >= self.threshold_).astype(int)

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

In [65]:
fold_results = run_evaluation(
    splits=splits,
    X=X,
    y=y,
    ProbeClass=HallucinationProbe,
)

print_summary(
    fold_results=fold_results,
    feature_dim=X.shape[1],
    n_samples=len(X),
    extract_time=0.0,
)


──────────────────────────────────────────────────
  Fold 1/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 79.49%  F1: 87.20%  AUROC: 97.18%
  Probe val  — Acc: 72.29%  F1: 82.71%  AUROC: 68.83%
  Probe test — Acc: 66.67%  F1: 79.28%  AUROC: 66.38%

──────────────────────────────────────────────────
  Fold 2/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 70.51%  F1: 82.62%  AUROC: 97.58%
  Probe val  — Acc: 71.08%  F1: 82.86%  AUROC: 67.45%
  Probe test — Acc: 71.74%  F1: 83.26%  AUROC: 76.14%

──────────────────────────────────────────────────
  Fold 3/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 70.30%  F1: 82.52%  AUROC: 97.59%
  Probe val  — Acc: 69.88%  F1: 82.27%  AUROC: 56.97%
  Probe test 

In [66]:
results = []

for C in [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]:
    class ProbeWithC(HallucinationProbe):
        def __init__(self):
            super().__init__(C=C)

    fold_results = run_evaluation(
        splits=splits,
        X=X,
        y=y,
        ProbeClass=ProbeWithC,
    )

    row = {
        "C": C,
        "test_auroc_mean": np.mean([r["test_auroc"] for r in fold_results]),
        "test_auroc_std": np.std([r["test_auroc"] for r in fold_results]),
        "test_f1_mean": np.mean([r["test_f1"] for r in fold_results]),
        "test_accuracy_mean": np.mean([r["test_accuracy"] for r in fold_results]),
        "val_auroc_mean": np.mean([r["val_auroc"] for r in fold_results]),
    }

    results.append(row)

results_df = pd.DataFrame(results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(results_df)

best = results_df.iloc[0]

print(
    f"Best C = {best['C']}\n"
    f"Test AUROC = {best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}\n"
    f"Test F1 = {best['test_f1_mean']:.4f}\n"
    f"Test Accuracy = {best['test_accuracy_mean']:.4f}"
)


──────────────────────────────────────────────────
  Fold 1/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 73.08%  F1: 83.46%  AUROC: 76.67%
  Probe val  — Acc: 72.29%  F1: 82.96%  AUROC: 72.62%
  Probe test — Acc: 68.84%  F1: 81.06%  AUROC: 71.81%

──────────────────────────────────────────────────
  Fold 2/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 71.37%  F1: 83.04%  AUROC: 75.87%
  Probe val  — Acc: 72.29%  F1: 83.45%  AUROC: 65.93%
  Probe test — Acc: 70.29%  F1: 82.55%  AUROC: 69.60%

──────────────────────────────────────────────────
  Fold 3/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 74.15%  F1: 83.40%  AUROC: 77.31%
  Probe val  — Acc: 74.70%  F1: 83.20%  AUROC: 63.93%
  Probe test 

KeyboardInterrupt: 

## Checkinh rich features

For this task created ``diagnostic_hidden_state_analysis.py`` - after getting info from this file trying to evaluate results here in Notebook

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score

from evaluate import run_evaluation, print_summary

In [ ]:
df = pd.read_parquet(
    "./artifacts/selected_rich_features/features_dataset_selected_rich_quantiles.parquet"
)

drop_cols = {"label", "prompt", "response", "source_index"}
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].to_numpy(dtype=np.float32)
y = df["label"].astype(int).to_numpy()

print("X:", X.shape)
print("y:", y.shape)
print("positive rate:", y.mean())

X: (689, 60566)
y: (689,)
positive rate: 0.7010159651669086


In [ ]:
def make_stratified_kfold_splits(y, n_splits=5, val_size=0.15, random_state=42):
    y = np.asarray(y)
    idx = np.arange(len(y))

    outer_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    splits = []

    for fold_id, (idx_train_val, idx_test) in enumerate(outer_cv.split(idx, y)):
        inner_cv = StratifiedShuffleSplit(
            n_splits=1,
            test_size=val_size,
            random_state=random_state + fold_id,
        )

        inner_train_pos, inner_val_pos = next(
            inner_cv.split(idx_train_val, y[idx_train_val])
        )

        idx_train = idx_train_val[inner_train_pos]
        idx_val = idx_train_val[inner_val_pos]

        splits.append((idx_train, idx_val, idx_test))

    return splits


splits = make_stratified_kfold_splits(
    y,
    n_splits=5,
    val_size=0.15,
    random_state=42,
)

for i, (tr, va, te) in enumerate(splits, start=1):
    print(
        f"Fold {i}: train={len(tr)}, val={len(va)}, test={len(te)}, "
        f"train_pos={y[tr].mean():.3f}, val_pos={y[va].mean():.3f}, test_pos={y[te].mean():.3f}"
    )

Fold 1: train=468, val=83, test=138, train_pos=0.701, val_pos=0.699, test_pos=0.703
Fold 2: train=468, val=83, test=138, train_pos=0.701, val_pos=0.699, test_pos=0.703
Fold 3: train=468, val=83, test=138, train_pos=0.701, val_pos=0.699, test_pos=0.703
Fold 4: train=468, val=83, test=138, train_pos=0.703, val_pos=0.699, test_pos=0.696
Fold 5: train=469, val=83, test=137, train_pos=0.701, val_pos=0.699, test_pos=0.701


In [ ]:
class ThresholdMixin:
    def fit_hyperparameters(self, X_val, y_val):
        probs = self.predict_proba(X_val)[:, 1]

        candidates = np.unique(
            np.concatenate([probs, np.linspace(0.05, 0.95, 181)])
        )

        best_threshold = 0.5
        best_f1 = -1.0

        for threshold in candidates:
            preds = (probs >= threshold).astype(int)
            score = f1_score(y_val, preds, zero_division=0)

            if score > best_f1:
                best_f1 = score
                best_threshold = float(threshold)

        self.threshold_ = best_threshold
        return self

    def predict(self, X):
        probs = self.predict_proba(X)[:, 1]
        return (probs >= self.threshold_).astype(int)


class LogisticPCAProbe(ThresholdMixin, BaseEstimator, ClassifierMixin):
    def __init__(self, n_components=128, C=0.3):
        self.n_components = n_components
        self.C = C
        self.threshold_ = 0.5
        self.model_ = None

    def fit(self, X, y):
        n_components = min(self.n_components, X.shape[0] - 1, X.shape[1])

        self.model_ = make_pipeline(
            StandardScaler(),
            PCA(n_components=n_components, random_state=42),
            LogisticRegression(
                C=self.C,
                class_weight="balanced",
                max_iter=5000,
                solver="lbfgs",
            ),
        )
        self.model_.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(X)


class LogisticNoPCAProbe(ThresholdMixin, BaseEstimator, ClassifierMixin):
    def __init__(self, C=0.03):
        self.C = C
        self.threshold_ = 0.5
        self.model_ = None

    def fit(self, X, y):
        self.model_ = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=self.C,
                class_weight="balanced",
                max_iter=5000,
                solver="lbfgs",
            ),
        )
        self.model_.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(X)


class HistGradientBoostingProbe(ThresholdMixin, BaseEstimator, ClassifierMixin):
    def __init__(self, learning_rate=0.03, max_iter=150, l2_regularization=1.0):
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.l2_regularization = l2_regularization
        self.threshold_ = 0.5
        self.model_ = None

    def fit(self, X, y):
        self.model_ = HistGradientBoostingClassifier(
            learning_rate=self.learning_rate,
            max_iter=self.max_iter,
            l2_regularization=self.l2_regularization,
            random_state=42,
        )
        self.model_.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(X)


class LinearSVMProbe(ThresholdMixin, BaseEstimator, ClassifierMixin):
    def __init__(self, alpha=0.0003):
        self.alpha = alpha
        self.threshold_ = 0.5
        self.model_ = None

    def fit(self, X, y):
        base = make_pipeline(
            StandardScaler(),
            SGDClassifier(
                loss="hinge",
                alpha=self.alpha,
                class_weight="balanced",
                max_iter=5000,
                random_state=42,
            ),
        )

        self.model_ = CalibratedClassifierCV(base, method="sigmoid", cv=3)
        self.model_.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

In [ ]:
def evaluate_probe_class(name, ProbeClass):
    fold_results = run_evaluation(
        splits=splits,
        X=X,
        y=y,
        ProbeClass=ProbeClass,
    )

    row = {
        "probe": name,
        "test_auroc_mean": np.mean([r["test_auroc"] for r in fold_results]),
        "test_auroc_std": np.std([r["test_auroc"] for r in fold_results]),
        "test_f1_mean": np.mean([r["test_f1"] for r in fold_results]),
        "test_f1_std": np.std([r["test_f1"] for r in fold_results]),
        "test_accuracy_mean": np.mean([r["test_accuracy"] for r in fold_results]),
        "test_accuracy_std": np.std([r["test_accuracy"] for r in fold_results]),
        "val_auroc_mean": np.mean([r["val_auroc"] for r in fold_results]),
        "train_auroc_mean": np.mean([r["train_auroc"] for r in fold_results]),
    }

    return row, fold_results

In [ ]:
import numpy as np
import pandas as pd

#OLD_PARQUET = "./artifacts/features_dataset.parquet"
#NEW_PARQUET = "./artifacts/selected_rich_features/features_dataset_selected_rich.parquet"

OLD_PARQUET = "./artifacts/selected_rich_features/features_dataset_selected_rich.parquet"
NEW_PARQUET = "./artifacts/selected_rich_features/features_dataset_selected_rich_quantiles.parquet"

DATASETS = {
    "old_baseline_aggregation": OLD_PARQUET,
    "new_selected_rich": NEW_PARQUET,
}


def load_xy(path):
    df = pd.read_parquet(path)
    drop_cols = {"label", "prompt", "response", "source_index", "id"}
    feature_cols = [c for c in df.columns if c not in drop_cols]

    X = df[feature_cols].to_numpy(dtype=np.float32)
    y = df["label"].astype(int).to_numpy()

    return X, y


comparison_rows = []
comparison_fold_results = {}

# Important: Create the splits once based on the 'y' values ​​from the old Parquet file.
# Then, use the same indices for the new Parquet file.
X_ref, y_ref = load_xy(OLD_PARQUET)

splits = make_stratified_kfold_splits(
    y_ref,
    n_splits=5,
    val_size=0.15,
    random_state=42,
)

probe_specs_fast = []

for n_components in [64, 128]:
    for C in [0.03, 0.3]:
        class Probe(LogisticPCAProbe):
            def __init__(self):
                super().__init__(n_components=n_components, C=C)

        probe_specs_fast.append((f"logreg_pca{n_components}_C{C}", Probe))

for C in [0.01, 0.03]:
    class Probe(LogisticNoPCAProbe):
        def __init__(self):
            super().__init__(C=C)

    probe_specs_fast.append((f"logreg_no_pca_C{C}", Probe))

for lr in [0.03]:
    for l2 in [1.0]:
        class Probe(HistGradientBoostingProbe):
            def __init__(self):
                super().__init__(
                    learning_rate=lr,
                    max_iter=120,
                    l2_regularization=l2,
                )

        probe_specs_fast.append((f"hgb_lr{lr}_l2{l2}", Probe))


for dataset_name, parquet_path in DATASETS.items():
    print(f"\n==============================")
    print(f"DATASET: {dataset_name}")
    print(f"PATH   : {parquet_path}")
    print(f"==============================")

    X, y = load_xy(parquet_path)

    assert np.array_equal(y, y_ref), "Labels differ between parquet files"

    print("X:", X.shape)
    print("Positive rate:", y.mean())

    for probe_name, ProbeClass in probe_specs_fast:
        print(f"Running {dataset_name} | {probe_name}...")

        fold_results = run_evaluation(
            splits=splits,
            X=X,
            y=y,
            ProbeClass=ProbeClass,
        )

        row = {
            "dataset": dataset_name,
            "probe": probe_name,
            "n_features": X.shape[1],
            "test_auroc_mean": np.mean([r["test_auroc"] for r in fold_results]),
            "test_auroc_std": np.std([r["test_auroc"] for r in fold_results]),
            "test_f1_mean": np.mean([r["test_f1"] for r in fold_results]),
            "test_f1_std": np.std([r["test_f1"] for r in fold_results]),
            "test_accuracy_mean": np.mean([r["test_accuracy"] for r in fold_results]),
            "test_accuracy_std": np.std([r["test_accuracy"] for r in fold_results]),
            "val_auroc_mean": np.mean([r["val_auroc"] for r in fold_results]),
            "train_auroc_mean": np.mean([r["train_auroc"] for r in fold_results]),
        }

        comparison_rows.append(row)
        comparison_fold_results[(dataset_name, probe_name)] = fold_results


comparison_df = pd.DataFrame(comparison_rows).sort_values(
    ["probe", "test_auroc_mean"],
    ascending=[True, False],
)

display(comparison_df)


DATASET: old_baseline_aggregation
PATH   : ./artifacts/selected_rich_features/features_dataset_selected_rich.parquet
X: (689, 20878)
Positive rate: 0.7010159651669086
Running old_baseline_aggregation | logreg_pca64_C0.03...

──────────────────────────────────────────────────
  Fold 1/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 79.70%  F1: 87.25%  AUROC: 95.07%
  Probe val  — Acc: 73.49%  F1: 83.33%  AUROC: 75.10%
  Probe test — Acc: 73.19%  F1: 83.11%  AUROC: 71.64%

──────────────────────────────────────────────────
  Fold 2/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 78.21%  F1: 86.51%  AUROC: 93.78%
  Probe val  — Acc: 74.70%  F1: 84.21%  AUROC: 77.03%
  Probe test — Acc: 73.19%  F1: 83.26%  AUROC: 77.14%

──────────────────────────────────────────────────
  Fold 3/5  —  train=468  val

,dataset,probe,n_features,test_auroc_mean,test_auroc_std,test_f1_mean,test_f1_std,test_accuracy_mean,test_accuracy_std,val_auroc_mean,train_auroc_mean
6,old_baseline_aggregation,hgb_lr0.03_l21.0,20878,0.783457,0.019631,0.843920,0.006562,0.759082,0.011358,0.765931,1.000000
13,new_selected_rich,hgb_lr0.03_l21.0,60566,0.769509,0.027389,0.841703,0.012030,0.751772,0.023586,0.771310,1.000000
11,new_selected_rich,logreg_no_pca_C0.01,60566,0.767317,0.022091,0.835807,0.015487,0.743097,0.017016,0.752000,1.000000
4,old_baseline_aggregation,logreg_no_pca_C0.01,20878,0.762384,0.020360,0.834274,0.007441,0.738739,0.015405,0.754069,1.000000
12,new_selected_rich,logreg_no_pca_C0.03,60566,0.767317,0.022091,0.835807,0.015487,0.743097,0.017016,0.752000,1.000000
5,old_baseline_aggregation,logreg_no_pca_C0.03,20878,0.762384,0.020360,0.834274,0.007441,0.738739,0.015405,0.754069,1.000000
9,new_selected_rich,logreg_pca128_C0.03,60566,0.750027,0.016872,0.832568,0.015251,0.731461,0.024642,0.735862,0.945877
2,old_baseline_aggregation,logreg_pca128_C0.03,20878,0.749493,0.018186,0.835380,0.015148,0.734381,0.026335,0.728000,0.946306
10,new_selected_rich,logreg_pca128_C0.3,60566,0.750027,0.016872,0.832568,0.015251,0.731461,0.024642,0.735862,0.945877
3,old_baseline_aggregation,logreg_pca128_C0.3,20878,0.749493,0.018186,0.835380,0.015148,0.734381,0.026335,0.728000,0.946306


In [ ]:
pivot = comparison_df.pivot(
    index="probe",
    columns="dataset",
    values=[
        "test_auroc_mean",
        "test_f1_mean",
        "test_accuracy_mean",
        "val_auroc_mean",
        "train_auroc_mean",
    ],
)

delta_rows = []

for probe in comparison_df["probe"].unique():
    old = comparison_df[
        (comparison_df["probe"] == probe)
        & (comparison_df["dataset"] == "old_baseline_aggregation")
    ].iloc[0]

    new = comparison_df[
        (comparison_df["probe"] == probe)
        & (comparison_df["dataset"] == "new_selected_rich")
    ].iloc[0]

    delta_rows.append({
        "probe": probe,
        "old_test_auroc": old["test_auroc_mean"],
        "new_test_auroc": new["test_auroc_mean"],
        "delta_test_auroc": new["test_auroc_mean"] - old["test_auroc_mean"],
        "old_test_f1": old["test_f1_mean"],
        "new_test_f1": new["test_f1_mean"],
        "delta_test_f1": new["test_f1_mean"] - old["test_f1_mean"],
        "old_test_accuracy": old["test_accuracy_mean"],
        "new_test_accuracy": new["test_accuracy_mean"],
        "delta_test_accuracy": new["test_accuracy_mean"] - old["test_accuracy_mean"],
        "old_train_auroc": old["train_auroc_mean"],
        "new_train_auroc": new["train_auroc_mean"],
        "old_val_auroc": old["val_auroc_mean"],
        "new_val_auroc": new["val_auroc_mean"],
    })

delta_df = pd.DataFrame(delta_rows).sort_values(
    "delta_test_auroc",
    ascending=False,
)

display(delta_df)

print("Average delta AUROC:", delta_df["delta_test_auroc"].mean())
print("Best old AUROC:", comparison_df[comparison_df["dataset"] == "old_baseline_aggregation"]["test_auroc_mean"].max())
print("Best new AUROC:", comparison_df[comparison_df["dataset"] == "new_selected_rich"]["test_auroc_mean"].max())

,probe,old_test_auroc,new_test_auroc,delta_test_auroc,old_test_f1,new_test_f1,delta_test_f1,old_test_accuracy,new_test_accuracy,delta_test_accuracy,old_train_auroc,new_train_auroc,old_val_auroc,new_val_auroc
1,logreg_no_pca_C0.01,0.762384,0.767317,0.004933,0.834274,0.835807,0.001533,0.738739,0.743097,0.004358,1.000000,1.000000,0.754069,0.752000
2,logreg_no_pca_C0.03,0.762384,0.767317,0.004933,0.834274,0.835807,0.001533,0.738739,0.743097,0.004358,1.000000,1.000000,0.754069,0.752000
3,logreg_pca128_C0.03,0.749493,0.750027,0.000534,0.835380,0.832568,-0.002812,0.734381,0.731461,-0.002920,0.946306,0.945877,0.728000,0.735862
4,logreg_pca128_C0.3,0.749493,0.750027,0.000534,0.835380,0.832568,-0.002812,0.734381,0.731461,-0.002920,0.946306,0.945877,0.728000,0.735862
5,logreg_pca64_C0.03,0.749493,0.750027,0.000534,0.835380,0.832568,-0.002812,0.734381,0.731461,-0.002920,0.946306,0.945877,0.728000,0.735862
6,logreg_pca64_C0.3,0.749493,0.750027,0.000534,0.835380,0.832568,-0.002812,0.734381,0.731461,-0.002920,0.946306,0.945877,0.728000,0.735862
0,hgb_lr0.03_l21.0,0.783457,0.769509,-0.013949,0.843920,0.841703,-0.002217,0.759082,0.751772,-0.007310,1.000000,1.000000,0.765931,0.771310


Average delta AUROC: -0.00027827240191185306
Best old AUROC: 0.7834570606383531
Best new AUROC: 0.7695085282038387


## Checking feature selection methods

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score


def rank_features_supervised(X_train, y_train, method="corr"):
    X_train = np.asarray(X_train)
    y_train = np.asarray(y_train)

    if method == "corr":
        y_centered = y_train - y_train.mean()
        X_centered = X_train - X_train.mean(axis=0)

        numerator = np.abs(X_centered.T @ y_centered)
        denominator = (
            np.sqrt((X_centered ** 2).sum(axis=0))
            * np.sqrt((y_centered ** 2).sum())
            + 1e-12
        )

        scores = numerator / denominator

    elif method == "auc":
        scores = np.zeros(X_train.shape[1])

        for j in range(X_train.shape[1]):
            x = X_train[:, j]

            if np.std(x) < 1e-12:
                scores[j] = 0.5
                continue

            try:
                auc = roc_auc_score(y_train, x)
                scores[j] = max(auc, 1 - auc)
            except ValueError:
                scores[j] = 0.5

    elif method == "mi":
        scores = mutual_info_classif(
            X_train,
            y_train,
            discrete_features=False,
            random_state=42,
        )

    else:
        raise ValueError(f"Unknown method: {method}")

    return scores


def select_top_k_features(X_train, y_train, k=500, method="corr"):
    scores = rank_features_supervised(X_train, y_train, method=method)
    k = min(k, X_train.shape[1])

    selected_idx = np.argsort(scores)[-k:][::-1]

    return selected_idx, scores

In [ ]:
class FeatureSelectedProbe:
    def __init__(self, BaseProbeClass, k=500, method="corr"):
        self.BaseProbeClass = BaseProbeClass
        self.k = k
        self.method = method
        self.selected_idx_ = None
        self.feature_scores_ = None
        self.probe_ = None

    def fit(self, X, y):
        self.selected_idx_, self.feature_scores_ = select_top_k_features(
            X,
            y,
            k=self.k,
            method=self.method,
        )

        self.probe_ = self.BaseProbeClass()
        self.probe_.fit(X[:, self.selected_idx_], y)

        return self

    def fit_hyperparameters(self, X_val, y_val):
        if hasattr(self.probe_, "fit_hyperparameters"):
            self.probe_.fit_hyperparameters(
                X_val[:, self.selected_idx_],
                y_val,
            )
        return self

    def predict(self, X):
        return self.probe_.predict(X[:, self.selected_idx_])

    def predict_proba(self, X):
        return self.probe_.predict_proba(X[:, self.selected_idx_])

In [ ]:
selection_experiments = []

# base models we had before
base_probe_specs = [
    ("hgb_lr0.03_l21.0", HistGradientBoostingProbe),
    ("logreg_no_pca_C0.01", LogisticNoPCAProbe),
    ("logreg_pca128_C0.03", LogisticPCAProbe),
]

# full features baseline
for probe_name, ProbeClass in base_probe_specs:
    print(f"Running FULL | {probe_name}")

    fold_results = run_evaluation(
        splits=splits,
        X=X,
        y=y,
        ProbeClass=ProbeClass,
    )

    selection_experiments.append({
        "mode": "full",
        "method": None,
        "k": X.shape[1],
        "probe": probe_name,
        "test_auroc_mean": np.mean([r["test_auroc"] for r in fold_results]),
        "test_auroc_std": np.std([r["test_auroc"] for r in fold_results]),
        "test_f1_mean": np.mean([r["test_f1"] for r in fold_results]),
        "test_accuracy_mean": np.mean([r["test_accuracy"] for r in fold_results]),
        "val_auroc_mean": np.mean([r["val_auroc"] for r in fold_results]),
        "train_auroc_mean": np.mean([r["train_auroc"] for r in fold_results]),
    })


# selected features
for method in ["corr", "auc", "mi"]:
    for k in [100, 200, 500, 1000, 2000]:
        for probe_name, BaseProbeClass in base_probe_specs:

            class SelectedProbe:
                def __init__(self):
                    self.model = FeatureSelectedProbe(
                        BaseProbeClass=BaseProbeClass,
                        k=k,
                        method=method,
                    )

                def fit(self, X_train, y_train):
                    self.model.fit(X_train, y_train)
                    return self

                def fit_hyperparameters(self, X_val, y_val):
                    self.model.fit_hyperparameters(X_val, y_val)
                    return self

                def predict(self, X_test):
                    return self.model.predict(X_test)

                def predict_proba(self, X_test):
                    return self.model.predict_proba(X_test)

            print(f"Running SELECTED | method={method} | k={k} | {probe_name}")

            fold_results = run_evaluation(
                splits=splits,
                X=X,
                y=y,
                ProbeClass=SelectedProbe,
            )

            selection_experiments.append({
                "mode": "selected",
                "method": method,
                "k": k,
                "probe": probe_name,
                "test_auroc_mean": np.mean([r["test_auroc"] for r in fold_results]),
                "test_auroc_std": np.std([r["test_auroc"] for r in fold_results]),
                "test_f1_mean": np.mean([r["test_f1"] for r in fold_results]),
                "test_accuracy_mean": np.mean([r["test_accuracy"] for r in fold_results]),
                "val_auroc_mean": np.mean([r["val_auroc"] for r in fold_results]),
                "train_auroc_mean": np.mean([r["train_auroc"] for r in fold_results]),
            })


selection_results_df = pd.DataFrame(selection_experiments).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(selection_results_df.head(30))

best = selection_results_df.iloc[0]

print(
    f"BEST:\n"
    f"mode={best['mode']}, method={best['method']}, k={best['k']}, probe={best['probe']}\n"
    f"Test AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}\n"
    f"Test F1={best['test_f1_mean']:.4f}\n"
    f"Test Accuracy={best['test_accuracy_mean']:.4f}\n"
    f"Val AUROC={best['val_auroc_mean']:.4f}\n"
    f"Train AUROC={best['train_auroc_mean']:.4f}"
)

Running FULL | hgb_lr0.03_l21.0

──────────────────────────────────────────────────
  Fold 1/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 100.00%  F1: 100.00%  AUROC: 100.00%
  Probe val  — Acc: 74.70%  F1: 83.46%  AUROC: 80.21%
  Probe test — Acc: 74.64%  F1: 83.72%  AUROC: 78.83%

──────────────────────────────────────────────────
  Fold 2/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 100.00%  F1: 100.00%  AUROC: 100.00%
  Probe val  — Acc: 78.31%  F1: 86.15%  AUROC: 75.86%
  Probe test — Acc: 77.54%  F1: 85.71%  AUROC: 81.27%

──────────────────────────────────────────────────
  Fold 3/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 100.00%  F1: 100.00%  AUROC: 100.00%
  Probe val  — Acc: 78.31%

,mode,method,k,probe,test_auroc_mean,test_auroc_std,test_f1_mean,test_accuracy_mean,val_auroc_mean,train_auroc_mean
30,selected,auc,2000,hgb_lr0.03_l21.0,0.792345,0.016307,0.847069,0.759061,0.764690,1.000000
4,selected,corr,100,logreg_no_pca_C0.01,0.792248,0.013925,0.835624,0.732942,0.762759,0.872715
27,selected,auc,1000,hgb_lr0.03_l21.0,0.792078,0.023853,0.849932,0.759092,0.773793,1.000000
45,selected,mi,2000,hgb_lr0.03_l21.0,0.791628,0.027962,0.835599,0.740209,0.752138,1.000000
24,selected,auc,500,hgb_lr0.03_l21.0,0.789521,0.034308,0.838396,0.748884,0.764966,1.000000
34,selected,mi,100,logreg_no_pca_C0.01,0.789230,0.032830,0.829169,0.732910,0.761655,0.881554
37,selected,mi,200,logreg_no_pca_C0.01,0.786695,0.045216,0.835409,0.732910,0.739310,0.926110
12,selected,corr,1000,hgb_lr0.03_l21.0,0.786400,0.027285,0.835534,0.745943,0.772828,1.000000
19,selected,auc,100,logreg_no_pca_C0.01,0.785981,0.013853,0.836518,0.751835,0.764552,0.870055
0,full,NaN,20878,hgb_lr0.03_l21.0,0.785756,0.021209,0.845753,0.763430,0.767310,1.000000


BEST:
mode=selected, method=auc, k=2000, probe=hgb_lr0.03_l21.0
Test AUROC=0.7923 ± 0.0163
Test F1=0.8471
Test Accuracy=0.7591
Val AUROC=0.7647
Train AUROC=1.0000


## Trying PCA (didn't work)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier


class SelectedPCAProbe:
    def __init__(
        self,
        BaseModel="logreg",
        selection_method="auc",
        k=2000,
        n_components=128,
    ):
        self.BaseModel = BaseModel
        self.selection_method = selection_method
        self.k = k
        self.n_components = n_components

        self.selected_idx_ = None
        self.threshold_ = 0.5
        self.model_ = None

    def fit(self, X, y):
        self.selected_idx_, _ = select_top_k_features(
            X,
            y,
            k=self.k,
            method=self.selection_method,
        )

        X_sel = X[:, self.selected_idx_]

        n_components = min(
            self.n_components,
            X_sel.shape[0] - 1,
            X_sel.shape[1],
        )

        if self.BaseModel == "logreg":
            self.model_ = make_pipeline(
                StandardScaler(),
                PCA(n_components=n_components, random_state=42),
                LogisticRegression(
                    C=0.03,
                    class_weight="balanced",
                    max_iter=5000,
                    solver="lbfgs",
                ),
            )

        elif self.BaseModel == "hgb":
            self.model_ = make_pipeline(
                StandardScaler(),
                PCA(n_components=n_components, random_state=42),
                HistGradientBoostingClassifier(
                    learning_rate=0.03,
                    max_iter=150,
                    l2_regularization=1.0,
                    random_state=42,
                ),
            )

        else:
            raise ValueError(self.BaseModel)

        self.model_.fit(X_sel, y)

        return self

    def fit_hyperparameters(self, X_val, y_val):
        probs = self.predict_proba(X_val)[:, 1]

        thresholds = np.linspace(0.05, 0.95, 181)

        best_thr = 0.5
        best_f1 = -1

        for thr in thresholds:
            preds = (probs >= thr).astype(int)
            score = f1_score(y_val, preds)

            if score > best_f1:
                best_f1 = score
                best_thr = thr

        self.threshold_ = best_thr

        return self

    def predict_proba(self, X):
        X_sel = X[:, self.selected_idx_]
        return self.model_.predict_proba(X_sel)

    def predict(self, X):
        probs = self.predict_proba(X)[:, 1]
        return (probs >= self.threshold_).astype(int)

In [ ]:
targeted_results = []

configs = []

for model_name in ["logreg", "hgb"]:
    for n_components in [64, 128, 256]:
        configs.append(
            (model_name, n_components)
        )

for model_name, n_components in configs:

    class Probe:
        def __init__(self):
            self.model = SelectedPCAProbe(
                BaseModel=model_name,
                selection_method="auc",
                k=2000,
                n_components=n_components,
            )

        def fit(self, X_train, y_train):
            self.model.fit(X_train, y_train)
            return self

        def fit_hyperparameters(self, X_val, y_val):
            self.model.fit_hyperparameters(X_val, y_val)
            return self

        def predict(self, X_test):
            return self.model.predict(X_test)

        def predict_proba(self, X_test):
            return self.model.predict_proba(X_test)

    experiment_name = f"{model_name}_pca{n_components}"

    print(f"Running {experiment_name}")

    fold_results = run_evaluation(
        splits=splits,
        X=X,
        y=y,
        ProbeClass=Probe,
    )

    targeted_results.append({
        "experiment": experiment_name,
        "test_auroc_mean": np.mean([r["test_auroc"] for r in fold_results]),
        "test_auroc_std": np.std([r["test_auroc"] for r in fold_results]),
        "test_f1_mean": np.mean([r["test_f1"] for r in fold_results]),
        "test_accuracy_mean": np.mean([r["test_accuracy"] for r in fold_results]),
        "val_auroc_mean": np.mean([r["val_auroc"] for r in fold_results]),
        "train_auroc_mean": np.mean([r["train_auroc"] for r in fold_results]),
    })

targeted_df = pd.DataFrame(targeted_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(targeted_df)

best = targeted_df.iloc[0]

print(
    f"\nBEST:\n"
    f"{best['experiment']}\n"
    f"Test AUROC = {best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}\n"
    f"Test F1 = {best['test_f1_mean']:.4f}\n"
    f"Test Accuracy = {best['test_accuracy_mean']:.4f}\n"
    f"Val AUROC = {best['val_auroc_mean']:.4f}\n"
    f"Train AUROC = {best['train_auroc_mean']:.4f}"
)

Running logreg_pca64

──────────────────────────────────────────────────
  Fold 1/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 77.78%  F1: 86.17%  AUROC: 89.27%
  Probe val  — Acc: 73.49%  F1: 83.08%  AUROC: 78.14%
  Probe test — Acc: 73.91%  F1: 83.49%  AUROC: 74.81%

──────────────────────────────────────────────────
  Fold 2/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 77.78%  F1: 85.98%  AUROC: 88.63%
  Probe val  — Acc: 77.11%  F1: 85.71%  AUROC: 77.52%
  Probe test — Acc: 71.01%  F1: 81.48%  AUROC: 79.08%

──────────────────────────────────────────────────
  Fold 3/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 80.56%  F1: 87.20%  AUROC: 88.33%
  Probe val  — Acc: 75.90%  F1: 84.85%  AUROC:

,experiment,test_auroc_mean,test_auroc_std,test_f1_mean,test_accuracy_mean,val_auroc_mean,train_auroc_mean
0,logreg_pca64,0.778295,0.023162,0.832541,0.737290,0.751724,0.887666
2,logreg_pca256,0.763630,0.025156,0.836854,0.740188,0.728000,0.994938
5,hgb_pca256,0.757358,0.017876,0.826663,0.722829,0.757931,1.000000
4,hgb_pca128,0.756325,0.009217,0.838647,0.744589,0.779034,1.000000
1,logreg_pca128,0.755419,0.025159,0.830438,0.732931,0.731172,0.947771
3,hgb_pca64,0.751347,0.015199,0.833060,0.732974,0.759034,1.000000



BEST:
logreg_pca64
Test AUROC = 0.7783 ± 0.0232
Test F1 = 0.8325
Test Accuracy = 0.7373
Val AUROC = 0.7517
Train AUROC = 0.8877


## Text leakage analysis

In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from scipy.stats import pointbiserialr

df_text = pd.read_csv("./data/dataset.csv")
df_text["label"] = df_text["label"].astype(float).astype(int)

df_text.head()

,prompt,response,label
0,<|im_start|>system\nYou are a helpful assistan...,An architect or engineer has a direct relation...,1
1,<|im_start|>system\nYou are a helpful assistan...,Based on the information provided in the conte...,1
2,<|im_start|>system\nYou are a helpful assistan...,"The ""appropriate"" (well defined), minimum effi...",1
3,<|im_start|>system\nYou are a helpful assistan...,The name of the longest bridge in Germany is D...,1
4,<|im_start|>system\nYou are a helpful assistan...,The present-day company that BankAmericard tur...,0


In [ ]:
def count_regex(text, pattern):
    return len(re.findall(pattern, str(text)))


def make_text_diagnostics(df):
    out = pd.DataFrame()
    out["label"] = df["label"].astype(int)

    prompt = df["prompt"].astype(str)
    response = df["response"].astype(str)
    full_text = prompt + response

    out["prompt_chars"] = prompt.str.len()
    out["response_chars"] = response.str.len()
    out["full_chars"] = full_text.str.len()

    out["prompt_words"] = prompt.str.split().str.len()
    out["response_words"] = response.str.split().str.len()
    out["full_words"] = full_text.str.split().str.len()

    out["response_ratio_chars"] = out["response_chars"] / (out["full_chars"] + 1e-9)
    out["response_ratio_words"] = out["response_words"] / (out["full_words"] + 1e-9)

    out["response_sentences"] = response.apply(lambda x: count_regex(x, r"[.!?]+"))
    out["response_commas"] = response.apply(lambda x: count_regex(x, r","))
    out["response_digits"] = response.apply(lambda x: count_regex(x, r"\d"))
    out["response_quotes"] = response.apply(lambda x: count_regex(x, r"[\"']"))
    out["response_upper_chars"] = response.apply(lambda x: sum(ch.isupper() for ch in x))
    out["response_question_marks"] = response.apply(lambda x: count_regex(x, r"\?"))
    out["response_exclamation_marks"] = response.apply(lambda x: count_regex(x, r"!"))

    out["response_avg_word_len"] = response.apply(
        lambda x: np.mean([len(w) for w in str(x).split()]) if len(str(x).split()) else 0
    )

    out["has_endoftext"] = response.str.contains("<\\|endoftext\\|>", regex=True).astype(int)
    out["has_im_start"] = response.str.contains("<\\|im_start\\|>", regex=True).astype(int)
    out["has_im_end"] = response.str.contains("<\\|im_end\\|>", regex=True).astype(int)

    hedge_words = [
        "maybe", "probably", "likely", "possibly", "seems", "appears",
        "might", "could", "may", "generally", "typically", "often",
        "usually", "suggests", "approximately"
    ]

    refusal_words = [
        "cannot", "can't", "unable", "sorry", "not enough information",
        "insufficient", "unknown", "does not provide", "not specified"
    ]

    generic_words = [
        "important", "significant", "various", "different", "several",
        "many", "some", "overall", "in general"
    ]

    for group_name, words in [
        ("hedge", hedge_words),
        ("refusal", refusal_words),
        ("generic", generic_words),
    ]:
        pattern = r"\b(" + "|".join(re.escape(w) for w in words) + r")\b"
        out[f"{group_name}_count"] = response.str.lower().apply(
            lambda x: count_regex(x, pattern)
        )

    return out


text_diag = make_text_diagnostics(df_text)
text_diag.head()

,label,prompt_chars,response_chars,full_chars,prompt_words,response_words,full_words,response_ratio_chars,response_ratio_words,response_sentences,...,response_upper_chars,response_question_marks,response_exclamation_marks,response_avg_word_len,has_endoftext,has_im_start,has_im_end,hedge_count,refusal_count,generic_count
0,1,1122,87,1209,164,11,175,0.071960,0.062857,1,...,1,0,0,7.000000,1,0,0,0,0,0
1,1,980,291,1271,146,45,191,0.228954,0.235602,3,...,5,1,0,5.466667,1,0,0,1,0,0
2,1,1184,166,1350,179,24,203,0.122963,0.118227,1,...,1,0,0,5.958333,1,0,0,0,0,0
3,1,1357,67,1424,207,10,217,0.047051,0.046083,1,...,3,0,0,5.800000,1,0,0,0,0,0
4,0,1016,98,1114,155,13,168,0.087971,0.077381,1,...,5,0,0,6.615385,1,0,0,0,0,0


In [ ]:
def feature_signal_table(df_features, label_col="label"):
    y = df_features[label_col].values
    rows = []

    for col in df_features.columns:
        if col == label_col:
            continue

        x = df_features[col].replace([np.inf, -np.inf], np.nan).fillna(0).values

        if np.std(x) < 1e-12:
            continue

        corr = np.corrcoef(x, y)[0, 1]

        try:
            auc = roc_auc_score(y, x)
            auc_oriented = max(auc, 1 - auc)
        except Exception:
            auc_oriented = np.nan

        mean_0 = x[y == 0].mean()
        mean_1 = x[y == 1].mean()

        std_0 = x[y == 0].std()
        std_1 = x[y == 1].std()

        pooled_std = np.sqrt((std_0 ** 2 + std_1 ** 2) / 2) + 1e-12
        cohen_d = (mean_1 - mean_0) / pooled_std

        rows.append({
            "feature": col,
            "auc_oriented": auc_oriented,
            "abs_corr": abs(corr),
            "corr": corr,
            "abs_cohen_d": abs(cohen_d),
            "cohen_d": cohen_d,
            "mean_truthful_0": mean_0,
            "mean_hallucinated_1": mean_1,
        })

    return pd.DataFrame(rows).sort_values("auc_oriented", ascending=False)


text_signal = feature_signal_table(text_diag)
display(text_signal)

,feature,auc_oriented,abs_corr,corr,abs_cohen_d,cohen_d,mean_truthful_0,mean_hallucinated_1
8,response_sentences,0.670918,0.215121,0.215121,0.534650,0.534650,3.737864,7.714286
5,full_words,0.651787,0.237578,0.237578,0.571863,0.571863,262.966019,333.213251
2,full_chars,0.648330,0.234462,0.234462,0.563394,0.563394,1705.834951,2131.532091
1,response_chars,0.644088,0.236680,0.236680,0.573059,0.573059,417.669903,790.409938
4,response_words,0.644073,0.238574,0.238574,0.578785,0.578785,65.432039,126.929607
9,response_commas,0.643656,0.224929,0.224929,0.554601,0.554601,3.077670,6.730849
6,response_ratio_chars,0.637852,0.231187,0.231187,0.544962,0.544962,0.204898,0.303962
7,response_ratio_words,0.637360,0.230993,0.230993,0.544423,0.544423,0.205738,0.308499
12,response_upper_chars,0.600861,0.117382,0.117382,0.261362,0.261362,13.825243,20.807453
15,response_avg_word_len,0.588941,0.005697,-0.005697,0.013251,-0.013251,5.847341,5.827298


In [ ]:
df_dup = df_text.copy()

df_dup["prompt_clean"] = (
    df_dup["prompt"]
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df_dup["response_clean"] = (
    df_dup["response"]
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df_dup["full_clean"] = df_dup["prompt_clean"] + " " + df_dup["response_clean"]

print("Exact duplicate prompts:", df_dup["prompt_clean"].duplicated().sum())
print("Exact duplicate responses:", df_dup["response_clean"].duplicated().sum())
print("Exact duplicate full samples:", df_dup["full_clean"].duplicated().sum())

prompt_dups = (
    df_dup.groupby("prompt_clean")["label"]
    .agg(["count", "mean"])
    .query("count > 1")
    .sort_values("count", ascending=False)
)

response_dups = (
    df_dup.groupby("response_clean")["label"]
    .agg(["count", "mean"])
    .query("count > 1")
    .sort_values("count", ascending=False)
)

print("\nPrompt duplicate groups:")
display(prompt_dups.head(20))

print("\nResponse duplicate groups:")
display(response_dups.head(20))

Exact duplicate prompts: 0
Exact duplicate responses: 14
Exact duplicate full samples: 0

Prompt duplicate groups:


,count,mean
prompt_clean,,



Response duplicate groups:


,count,mean
response_clean,,
you are a helpful assistant.moid<|endoftext|>,9,1.0
unable to answer based on given context.<|endoftext|>,5,0.0
you are a helpful assistant.moid.<|endoftext|>,3,1.0


In [ ]:
print("TEXT LEAKAGE ANALYSIS SUMMARY")
print("=" * 80)
print("Rows:", len(df_text))
print("Label counts:")
print(df_text["label"].value_counts().sort_index())
print()

print("Top text/stat features by oriented AUROC:")
display(text_signal.head(20))

print("Duplicate summary:")
print("Exact duplicate prompts:", df_dup["prompt_clean"].duplicated().sum())
print("Exact duplicate responses:", df_dup["response_clean"].duplicated().sum())
print("Exact duplicate full samples:", df_dup["full_clean"].duplicated().sum())

print("\nLength means by label:")
display(
    text_diag.groupby("label")[
        [
            "prompt_words",
            "response_words",
            "response_ratio_words",
            "prompt_chars",
            "response_chars",
            "response_ratio_chars",
        ]
    ].mean()
)

TEXT LEAKAGE ANALYSIS SUMMARY
Rows: 689
Label counts:
label
0    206
1    483
Name: count, dtype: int64

Top text/stat features by oriented AUROC:


,feature,auc_oriented,abs_corr,corr,abs_cohen_d,cohen_d,mean_truthful_0,mean_hallucinated_1
8,response_sentences,0.670918,0.215121,0.215121,0.534650,0.534650,3.737864,7.714286
5,full_words,0.651787,0.237578,0.237578,0.571863,0.571863,262.966019,333.213251
2,full_chars,0.648330,0.234462,0.234462,0.563394,0.563394,1705.834951,2131.532091
1,response_chars,0.644088,0.236680,0.236680,0.573059,0.573059,417.669903,790.409938
4,response_words,0.644073,0.238574,0.238574,0.578785,0.578785,65.432039,126.929607
9,response_commas,0.643656,0.224929,0.224929,0.554601,0.554601,3.077670,6.730849
6,response_ratio_chars,0.637852,0.231187,0.231187,0.544962,0.544962,0.204898,0.303962
7,response_ratio_words,0.637360,0.230993,0.230993,0.544423,0.544423,0.205738,0.308499
12,response_upper_chars,0.600861,0.117382,0.117382,0.261362,0.261362,13.825243,20.807453
15,response_avg_word_len,0.588941,0.005697,-0.005697,0.013251,-0.013251,5.847341,5.827298


Duplicate summary:
Exact duplicate prompts: 0
Exact duplicate responses: 14
Exact duplicate full samples: 0

Length means by label:


,prompt_words,response_words,response_ratio_words,prompt_chars,response_chars,response_ratio_chars
label,,,,,,
0,197.533981,65.432039,0.205738,1288.165049,417.669903,0.204898
1,206.283644,126.929607,0.308499,1341.122153,790.409938,0.303962


## Trying to get more text features - tf-idf

In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score

from evaluate import run_evaluation

In [ ]:
df_text = pd.read_csv("./data/dataset.csv")
df_text["label"] = df_text["label"].astype(float).astype(int)

y_text = df_text["label"].to_numpy()

# using samе splits, as before
splits_text = make_stratified_kfold_splits(
    y_text,
    n_splits=5,
    val_size=0.15,
    random_state=42,
)

In [ ]:
def count_regex(text, pattern):
    return len(re.findall(pattern, str(text)))


def make_text_stats_df(df):
    out = pd.DataFrame(index=df.index)

    prompt = df["prompt"].astype(str)
    response = df["response"].astype(str)
    full = prompt + " " + response

    out["prompt_chars"] = prompt.str.len()
    out["response_chars"] = response.str.len()
    out["full_chars"] = full.str.len()

    out["prompt_words"] = prompt.str.split().str.len()
    out["response_words"] = response.str.split().str.len()
    out["full_words"] = full.str.split().str.len()

    out["response_ratio_chars"] = out["response_chars"] / (out["full_chars"] + 1e-9)
    out["response_ratio_words"] = out["response_words"] / (out["full_words"] + 1e-9)

    out["response_sentences"] = response.apply(lambda x: count_regex(x, r"[.!?]+"))
    out["response_commas"] = response.apply(lambda x: count_regex(x, r","))
    out["response_digits"] = response.apply(lambda x: count_regex(x, r"\d"))
    out["response_quotes"] = response.apply(lambda x: count_regex(x, r"[\"']"))
    out["response_upper_chars"] = response.apply(lambda x: sum(ch.isupper() for ch in x))
    out["response_question_marks"] = response.apply(lambda x: count_regex(x, r"\?"))
    out["response_exclamation_marks"] = response.apply(lambda x: count_regex(x, r"!"))

    out["response_avg_word_len"] = response.apply(
        lambda x: np.mean([len(w) for w in str(x).split()]) if len(str(x).split()) else 0
    )

    hedge_words = [
        "maybe", "probably", "likely", "possibly", "seems", "appears",
        "might", "could", "may", "generally", "typically", "often",
        "usually", "suggests", "approximately"
    ]

    refusal_words = [
        "cannot", "can't", "unable", "sorry", "not enough information",
        "insufficient", "unknown", "does not provide", "not specified"
    ]

    generic_words = [
        "important", "significant", "various", "different", "several",
        "many", "some", "overall", "in general"
    ]

    for group_name, words in [
        ("hedge", hedge_words),
        ("refusal", refusal_words),
        ("generic", generic_words),
    ]:
        pattern = r"\b(" + "|".join(re.escape(w) for w in words) + r")\b"
        out[f"{group_name}_count"] = response.str.lower().apply(
            lambda x: count_regex(x, pattern)
        )

    return out.astype(np.float32)


text_stats = make_text_stats_df(df_text)
X_stats = text_stats.to_numpy(dtype=np.float32)

print("Text stats shape:", X_stats.shape)

Text stats shape: (689, 19)


In [ ]:
class TextStatsLogRegProbe:
    def __init__(self, C=0.3):
        self.C = C
        self.threshold_ = 0.5
        self.model_ = None

    def fit(self, X, y):
        self.model_ = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                C=self.C,
                class_weight="balanced",
                max_iter=5000,
                solver="lbfgs",
            )),
        ])
        self.model_.fit(X, y)
        return self

    def fit_hyperparameters(self, X_val, y_val):
        probs = self.predict_proba(X_val)[:, 1]
        best_thr, best_f1 = 0.5, -1

        for thr in np.linspace(0.05, 0.95, 181):
            preds = (probs >= thr).astype(int)
            score = f1_score(y_val, preds, zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_thr = thr

        self.threshold_ = best_thr
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= self.threshold_).astype(int)

In [ ]:
class TfidfLogRegProbe:
    def __init__(
        self,
        text_col="response",
        analyzer="word",
        ngram_range=(1, 2),
        max_features=5000,
        C=1.0,
    ):
        self.text_col = text_col
        self.analyzer = analyzer
        self.ngram_range = ngram_range
        self.max_features = max_features
        self.C = C

        self.threshold_ = 0.5
        self.model_ = None

    def _select_text(self, X):
        X = pd.DataFrame(X, columns=["prompt", "response"])

        if self.text_col == "full":
            return (X["prompt"].astype(str) + " " + X["response"].astype(str)).values

        col_idx = 0 if self.text_col == "prompt" else 1
        return X.iloc[:, col_idx].astype(str).values

    def fit(self, X, y):
        vectorizer = TfidfVectorizer(
            lowercase=True,
            analyzer=self.analyzer,
            ngram_range=self.ngram_range,
            max_features=self.max_features,
            min_df=2,
            sublinear_tf=True,
        )

        self.model_ = Pipeline([
            ("select_text", FunctionTransformer(self._select_text, validate=False)),
            ("tfidf", vectorizer),
            ("clf", LogisticRegression(
                C=self.C,
                class_weight="balanced",
                max_iter=5000,
                solver="liblinear",
            )),
        ])

        self.model_.fit(X, y)
        return self

    def fit_hyperparameters(self, X_val, y_val):
        probs = self.predict_proba(X_val)[:, 1]
        best_thr, best_f1 = 0.5, -1

        for thr in np.linspace(0.05, 0.95, 181):
            preds = (probs >= thr).astype(int)
            score = f1_score(y_val, preds, zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_thr = thr

        self.threshold_ = best_thr
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= self.threshold_).astype(int)

In [ ]:
text_results = []

def summarize_fold_results(name, fold_results):
    return {
        "model": name,
        "test_auroc_mean": np.mean([r["test_auroc"] for r in fold_results]),
        "test_auroc_std": np.std([r["test_auroc"] for r in fold_results]),
        "test_f1_mean": np.mean([r["test_f1"] for r in fold_results]),
        "test_accuracy_mean": np.mean([r["test_accuracy"] for r in fold_results]),
        "val_auroc_mean": np.mean([r["val_auroc"] for r in fold_results]),
        "train_auroc_mean": np.mean([r["train_auroc"] for r in fold_results]),
    }


# 1. stats only
for C in [0.03, 0.1, 0.3, 1.0]:
    class StatsProbe:
        def __init__(self):
            self.model = TextStatsLogRegProbe(C=C)

        def fit(self, X_train, y_train):
            self.model.fit(X_train, y_train)
            return self

        def fit_hyperparameters(self, X_val, y_val):
            self.model.fit_hyperparameters(X_val, y_val)
            return self

        def predict(self, X_test):
            return self.model.predict(X_test)

        def predict_proba(self, X_test):
            return self.model.predict_proba(X_test)

    print(f"Running stats_logreg_C{C}")
    fold_results = run_evaluation(
        splits=splits_text,
        X=X_stats,
        y=y_text,
        ProbeClass=StatsProbe,
    )
    text_results.append(summarize_fold_results(f"stats_logreg_C{C}", fold_results))


# 2. TF-IDF variants
tfidf_configs = [
    ("response_word_1_2", "response", "word", (1, 2), 5000, 1.0),
    ("response_word_1_3", "response", "word", (1, 3), 10000, 1.0),
    ("response_char_3_5", "response", "char_wb", (3, 5), 10000, 1.0),
    ("full_word_1_2", "full", "word", (1, 2), 10000, 1.0),
    ("full_char_3_5", "full", "char_wb", (3, 5), 15000, 1.0),
    ("prompt_word_1_2", "prompt", "word", (1, 2), 5000, 1.0),
]

for name, text_col, analyzer, ngram_range, max_features, C in tfidf_configs:
    class Probe:
        def __init__(self):
            self.model = TfidfLogRegProbe(
                text_col=text_col,
                analyzer=analyzer,
                ngram_range=ngram_range,
                max_features=max_features,
                C=C,
            )

        def fit(self, X_train, y_train):
            self.model.fit(X_train, y_train)
            return self

        def fit_hyperparameters(self, X_val, y_val):
            self.model.fit_hyperparameters(X_val, y_val)
            return self

        def predict(self, X_test):
            return self.model.predict(X_test)

        def predict_proba(self, X_test):
            return self.model.predict_proba(X_test)

    print(f"Running tfidf_{name}")
    fold_results = run_evaluation(
        splits=splits_text,
        X=df_text[["prompt", "response"]].to_numpy(dtype=object),
        y=y_text,
        ProbeClass=Probe,
    )
    text_results.append(summarize_fold_results(f"tfidf_{name}", fold_results))


text_results_df = pd.DataFrame(text_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(text_results_df)

best = text_results_df.iloc[0]
print(
    f"BEST TEXT MODEL: {best['model']}\n"
    f"Test AUROC = {best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}\n"
    f"Test F1 = {best['test_f1_mean']:.4f}\n"
    f"Test Accuracy = {best['test_accuracy_mean']:.4f}\n"
    f"Val AUROC = {best['val_auroc_mean']:.4f}\n"
    f"Train AUROC = {best['train_auroc_mean']:.4f}"
)

Running stats_logreg_C0.03

──────────────────────────────────────────────────
  Fold 1/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 69.87%  F1: 81.12%  AUROC: 69.51%
  Probe val  — Acc: 73.49%  F1: 83.33%  AUROC: 70.69%
  Probe test — Acc: 68.84%  F1: 80.72%  AUROC: 66.08%

──────────────────────────────────────────────────
  Fold 2/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 70.09%  F1: 82.41%  AUROC: 69.44%
  Probe val  — Acc: 69.88%  F1: 82.27%  AUROC: 68.21%
  Probe test — Acc: 70.29%  F1: 82.55%  AUROC: 70.48%

──────────────────────────────────────────────────
  Fold 3/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 69.87%  F1: 80.61%  AUROC: 68.93%
  Probe val  — Acc: 75.90%  F1: 84.62%  

,model,test_auroc_mean,test_auroc_std,test_f1_mean,test_accuracy_mean,val_auroc_mean,train_auroc_mean
6,tfidf_response_char_3_5,0.695993,0.032398,0.821360,0.703914,0.706345,0.966013
3,stats_logreg_C1.0,0.671216,0.037604,0.817578,0.698117,0.664828,0.715286
1,stats_logreg_C0.1,0.670810,0.036566,0.818556,0.702465,0.674897,0.706021
0,stats_logreg_C0.03,0.670344,0.033894,0.821281,0.709732,0.677241,0.696590
2,stats_logreg_C0.3,0.670212,0.037783,0.814178,0.696678,0.672000,0.710979
5,tfidf_response_word_1_3,0.667213,0.036883,0.827529,0.712610,0.692690,0.990162
4,tfidf_response_word_1_2,0.666493,0.034112,0.815651,0.702391,0.705655,0.988408
8,tfidf_full_char_3_5,0.623463,0.049853,0.810851,0.687887,0.653793,0.982224
7,tfidf_full_word_1_2,0.618179,0.061693,0.824436,0.703904,0.628552,0.994645
9,tfidf_prompt_word_1_2,0.541449,0.072841,0.810391,0.684989,0.535448,0.986734


BEST TEXT MODEL: tfidf_response_char_3_5
Test AUROC = 0.6960 ± 0.0324
Test F1 = 0.8214
Test Accuracy = 0.7039
Val AUROC = 0.7063
Train AUROC = 0.9660


## Trying to create ensemble

In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.dummy import DummyClassifier

In [ ]:
# hidden-state rich features
df_hidden = pd.read_parquet(
    "./artifacts/selected_rich_features/features_dataset_selected_rich.parquet"
)

drop_cols = {"label", "prompt", "response", "source_index", "id"}
hidden_feature_cols = [c for c in df_hidden.columns if c not in drop_cols]

X_hidden_full = df_hidden[hidden_feature_cols].to_numpy(dtype=np.float32)
y = df_hidden["label"].astype(int).to_numpy()

# raw text
df_text = pd.read_csv("./data/dataset.csv")
X_text_pairs = df_text[["prompt", "response"]].to_numpy(dtype=object)

print("Hidden:", X_hidden_full.shape)
print("Text:", X_text_pairs.shape)
print("Positive rate:", y.mean())

Hidden: (689, 20878)
Text: (689, 2)
Positive rate: 0.7010159651669086


In [ ]:
def count_regex(text, pattern):
    return len(re.findall(pattern, str(text)))


def make_text_stats_df(df):
    out = pd.DataFrame(index=df.index)

    prompt = df["prompt"].astype(str)
    response = df["response"].astype(str)
    full = prompt + " " + response

    out["prompt_chars"] = prompt.str.len()
    out["response_chars"] = response.str.len()
    out["full_chars"] = full.str.len()

    out["prompt_words"] = prompt.str.split().str.len()
    out["response_words"] = response.str.split().str.len()
    out["full_words"] = full.str.split().str.len()

    out["response_ratio_chars"] = out["response_chars"] / (out["full_chars"] + 1e-9)
    out["response_ratio_words"] = out["response_words"] / (out["full_words"] + 1e-9)

    out["response_sentences"] = response.apply(lambda x: count_regex(x, r"[.!?]+"))
    out["response_commas"] = response.apply(lambda x: count_regex(x, r","))
    out["response_digits"] = response.apply(lambda x: count_regex(x, r"\d"))
    out["response_quotes"] = response.apply(lambda x: count_regex(x, r"[\"']"))
    out["response_upper_chars"] = response.apply(lambda x: sum(ch.isupper() for ch in x))
    out["response_question_marks"] = response.apply(lambda x: count_regex(x, r"\?"))
    out["response_exclamation_marks"] = response.apply(lambda x: count_regex(x, r"!"))

    out["response_avg_word_len"] = response.apply(
        lambda x: np.mean([len(w) for w in str(x).split()]) if len(str(x).split()) else 0
    )

    hedge_words = [
        "maybe", "probably", "likely", "possibly", "seems", "appears",
        "might", "could", "may", "generally", "typically", "often",
        "usually", "suggests", "approximately"
    ]

    refusal_words = [
        "cannot", "can't", "unable", "sorry", "not enough information",
        "insufficient", "unknown", "does not provide", "not specified"
    ]

    generic_words = [
        "important", "significant", "various", "different", "several",
        "many", "some", "overall", "in general"
    ]

    for group_name, words in [
        ("hedge", hedge_words),
        ("refusal", refusal_words),
        ("generic", generic_words),
    ]:
        pattern = r"\b(" + "|".join(re.escape(w) for w in words) + r")\b"
        out[f"{group_name}_count"] = response.str.lower().apply(
            lambda x: count_regex(x, pattern)
        )

    return out.astype(np.float32)


X_stats_full = make_text_stats_df(df_text).to_numpy(dtype=np.float32)
print("Stats:", X_stats_full.shape)

Stats: (689, 19)


In [ ]:
from sklearn.metrics import roc_auc_score


def rank_features_auc(X_train, y_train):
    scores = np.zeros(X_train.shape[1])

    for j in range(X_train.shape[1]):
        x = X_train[:, j]

        if np.std(x) < 1e-12:
            scores[j] = 0.5
            continue

        try:
            auc = roc_auc_score(y_train, x)
            scores[j] = max(auc, 1 - auc)
        except Exception:
            scores[j] = 0.5

    return scores


def select_top_k_auc(X_train, y_train, k=2000):
    scores = rank_features_auc(X_train, y_train)
    k = min(k, X_train.shape[1])
    idx = np.argsort(scores)[-k:][::-1]
    return idx

In [ ]:
def get_hidden_model():
    return HistGradientBoostingClassifier(
        learning_rate=0.03,
        max_iter=150,
        l2_regularization=1.0,
        random_state=42,
    )


def get_stats_model():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=5000,
            solver="lbfgs",
        )),
    ])


def get_tfidf_model():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            analyzer="char_wb",
            ngram_range=(3, 5),
            max_features=10000,
            min_df=2,
            sublinear_tf=True,
        )),
        ("clf", LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=5000,
            solver="liblinear",
        )),
    ])


def tune_threshold_by_f1(y_val, probs_val):
    best_thr = 0.5
    best_f1 = -1

    for thr in np.linspace(0.05, 0.95, 181):
        preds = (probs_val >= thr).astype(int)
        score = f1_score(y_val, preds, zero_division=0)

        if score > best_f1:
            best_f1 = score
            best_thr = thr

    return best_thr


def calc_metrics(y_true, probs, threshold=0.5):
    preds = (probs >= threshold).astype(int)

    return {
        "auroc": roc_auc_score(y_true, probs),
        "f1": f1_score(y_true, preds, zero_division=0),
        "accuracy": accuracy_score(y_true, preds),
    }

In [ ]:
def evaluate_stacking(
    splits,
    X_hidden,
    X_stats,
    X_text_pairs,
    y,
    hidden_k=2000,
):
    rows = []

    for fold_id, (idx_train, idx_val, idx_test) in enumerate(splits, start=1):
        print(f"Fold {fold_id}")

        y_train = y[idx_train]
        y_val = y[idx_val]
        y_test = y[idx_test]

        # -------------------------
        # Hidden model
        # -------------------------
        selected_idx = select_top_k_auc(
            X_hidden[idx_train],
            y_train,
            k=hidden_k,
        )

        hidden_model = get_hidden_model()
        hidden_model.fit(X_hidden[idx_train][:, selected_idx], y_train)

        hidden_val = hidden_model.predict_proba(
            X_hidden[idx_val][:, selected_idx]
        )[:, 1]
        hidden_test = hidden_model.predict_proba(
            X_hidden[idx_test][:, selected_idx]
        )[:, 1]

        # -------------------------
        # Stats model
        # -------------------------
        stats_model = get_stats_model()
        stats_model.fit(X_stats[idx_train], y_train)

        stats_val = stats_model.predict_proba(X_stats[idx_val])[:, 1]
        stats_test = stats_model.predict_proba(X_stats[idx_test])[:, 1]

        # -------------------------
        # TF-IDF response char model
        # -------------------------
        response_train = X_text_pairs[idx_train, 1].astype(str)
        response_val = X_text_pairs[idx_val, 1].astype(str)
        response_test = X_text_pairs[idx_test, 1].astype(str)

        tfidf_model = get_tfidf_model()
        tfidf_model.fit(response_train, y_train)

        tfidf_val = tfidf_model.predict_proba(response_val)[:, 1]
        tfidf_test = tfidf_model.predict_proba(response_test)[:, 1]

        # -------------------------
        # Single models
        # -------------------------
        single_sources = {
            "hidden_only": (hidden_val, hidden_test),
            "stats_only": (stats_val, stats_test),
            "tfidf_only": (tfidf_val, tfidf_test),
        }

        for name, (val_probs, test_probs) in single_sources.items():
            thr = tune_threshold_by_f1(y_val, val_probs)
            m = calc_metrics(y_test, test_probs, threshold=thr)

            rows.append({
                "fold": fold_id,
                "model": name,
                "threshold": thr,
                "test_auroc": m["auroc"],
                "test_f1": m["f1"],
                "test_accuracy": m["accuracy"],
            })

        # -------------------------
        # Stacking combinations
        # Meta model learns on VAL only,
        # evaluates on TEST only.
        # -------------------------
        combinations = {
            "stack_hidden_stats": (
                np.column_stack([hidden_val, stats_val]),
                np.column_stack([hidden_test, stats_test]),
            ),
            "stack_hidden_tfidf": (
                np.column_stack([hidden_val, tfidf_val]),
                np.column_stack([hidden_test, tfidf_test]),
            ),
            "stack_stats_tfidf": (
                np.column_stack([stats_val, tfidf_val]),
                np.column_stack([stats_test, tfidf_test]),
            ),
            "stack_hidden_stats_tfidf": (
                np.column_stack([hidden_val, stats_val, tfidf_val]),
                np.column_stack([hidden_test, stats_test, tfidf_test]),
            ),
        }

        for name, (Z_val, Z_test) in combinations.items():
            meta = LogisticRegression(
                C=1.0,
                class_weight="balanced",
                max_iter=5000,
                solver="lbfgs",
            )

            meta.fit(Z_val, y_val)
            meta_val = meta.predict_proba(Z_val)[:, 1]
            meta_test = meta.predict_proba(Z_test)[:, 1]

            thr = tune_threshold_by_f1(y_val, meta_val)
            m = calc_metrics(y_test, meta_test, threshold=thr)

            rows.append({
                "fold": fold_id,
                "model": name,
                "threshold": thr,
                "test_auroc": m["auroc"],
                "test_f1": m["f1"],
                "test_accuracy": m["accuracy"],
            })

    return pd.DataFrame(rows)

In [ ]:
stacking_df = evaluate_stacking(
    splits=splits,
    X_hidden=X_hidden_full,
    X_stats=X_stats_full,
    X_text_pairs=X_text_pairs,
    y=y,
    hidden_k=2000,
)

summary = (
    stacking_df
    .groupby("model")
    .agg(
        test_auroc_mean=("test_auroc", "mean"),
        test_auroc_std=("test_auroc", "std"),
        test_f1_mean=("test_f1", "mean"),
        test_f1_std=("test_f1", "std"),
        test_accuracy_mean=("test_accuracy", "mean"),
        test_accuracy_std=("test_accuracy", "std"),
        threshold_mean=("threshold", "mean"),
    )
    .reset_index()
    .sort_values("test_auroc_mean", ascending=False)
)

display(summary)

best = summary.iloc[0]
print(
    f"BEST MODEL: {best['model']}\n"
    f"Test AUROC = {best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}\n"
    f"Test F1 = {best['test_f1_mean']:.4f}\n"
    f"Test Accuracy = {best['test_accuracy_mean']:.4f}"
)

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


,model,test_auroc_mean,test_auroc_std,test_f1_mean,test_f1_std,test_accuracy_mean,test_accuracy_std,threshold_mean
0,hidden_only,0.792345,0.018232,0.847069,0.021747,0.759061,0.028814,0.292
2,stack_hidden_stats_tfidf,0.786420,0.021592,0.844715,0.007349,0.750354,0.009961,0.296
3,stack_hidden_tfidf,0.786308,0.017348,0.841700,0.013772,0.748916,0.014010,0.311
1,stack_hidden_stats,0.785624,0.022766,0.850098,0.017291,0.760499,0.027372,0.310
4,stack_stats_tfidf,0.712243,0.039700,0.824226,0.006141,0.709754,0.015558,0.331
6,tfidf_only,0.695993,0.036222,0.821360,0.015374,0.703914,0.018113,0.226
5,stats_only,0.671216,0.042043,0.817578,0.017340,0.698117,0.013992,0.238


BEST MODEL: hidden_only
Test AUROC = 0.7923 ± 0.0182
Test F1 = 0.8471
Test Accuracy = 0.7591


## Checking better probes

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score

from evaluate import run_evaluation

In [ ]:
class ThresholdTuningMixin:
    def fit_hyperparameters(self, X_val, y_val):
        probs = self.predict_proba(X_val)[:, 1]

        best_thr = 0.5
        best_f1 = -1.0

        for thr in np.linspace(0.05, 0.95, 181):
            preds = (probs >= thr).astype(int)
            score = f1_score(y_val, preds, zero_division=0)

            if score > best_f1:
                best_f1 = score
                best_thr = thr

        self.threshold_ = best_thr
        return self

    def predict(self, X):
        probs = self.predict_proba(X)[:, 1]
        return (probs >= self.threshold_).astype(int)


class SelectedFeatureProbe(ThresholdTuningMixin, BaseEstimator, ClassifierMixin):
    def __init__(self, model_factory, k=2000):
        self.model_factory = model_factory
        self.k = k
        self.selected_idx_ = None
        self.model_ = None
        self.threshold_ = 0.5

    def fit(self, X, y):
        self.selected_idx_ = select_top_k_auc(X, y, k=self.k)
        X_sel = X[:, self.selected_idx_]

        self.model_ = self.model_factory()
        self.model_.fit(X_sel, y)

        return self

    def predict_proba(self, X):
        X_sel = X[:, self.selected_idx_]

        if hasattr(self.model_, "predict_proba"):
            return self.model_.predict_proba(X_sel)

        scores = self.model_.decision_function(X_sel)
        probs_1 = 1 / (1 + np.exp(-scores))
        return np.column_stack([1 - probs_1, probs_1])

In [ ]:
probe_specs = []

# 0. best current baseline
probe_specs.append((
    "current_best_hgb_top2000",
    lambda: SelectedFeatureProbe(
        model_factory=lambda: HistGradientBoostingClassifier(
            learning_rate=0.03,
            max_iter=150,
            l2_regularization=1.0,
            random_state=42,
        ),
        k=2000,
    )
))

# 1. calibrated boosting
probe_specs.append((
    "calibrated_hgb_sigmoid",
    lambda: SelectedFeatureProbe(
        model_factory=lambda: CalibratedClassifierCV(
            estimator=HistGradientBoostingClassifier(
                learning_rate=0.03,
                max_iter=120,
                l2_regularization=1.0,
                random_state=42,
            ),
            method="sigmoid",
            cv=3,
        ),
        k=2000,
    )
))

probe_specs.append((
    "gradient_boosting",
    lambda: SelectedFeatureProbe(
        model_factory=lambda: GradientBoostingClassifier(
            n_estimators=150,
            learning_rate=0.03,
            max_depth=2,
            random_state=42,
        ),
        k=2000,
    )
))

probe_specs.append((
    "extra_trees",
    lambda: SelectedFeatureProbe(
        model_factory=lambda: ExtraTreesClassifier(
            n_estimators=300,
            max_depth=4,
            min_samples_leaf=8,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        k=2000,
    )
))

# 2. margin-based models
probe_specs.append((
    "linear_svc_calibrated",
    lambda: SelectedFeatureProbe(
        model_factory=lambda: CalibratedClassifierCV(
            estimator=Pipeline([
                ("scaler", StandardScaler()),
                ("svc", LinearSVC(
                    C=0.03,
                    class_weight="balanced",
                    max_iter=10000,
                    random_state=42,
                )),
            ]),
            method="sigmoid",
            cv=3,
        ),
        k=2000,
    )
))

probe_specs.append((
    "sgd_log_loss",
    lambda: SelectedFeatureProbe(
        model_factory=lambda: Pipeline([
            ("scaler", StandardScaler()),
            ("sgd", SGDClassifier(
                loss="log_loss",
                alpha=0.0003,
                penalty="elasticnet",
                l1_ratio=0.15,
                class_weight="balanced",
                max_iter=5000,
                random_state=42,
            )),
        ]),
        k=2000,
    )
))

probe_specs.append((
    "sgd_modified_huber",
    lambda: SelectedFeatureProbe(
        model_factory=lambda: Pipeline([
            ("scaler", StandardScaler()),
            ("sgd", SGDClassifier(
                loss="modified_huber",
                alpha=0.0003,
                penalty="elasticnet",
                l1_ratio=0.15,
                class_weight="balanced",
                max_iter=5000,
                random_state=42,
            )),
        ]),
        k=2000,
    )
))

probe_specs.append((
    "rbf_svc_calibrated",
    lambda: SelectedFeatureProbe(
        model_factory=lambda: Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(
                C=1.0,
                gamma="scale",
                kernel="rbf",
                class_weight="balanced",
                probability=True,
                random_state=42,
            )),
        ]),
        k=1000,
    )
))

# 3. shallow neural heads
probe_specs.append((
    "mlp_64",
    lambda: SelectedFeatureProbe(
        model_factory=lambda: Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPClassifier(
                hidden_layer_sizes=(64,),
                activation="relu",
                alpha=0.01,
                learning_rate_init=0.001,
                max_iter=500,
                early_stopping=True,
                random_state=42,
            )),
        ]),
        k=2000,
    )
))

probe_specs.append((
    "mlp_128_32",
    lambda: SelectedFeatureProbe(
        model_factory=lambda: Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPClassifier(
                hidden_layer_sizes=(128, 32),
                activation="relu",
                alpha=0.03,
                learning_rate_init=0.001,
                max_iter=500,
                early_stopping=True,
                random_state=42,
            )),
        ]),
        k=2000,
    )
))

In [ ]:
probe_results = []

for name, factory in probe_specs:
    class Probe:
        def __init__(self):
            self.model = factory()

        def fit(self, X_train, y_train):
            self.model.fit(X_train, y_train)
            return self

        def fit_hyperparameters(self, X_val, y_val):
            self.model.fit_hyperparameters(X_val, y_val)
            return self

        def predict(self, X_test):
            return self.model.predict(X_test)

        def predict_proba(self, X_test):
            return self.model.predict_proba(X_test)

    print(f"Running {name}")

    fold_results = run_evaluation(
        splits=splits,
        X=X_hidden_full,
        y=y,
        ProbeClass=Probe,
    )

    probe_results.append({
        "probe": name,
        "test_auroc_mean": np.mean([r["test_auroc"] for r in fold_results]),
        "test_auroc_std": np.std([r["test_auroc"] for r in fold_results]),
        "test_f1_mean": np.mean([r["test_f1"] for r in fold_results]),
        "test_accuracy_mean": np.mean([r["test_accuracy"] for r in fold_results]),
        "val_auroc_mean": np.mean([r["val_auroc"] for r in fold_results]),
        "train_auroc_mean": np.mean([r["train_auroc"] for r in fold_results]),
    })

probe_results_df = pd.DataFrame(probe_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(probe_results_df)

best = probe_results_df.iloc[0]
print(
    f"BEST PROBE: {best['probe']}\n"
    f"Test AUROC = {best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}\n"
    f"Test F1 = {best['test_f1_mean']:.4f}\n"
    f"Test Accuracy = {best['test_accuracy_mean']:.4f}\n"
    f"Val AUROC = {best['val_auroc_mean']:.4f}\n"
    f"Train AUROC = {best['train_auroc_mean']:.4f}"
)

Running current_best_hgb_top2000

──────────────────────────────────────────────────
  Fold 1/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 100.00%  F1: 100.00%  AUROC: 100.00%
  Probe val  — Acc: 77.11%  F1: 85.04%  AUROC: 81.86%
  Probe test — Acc: 71.74%  F1: 81.16%  AUROC: 76.31%

──────────────────────────────────────────────────
  Fold 2/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 99.57%  F1: 99.70%  AUROC: 100.00%
  Probe val  — Acc: 78.31%  F1: 86.57%  AUROC: 75.24%
  Probe test — Acc: 76.09%  F1: 85.07%  AUROC: 79.53%

──────────────────────────────────────────────────
  Fold 3/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 100.00%  F1: 100.00%  AUROC: 100.00%
  Probe val  — Acc: 78.31% 

,probe,test_auroc_mean,test_auroc_std,test_f1_mean,test_accuracy_mean,val_auroc_mean,train_auroc_mean
3,extra_trees,0.792826,0.022113,0.833274,0.728573,0.773379,0.928671
0,current_best_hgb_top2000,0.792345,0.016307,0.847069,0.759061,0.764690,1.000000
2,gradient_boosting,0.791753,0.012858,0.845119,0.760531,0.775862,0.990393
1,calibrated_hgb_sigmoid,0.785442,0.018150,0.837959,0.745996,0.771034,0.999564
7,rbf_svc_calibrated,0.784524,0.024258,0.843430,0.753242,0.766207,0.975780
8,mlp_64,0.768073,0.040412,0.832078,0.728552,0.744000,0.938251
9,mlp_128_32,0.750284,0.040351,0.831709,0.724246,0.744828,0.945456
4,linear_svc_calibrated,0.740290,0.024856,0.823367,0.719867,0.717517,0.998899
5,sgd_log_loss,0.669268,0.026373,0.816161,0.728562,0.648207,0.998508
6,sgd_modified_huber,0.628229,0.030614,0.801473,0.711139,0.614345,0.986527


BEST PROBE: extra_trees
Test AUROC = 0.7928 ± 0.0221
Test F1 = 0.8333
Test Accuracy = 0.7286
Val AUROC = 0.7734
Train AUROC = 0.9287


## Best are ExtraTrees and CatBoost, so let's try to tune them

In [ ]:
import optuna
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import roc_auc_score


def evaluate_extratrees_params(params, X, y, splits, k=2000):
    aucs = []

    for idx_train, idx_val, idx_test in splits:
        selected_idx = select_top_k_auc(X[idx_train], y[idx_train], k=k)

        model = ExtraTreesClassifier(
            **params,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X[idx_train][:, selected_idx], y[idx_train])
        probs = model.predict_proba(X[idx_test][:, selected_idx])[:, 1]

        aucs.append(roc_auc_score(y[idx_test], probs))

    return float(np.mean(aucs))


def objective_extratrees(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=100),
        "max_depth": trial.suggest_int("max_depth", 2, 12),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
        "max_features": trial.suggest_float("max_features", 0.05, 0.8),
        "bootstrap": trial.suggest_categorical("bootstrap", [False, True]),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
    }

    return evaluate_extratrees_params(
        params=params,
        X=X_hidden_full,
        y=y,
        splits=splits,
        k=2000,
    )


study_et = optuna.create_study(direction="maximize")
study_et.optimize(objective_extratrees, n_trials=40)

print("Best ExtraTrees AUROC:", study_et.best_value)
print("Best ExtraTrees params:")
print(study_et.best_params)

[I 2026-05-09 21:31:37,194] A new study created in memory with name: no-name-d5320ae6-0848-43cb-ac8b-05510e5b9ca8
[I 2026-05-09 21:32:30,514] Trial 0 finished with value: 0.7971863263367538 and parameters: {'n_estimators': 700, 'max_depth': 8, 'min_samples_leaf': 14, 'min_samples_split': 18, 'max_features': 0.07315882671241018, 'bootstrap': False, 'criterion': 'gini'}. Best is trial 0 with value: 0.7971863263367538.
[I 2026-05-09 21:33:21,990] Trial 1 finished with value: 0.7935268168955621 and parameters: {'n_estimators': 300, 'max_depth': 3, 'min_samples_leaf': 20, 'min_samples_split': 23, 'max_features': 0.7033125657921049, 'bootstrap': False, 'criterion': 'entropy'}. Best is trial 0 with value: 0.7971863263367538.
[I 2026-05-09 21:34:24,283] Trial 2 finished with value: 0.8011871709751706 and parameters: {'n_estimators': 700, 'max_depth': 12, 'min_samples_leaf': 4, 'min_samples_split': 17, 'max_features': 0.6978420165072609, 'bootstrap': False, 'criterion': 'entropy'}. Best is tria

Best ExtraTrees AUROC: 0.8022029945999017
Best ExtraTrees params:
{'n_estimators': 300, 'max_depth': 11, 'min_samples_leaf': 3, 'min_samples_split': 24, 'max_features': 0.45607614218952963, 'bootstrap': False, 'criterion': 'entropy'}


In [ ]:
from catboost import CatBoostClassifier


def evaluate_catboost_params(params, X, y, splits, k=2000):
    aucs = []

    for idx_train, idx_val, idx_test in splits:
        selected_idx = select_top_k_auc(X[idx_train], y[idx_train], k=k)

        model = CatBoostClassifier(
            **params,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=42,
            verbose=False,
            allow_writing_files=False,
        )

        model.fit(
            X[idx_train][:, selected_idx],
            y[idx_train],
            eval_set=(X[idx_val][:, selected_idx], y[idx_val]),
            use_best_model=True,
        )

        probs = model.predict_proba(X[idx_test][:, selected_idx])[:, 1]
        aucs.append(roc_auc_score(y[idx_test], probs))

    return float(np.mean(aucs))


def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 700, step=100),
        "depth": trial.suggest_int("depth", 2, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.1, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
        "border_count": trial.suggest_int("border_count", 32, 254),
        "auto_class_weights": trial.suggest_categorical(
            "auto_class_weights",
            ["Balanced", "SqrtBalanced", None],
        ),
    }

    return evaluate_catboost_params(
        params=params,
        X=X_hidden_full,
        y=y,
        splits=splits,
        k=2000,
    )


study_cb = optuna.create_study(direction="maximize")
study_cb.optimize(objective_catboost, n_trials=40)

print("Best CatBoost AUROC:", study_cb.best_value)
print("Best CatBoost params:")
print(study_cb.best_params)

[I 2026-05-09 22:07:22,085] A new study created in memory with name: no-name-ab3ed9eb-396a-4fa0-bb4a-d90121222e4d
[I 2026-05-09 22:08:44,788] Trial 0 finished with value: 0.7860461729847417 and parameters: {'iterations': 200, 'depth': 5, 'learning_rate': 0.030793137157853798, 'l2_leaf_reg': 3.0503342037492294, 'random_strength': 0.4569598814073298, 'bagging_temperature': 1.1094256807877438, 'border_count': 234, 'auto_class_weights': 'Balanced'}. Best is trial 0 with value: 0.7860461729847417.
[I 2026-05-09 22:09:37,863] Trial 1 finished with value: 0.7798199767711963 and parameters: {'iterations': 100, 'depth': 3, 'learning_rate': 0.026810529437761216, 'l2_leaf_reg': 1.0999335074746193, 'random_strength': 0.11783545666841422, 'bagging_temperature': 2.203103194925351, 'border_count': 122, 'auto_class_weights': 'SqrtBalanced'}. Best is trial 0 with value: 0.7860461729847417.
[I 2026-05-09 22:10:40,754] Trial 2 finished with value: 0.7624633370551305 and parameters: {'iterations': 400, 'd

Best CatBoost AUROC: 0.8027567865424604
Best CatBoost params:
{'iterations': 500, 'depth': 4, 'learning_rate': 0.019959619273879407, 'l2_leaf_reg': 2.162455881138666, 'random_strength': 0.6740545508815552, 'bagging_temperature': 0.9808801733520118, 'border_count': 81, 'auto_class_weights': 'Balanced'}


In [ ]:
tuning_summary = pd.DataFrame([
    {
        "model": "ExtraTrees tuned",
        "best_cv_auroc": study_et.best_value,
        "best_params": study_et.best_params,
    },
    {
        "model": "CatBoost tuned",
        "best_cv_auroc": study_cb.best_value,
        "best_params": study_cb.best_params,
    },
]).sort_values("best_cv_auroc", ascending=False)

display(tuning_summary)

,model,best_cv_auroc,best_params
1,CatBoost tuned,0.802757,"{'iterations': 500, 'depth': 4, 'learning_rate..."
0,ExtraTrees tuned,0.802203,"{'n_estimators': 300, 'max_depth': 11, 'min_sa..."


## importance + stability selection across folds

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import ExtraTreesClassifier
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

In [ ]:
BEST_ET_PARAMS = study_et.best_params
BEST_CB_PARAMS = study_cb.best_params

In [ ]:
def evaluate_selected_features(model_name, selected_features_by_fold, X, y, splits):
    aucs = []

    for fold_id, (idx_train, idx_val, idx_test) in enumerate(splits):
        selected_idx = selected_features_by_fold[fold_id]

        if model_name == "extratrees":
            model = ExtraTreesClassifier(
                **BEST_ET_PARAMS,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            )
            model.fit(X[idx_train][:, selected_idx], y[idx_train])

        elif model_name == "catboost":
            model = CatBoostClassifier(
                **BEST_CB_PARAMS,
                loss_function="Logloss",
                eval_metric="AUC",
                random_seed=42,
                verbose=False,
                allow_writing_files=False,
            )
            model.fit(
                X[idx_train][:, selected_idx],
                y[idx_train],
                eval_set=(X[idx_val][:, selected_idx], y[idx_val]),
                use_best_model=True,
            )

        probs = model.predict_proba(X[idx_test][:, selected_idx])[:, 1]
        aucs.append(roc_auc_score(y[idx_test], probs))

    return np.mean(aucs), np.std(aucs)

In [ ]:
def get_model_importance(model):
    if hasattr(model, "feature_importances_"):
        return np.asarray(model.feature_importances_)

    try:
        return np.asarray(model.get_feature_importance())
    except Exception:
        raise ValueError("Model has no feature importance")

In [ ]:
def build_importance_selected_features(
    model_name,
    X,
    y,
    splits,
    base_k=2000,
    final_k=500,
):
    selected_by_fold = []

    for fold_id, (idx_train, idx_val, idx_test) in enumerate(splits):
        print(f"{model_name} fold {fold_id + 1}")

        base_idx = select_top_k_auc(X[idx_train], y[idx_train], k=base_k)

        if model_name == "extratrees":
            model = ExtraTreesClassifier(
                **BEST_ET_PARAMS,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            )
            model.fit(X[idx_train][:, base_idx], y[idx_train])

        elif model_name == "catboost":
            model = CatBoostClassifier(
                **BEST_CB_PARAMS,
                loss_function="Logloss",
                eval_metric="AUC",
                random_seed=42,
                verbose=False,
                allow_writing_files=False,
            )
            model.fit(
                X[idx_train][:, base_idx],
                y[idx_train],
                eval_set=(X[idx_val][:, base_idx], y[idx_val]),
                use_best_model=True,
            )

        importance = get_model_importance(model)
        local_top = np.argsort(importance)[-final_k:][::-1]
        final_idx = base_idx[local_top]

        selected_by_fold.append(final_idx)

    return selected_by_fold

In [ ]:
def build_stability_selected_features(
    model_name,
    X,
    y,
    splits,
    base_k=2000,
    per_fold_k=500,
    final_k=500,
):
    all_selected = []

    fold_selected = build_importance_selected_features(
        model_name=model_name,
        X=X,
        y=y,
        splits=splits,
        base_k=base_k,
        final_k=per_fold_k,
    )

    for arr in fold_selected:
        all_selected.extend(arr.tolist())

    counts = pd.Series(all_selected).value_counts()
    stable_idx = counts.head(final_k).index.to_numpy(dtype=int)

    return [stable_idx for _ in splits], counts

In [ ]:
feature_selection_results = []

for model_name in ["extratrees", "catboost"]:
    for final_k in [200, 500, 1000, 1500]:
        print(f"\nIMPORTANCE SELECTION | {model_name} | final_k={final_k}")

        selected_by_fold = build_importance_selected_features(
            model_name=model_name,
            X=X_hidden_full,
            y=y,
            splits=splits,
            base_k=2000,
            final_k=final_k,
        )

        mean_auc, std_auc = evaluate_selected_features(
            model_name=model_name,
            selected_features_by_fold=selected_by_fold,
            X=X_hidden_full,
            y=y,
            splits=splits,
        )

        feature_selection_results.append({
            "selection": "importance_per_fold",
            "model": model_name,
            "base_k": 2000,
            "final_k": final_k,
            "test_auroc_mean": mean_auc,
            "test_auroc_std": std_auc,
        })


for model_name in ["extratrees", "catboost"]:
    for final_k in [200, 500, 1000, 1500]:
        print(f"\nSTABILITY SELECTION | {model_name} | final_k={final_k}")

        selected_by_fold, counts = build_stability_selected_features(
            model_name=model_name,
            X=X_hidden_full,
            y=y,
            splits=splits,
            base_k=2000,
            per_fold_k=500,
            final_k=final_k,
        )

        mean_auc, std_auc = evaluate_selected_features(
            model_name=model_name,
            selected_features_by_fold=selected_by_fold,
            X=X_hidden_full,
            y=y,
            splits=splits,
        )

        feature_selection_results.append({
            "selection": "stability_across_folds",
            "model": model_name,
            "base_k": 2000,
            "final_k": final_k,
            "test_auroc_mean": mean_auc,
            "test_auroc_std": std_auc,
        })


fs_results_df = pd.DataFrame(feature_selection_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(fs_results_df)

best = fs_results_df.iloc[0]
print(
    f"BEST:\n"
    f"{best['selection']} | {best['model']} | final_k={best['final_k']}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}"
)


IMPORTANCE SELECTION | extratrees | final_k=200
extratrees fold 1
extratrees fold 2
extratrees fold 3
extratrees fold 4
extratrees fold 5

IMPORTANCE SELECTION | extratrees | final_k=500
extratrees fold 1
extratrees fold 2
extratrees fold 3
extratrees fold 4
extratrees fold 5

IMPORTANCE SELECTION | extratrees | final_k=1000
extratrees fold 1
extratrees fold 2
extratrees fold 3
extratrees fold 4
extratrees fold 5

IMPORTANCE SELECTION | extratrees | final_k=1500
extratrees fold 1
extratrees fold 2
extratrees fold 3
extratrees fold 4
extratrees fold 5

IMPORTANCE SELECTION | catboost | final_k=200
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5

IMPORTANCE SELECTION | catboost | final_k=500
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5

IMPORTANCE SELECTION | catboost | final_k=1000
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5

IMPORTANCE SELECTION | catboost | final_k=1500
catboost fol

,selection,model,base_k,final_k,test_auroc_mean,test_auroc_std
12,stability_across_folds,catboost,2000,200,0.833269,0.026411
8,stability_across_folds,extratrees,2000,200,0.830285,0.026726
9,stability_across_folds,extratrees,2000,500,0.823067,0.020144
13,stability_across_folds,catboost,2000,500,0.822538,0.021581
10,stability_across_folds,extratrees,2000,1000,0.815969,0.021822
11,stability_across_folds,extratrees,2000,1500,0.810471,0.025343
2,importance_per_fold,extratrees,2000,1000,0.800942,0.023269
14,stability_across_folds,catboost,2000,1000,0.798635,0.028071
1,importance_per_fold,extratrees,2000,500,0.797217,0.024999
3,importance_per_fold,extratrees,2000,1500,0.796744,0.021815


BEST:
stability_across_folds | catboost | final_k=200
AUROC=0.8333 ± 0.0264


In [ ]:
baseline_rows = []

for model_name in ["extratrees", "catboost"]:
    selected_by_fold = [
        select_top_k_auc(X_hidden_full[idx_train], y[idx_train], k=2000)
        for idx_train, _, _ in splits
    ]

    mean_auc, std_auc = evaluate_selected_features(
        model_name=model_name,
        selected_features_by_fold=selected_by_fold,
        X=X_hidden_full,
        y=y,
        splits=splits,
    )

    baseline_rows.append({
        "selection": "auc_top2000_baseline",
        "model": model_name,
        "base_k": 2000,
        "final_k": 2000,
        "test_auroc_mean": mean_auc,
        "test_auroc_std": std_auc,
    })

baseline_df = pd.DataFrame(baseline_rows)

display(
    pd.concat([baseline_df, fs_results_df], ignore_index=True)
    .sort_values("test_auroc_mean", ascending=False)
)

,selection,model,base_k,final_k,test_auroc_mean,test_auroc_std
2,stability_across_folds,catboost,2000,200,0.833269,0.026411
3,stability_across_folds,extratrees,2000,200,0.830285,0.026726
4,stability_across_folds,extratrees,2000,500,0.823067,0.020144
5,stability_across_folds,catboost,2000,500,0.822538,0.021581
6,stability_across_folds,extratrees,2000,1000,0.815969,0.021822
7,stability_across_folds,extratrees,2000,1500,0.810471,0.025343
1,auc_top2000_baseline,catboost,2000,2000,0.802757,0.015179
0,auc_top2000_baseline,extratrees,2000,2000,0.802203,0.023872
8,importance_per_fold,extratrees,2000,1000,0.800942,0.023269
9,stability_across_folds,catboost,2000,1000,0.798635,0.028071


## Finding best using SHAP (didn't work)

In [ ]:
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd


def get_catboost_model(params=None):
    model_params = dict(BEST_CB_PARAMS)

    if params is not None:
        model_params.update(params)

    return CatBoostClassifier(
        **model_params,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )


def catboost_shap_selected_features_for_fold(
    X,
    y,
    idx_train,
    idx_val,
    base_k=2000,
    shap_k=200,
):
    # supervised selection per AUC
    base_idx = select_top_k_auc(
        X[idx_train],
        y[idx_train],
        k=base_k,
    )

    X_train_base = X[idx_train][:, base_idx]
    y_train = y[idx_train]

    X_val_base = X[idx_val][:, base_idx]
    y_val = y[idx_val]

    # training CatBoost on base features
    model = get_catboost_model()
    model.fit(
        X_train_base,
        y_train,
        eval_set=(X_val_base, y_val),
        use_best_model=True,
    )

    # SHAP values
    # get_feature_importance(..., type="ShapValues") returns:
    # shape = (n_samples, n_features + 1)
    # last col — expected value, deleting it
    train_pool = Pool(X_train_base, y_train)
    shap_values = model.get_feature_importance(
        train_pool,
        type="ShapValues",
    )

    shap_values = shap_values[:, :-1]

    shap_importance = np.abs(shap_values).mean(axis=0)

    # get top-K by SHAP importance
    shap_k = min(shap_k, len(base_idx))
    local_top = np.argsort(shap_importance)[-shap_k:][::-1]

    final_idx = base_idx[local_top]

    return final_idx, shap_importance, base_idx

In [ ]:
def evaluate_catboost_shap_pruning(
    X,
    y,
    splits,
    base_k=2000,
    shap_k=200,
):
    aucs = []
    selected_by_fold = []

    for fold_id, (idx_train, idx_val, idx_test) in enumerate(splits, start=1):
        print(f"Fold {fold_id} | base_k={base_k} | shap_k={shap_k}")

        final_idx, shap_importance, base_idx = catboost_shap_selected_features_for_fold(
            X=X,
            y=y,
            idx_train=idx_train,
            idx_val=idx_val,
            base_k=base_k,
            shap_k=shap_k,
        )

        selected_by_fold.append(final_idx)

        model = get_catboost_model()
        model.fit(
            X[idx_train][:, final_idx],
            y[idx_train],
            eval_set=(X[idx_val][:, final_idx], y[idx_val]),
            use_best_model=True,
        )

        probs = model.predict_proba(X[idx_test][:, final_idx])[:, 1]
        auc = roc_auc_score(y[idx_test], probs)
        aucs.append(auc)

        print(f"  fold AUROC: {auc:.4f}")

    return {
        "base_k": base_k,
        "shap_k": shap_k,
        "test_auroc_mean": float(np.mean(aucs)),
        "test_auroc_std": float(np.std(aucs)),
        "selected_by_fold": selected_by_fold,
    }

In [67]:
shap_results = []

for shap_k in [50, 100, 150, 200, 300, 500]:
    result = evaluate_catboost_shap_pruning(
        X=X_hidden_full,
        y=y,
        splits=splits,
        base_k=2000,
        shap_k=shap_k,
    )

    shap_results.append({
        "method": "catboost_shap_pruning",
        "base_k": result["base_k"],
        "shap_k": result["shap_k"],
        "test_auroc_mean": result["test_auroc_mean"],
        "test_auroc_std": result["test_auroc_std"],
    })

shap_results_df = pd.DataFrame(shap_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(shap_results_df)

best_shap = shap_results_df.iloc[0]
print(
    f"BEST SHAP:\n"
    f"base_k={best_shap['base_k']}, shap_k={best_shap['shap_k']}\n"
    f"AUROC={best_shap['test_auroc_mean']:.4f} ± {best_shap['test_auroc_std']:.4f}"
)

Fold 1 | base_k=2000 | shap_k=50
  fold AUROC: 0.7672
Fold 2 | base_k=2000 | shap_k=50
  fold AUROC: 0.7976
Fold 3 | base_k=2000 | shap_k=50
  fold AUROC: 0.7160
Fold 4 | base_k=2000 | shap_k=50
  fold AUROC: 0.8088
Fold 5 | base_k=2000 | shap_k=50
  fold AUROC: 0.7828
Fold 1 | base_k=2000 | shap_k=100
  fold AUROC: 0.7694
Fold 2 | base_k=2000 | shap_k=100
  fold AUROC: 0.8019
Fold 3 | base_k=2000 | shap_k=100
  fold AUROC: 0.7986
Fold 4 | base_k=2000 | shap_k=100
  fold AUROC: 0.8038
Fold 5 | base_k=2000 | shap_k=100
  fold AUROC: 0.7678
Fold 1 | base_k=2000 | shap_k=150
  fold AUROC: 0.7843
Fold 2 | base_k=2000 | shap_k=150
  fold AUROC: 0.8122
Fold 3 | base_k=2000 | shap_k=150
  fold AUROC: 0.7956
Fold 4 | base_k=2000 | shap_k=150
  fold AUROC: 0.8073
Fold 5 | base_k=2000 | shap_k=150
  fold AUROC: 0.6824
Fold 1 | base_k=2000 | shap_k=200
  fold AUROC: 0.7431
Fold 2 | base_k=2000 | shap_k=200
  fold AUROC: 0.7723
Fold 3 | base_k=2000 | shap_k=200
  fold AUROC: 0.7822
Fold 4 | base_k

,method,base_k,shap_k,test_auroc_mean,test_auroc_std
5,catboost_shap_pruning,2000,500,0.796070,0.020574
1,catboost_shap_pruning,2000,100,0.788296,0.016173
4,catboost_shap_pruning,2000,300,0.787209,0.033901
3,catboost_shap_pruning,2000,200,0.777075,0.022011
2,catboost_shap_pruning,2000,150,0.776343,0.047949
0,catboost_shap_pruning,2000,50,0.774459,0.032415


BEST SHAP:
base_k=2000, shap_k=500
AUROC=0.7961 ± 0.0206


In [68]:
current_best = pd.DataFrame([
    {
        "method": "stability_catboost",
        "k": 200,
        "test_auroc_mean": 0.833269,
        "test_auroc_std": 0.026411,
    }
])

shap_compare = shap_results_df.rename(columns={"shap_k": "k"})[
    ["method", "k", "test_auroc_mean", "test_auroc_std"]
]

display(
    pd.concat([current_best, shap_compare], ignore_index=True)
    .sort_values("test_auroc_mean", ascending=False)
)

,method,k,test_auroc_mean,test_auroc_std
0,stability_catboost,200,0.833269,0.026411
1,catboost_shap_pruning,500,0.796070,0.020574
2,catboost_shap_pruning,100,0.788296,0.016173
3,catboost_shap_pruning,300,0.787209,0.033901
4,catboost_shap_pruning,200,0.777075,0.022011
5,catboost_shap_pruning,150,0.776343,0.047949
6,catboost_shap_pruning,50,0.774459,0.032415


## Stability selection tuning - catboost

In [69]:
def build_stability_selected_features_tuned(
    model_name,
    X,
    y,
    splits,
    base_k=2000,
    per_fold_k=500,
    final_k=200,
):
    fold_selected = build_importance_selected_features(
        model_name=model_name,
        X=X,
        y=y,
        splits=splits,
        base_k=base_k,
        final_k=per_fold_k,
    )

    all_selected = []
    for arr in fold_selected:
        all_selected.extend(arr.tolist())

    counts = pd.Series(all_selected).value_counts()
    stable_idx = counts.head(final_k).index.to_numpy(dtype=int)

    selected_by_fold = [stable_idx for _ in splits]

    return selected_by_fold, counts

In [70]:
stability_tuning_results = []

grid = []

for base_k in [1000, 1500, 2000, 3000]:
    for per_fold_k in [200, 300, 500, 800]:
        for final_k in [100, 150, 200, 300, 500]:
            if final_k <= per_fold_k and per_fold_k <= base_k:
                grid.append((base_k, per_fold_k, final_k))

print("Total configs:", len(grid))

Total configs: 68


In [71]:
for i, (base_k, per_fold_k, final_k) in enumerate(grid, start=1):
    print(
        f"\n[{i}/{len(grid)}] "
        f"CatBoost stability | base_k={base_k}, per_fold_k={per_fold_k}, final_k={final_k}"
    )

    selected_by_fold, counts = build_stability_selected_features_tuned(
        model_name="catboost",
        X=X_hidden_full,
        y=y,
        splits=splits,
        base_k=base_k,
        per_fold_k=per_fold_k,
        final_k=final_k,
    )

    mean_auc, std_auc = evaluate_selected_features(
        model_name="catboost",
        selected_features_by_fold=selected_by_fold,
        X=X_hidden_full,
        y=y,
        splits=splits,
    )

    stability_tuning_results.append({
        "model": "catboost",
        "base_k": base_k,
        "per_fold_k": per_fold_k,
        "final_k": final_k,
        "test_auroc_mean": mean_auc,
        "test_auroc_std": std_auc,
        "n_unique_seen": len(counts),
        "top_feature_frequency": int(counts.iloc[0]) if len(counts) else 0,
        "features_with_freq_ge_2": int((counts >= 2).sum()),
        "features_with_freq_ge_3": int((counts >= 3).sum()),
    })

stability_tuning_df = pd.DataFrame(stability_tuning_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(stability_tuning_df.head(30))


[1/68] CatBoost stability | base_k=1000, per_fold_k=200, final_k=100
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5

[2/68] CatBoost stability | base_k=1000, per_fold_k=200, final_k=150
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5

[3/68] CatBoost stability | base_k=1000, per_fold_k=200, final_k=200
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5

[4/68] CatBoost stability | base_k=1000, per_fold_k=300, final_k=100
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5

[5/68] CatBoost stability | base_k=1000, per_fold_k=300, final_k=150
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5

[6/68] CatBoost stability | base_k=1000, per_fold_k=300, final_k=200
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5

[7/68] CatBoost stability | base_k=1000, per_fold_k=300, final_k=300
catboost fold 1
catboost fold 

,model,base_k,per_fold_k,final_k,test_auroc_mean,test_auroc_std,n_unique_seen,top_feature_frequency,features_with_freq_ge_2,features_with_freq_ge_3
55,catboost,3000,300,150,0.844963,0.019266,1146,5,272,63
59,catboost,3000,500,150,0.844683,0.014370,1780,5,542,143
58,catboost,3000,500,100,0.844214,0.014578,1780,5,542,143
65,catboost,3000,800,200,0.842890,0.018051,2464,5,1063,368
54,catboost,3000,300,100,0.842459,0.019615,1146,5,272,63
34,catboost,2000,200,100,0.841681,0.024406,746,5,170,58
60,catboost,3000,500,200,0.839776,0.014499,1780,5,542,143
46,catboost,2000,800,100,0.838340,0.020803,2023,5,1114,563
53,catboost,3000,200,200,0.837957,0.019605,786,5,164,37
47,catboost,2000,800,150,0.837837,0.021332,2023,5,1114,563


In [72]:
previous_best = pd.DataFrame([
    {
        "model": "catboost",
        "selection": "previous_stability_best",
        "base_k": 2000,
        "per_fold_k": 500,
        "final_k": 200,
        "test_auroc_mean": 0.833269,
        "test_auroc_std": 0.026411,
    }
])

new_results = stability_tuning_df.copy()
new_results["selection"] = "stability_tuned"

compare_stability = pd.concat(
    [
        previous_best,
        new_results[
            [
                "model",
                "selection",
                "base_k",
                "per_fold_k",
                "final_k",
                "test_auroc_mean",
                "test_auroc_std",
            ]
        ],
    ],
    ignore_index=True,
).sort_values("test_auroc_mean", ascending=False)

display(compare_stability.head(20))

best = compare_stability.iloc[0]

print(
    f"BEST STABILITY RESULT:\n"
    f"{best['selection']} | {best['model']}\n"
    f"base_k={best['base_k']}, per_fold_k={best['per_fold_k']}, final_k={best['final_k']}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}"
)

,model,selection,base_k,per_fold_k,final_k,test_auroc_mean,test_auroc_std
1,catboost,stability_tuned,3000,300,150,0.844963,0.019266
2,catboost,stability_tuned,3000,500,150,0.844683,0.014370
3,catboost,stability_tuned,3000,500,100,0.844214,0.014578
4,catboost,stability_tuned,3000,800,200,0.842890,0.018051
5,catboost,stability_tuned,3000,300,100,0.842459,0.019615
6,catboost,stability_tuned,2000,200,100,0.841681,0.024406
7,catboost,stability_tuned,3000,500,200,0.839776,0.014499
8,catboost,stability_tuned,2000,800,100,0.838340,0.020803
9,catboost,stability_tuned,3000,200,200,0.837957,0.019605
10,catboost,stability_tuned,2000,800,150,0.837837,0.021332


BEST STABILITY RESULT:
stability_tuned | catboost
base_k=3000, per_fold_k=300, final_k=150
AUROC=0.8450 ± 0.0193


## Stability selection tuning - extratrees

In [73]:
best_stability_configs = [
    {"base_k": 3000, "per_fold_k": 300, "final_k": 150},
    {"base_k": 3000, "per_fold_k": 500, "final_k": 150},
    {"base_k": 3000, "per_fold_k": 500, "final_k": 100},
    {"base_k": 3000, "per_fold_k": 800, "final_k": 200},
    {"base_k": 3000, "per_fold_k": 300, "final_k": 100},
]

extra_stability_results = []

for cfg in best_stability_configs:
    print(f"\nExtraTrees stability check: {cfg}")

    selected_by_fold, counts = build_stability_selected_features_tuned(
        model_name="extratrees",
        X=X_hidden_full,
        y=y,
        splits=splits,
        base_k=cfg["base_k"],
        per_fold_k=cfg["per_fold_k"],
        final_k=cfg["final_k"],
    )

    mean_auc, std_auc = evaluate_selected_features(
        model_name="extratrees",
        selected_features_by_fold=selected_by_fold,
        X=X_hidden_full,
        y=y,
        splits=splits,
    )

    extra_stability_results.append({
        "model": "extratrees",
        **cfg,
        "test_auroc_mean": mean_auc,
        "test_auroc_std": std_auc,
    })

extra_stability_df = pd.DataFrame(extra_stability_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(extra_stability_df)


ExtraTrees stability check: {'base_k': 3000, 'per_fold_k': 300, 'final_k': 150}
extratrees fold 1
extratrees fold 2
extratrees fold 3
extratrees fold 4
extratrees fold 5

ExtraTrees stability check: {'base_k': 3000, 'per_fold_k': 500, 'final_k': 150}
extratrees fold 1
extratrees fold 2
extratrees fold 3
extratrees fold 4
extratrees fold 5

ExtraTrees stability check: {'base_k': 3000, 'per_fold_k': 500, 'final_k': 100}
extratrees fold 1
extratrees fold 2
extratrees fold 3
extratrees fold 4
extratrees fold 5

ExtraTrees stability check: {'base_k': 3000, 'per_fold_k': 800, 'final_k': 200}
extratrees fold 1
extratrees fold 2
extratrees fold 3
extratrees fold 4
extratrees fold 5

ExtraTrees stability check: {'base_k': 3000, 'per_fold_k': 300, 'final_k': 100}
extratrees fold 1
extratrees fold 2
extratrees fold 3
extratrees fold 4
extratrees fold 5


,model,base_k,per_fold_k,final_k,test_auroc_mean,test_auroc_std
3,extratrees,3000,800,200,0.838691,0.021760
2,extratrees,3000,500,100,0.834664,0.029322
0,extratrees,3000,300,150,0.831876,0.026565
4,extratrees,3000,300,100,0.831495,0.032074
1,extratrees,3000,500,150,0.830416,0.028486


## Trying ensemble extra trees + catboost

In [74]:
def evaluate_catboost_extratrees_ensemble(
    X,
    y,
    splits,
    base_k=3000,
    per_fold_k=300,
    final_k=150,
    weights=(0.5, 0.5),
):
    selected_by_fold, counts = build_stability_selected_features_tuned(
        model_name="catboost",
        X=X,
        y=y,
        splits=splits,
        base_k=base_k,
        per_fold_k=per_fold_k,
        final_k=final_k,
    )

    rows = []

    for fold_id, (idx_train, idx_val, idx_test) in enumerate(splits, start=1):
        selected_idx = selected_by_fold[fold_id - 1]

        cat = CatBoostClassifier(
            **BEST_CB_PARAMS,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=42,
            verbose=False,
            allow_writing_files=False,
        )

        cat.fit(
            X[idx_train][:, selected_idx],
            y[idx_train],
            eval_set=(X[idx_val][:, selected_idx], y[idx_val]),
            use_best_model=True,
        )

        et = ExtraTreesClassifier(
            **BEST_ET_PARAMS,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )

        et.fit(X[idx_train][:, selected_idx], y[idx_train])

        cat_probs = cat.predict_proba(X[idx_test][:, selected_idx])[:, 1]
        et_probs = et.predict_proba(X[idx_test][:, selected_idx])[:, 1]

        w_cat, w_et = weights
        ens_probs = w_cat * cat_probs + w_et * et_probs

        rows.append({
            "fold": fold_id,
            "catboost_auc": roc_auc_score(y[idx_test], cat_probs),
            "extratrees_auc": roc_auc_score(y[idx_test], et_probs),
            "ensemble_auc": roc_auc_score(y[idx_test], ens_probs),
            "weights": weights,
        })

    return pd.DataFrame(rows)

In [75]:
ensemble_results = []

for weights in [(0.8, 0.2), (0.7, 0.3), (0.6, 0.4), (0.5, 0.5), (0.4, 0.6)]:
    print(f"Testing ensemble weights={weights}")

    fold_df = evaluate_catboost_extratrees_ensemble(
        X=X_hidden_full,
        y=y,
        splits=splits,
        base_k=3000,
        per_fold_k=300,
        final_k=150,
        weights=weights,
    )

    ensemble_results.append({
        "weights": weights,
        "catboost_auc_mean": fold_df["catboost_auc"].mean(),
        "extratrees_auc_mean": fold_df["extratrees_auc"].mean(),
        "ensemble_auc_mean": fold_df["ensemble_auc"].mean(),
        "ensemble_auc_std": fold_df["ensemble_auc"].std(),
    })

ensemble_df = pd.DataFrame(ensemble_results).sort_values(
    "ensemble_auc_mean",
    ascending=False,
)

display(ensemble_df)

print("Current best single model AUROC: 0.8450")
print("Best ensemble AUROC:", ensemble_df.iloc[0]["ensemble_auc_mean"])

Testing ensemble weights=(0.8, 0.2)
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5
Testing ensemble weights=(0.7, 0.3)
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5
Testing ensemble weights=(0.6, 0.4)
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5
Testing ensemble weights=(0.5, 0.5)
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5
Testing ensemble weights=(0.4, 0.6)
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5


,weights,catboost_auc_mean,extratrees_auc_mean,ensemble_auc_mean,ensemble_auc_std
0,"(0.8, 0.2)",0.844963,0.835066,0.845504,0.022317
1,"(0.7, 0.3)",0.844963,0.835066,0.845396,0.022630
2,"(0.6, 0.4)",0.844963,0.835066,0.845342,0.023110
3,"(0.5, 0.5)",0.844963,0.835066,0.844329,0.023916
4,"(0.4, 0.6)",0.844963,0.835066,0.843475,0.023963


Current best single model AUROC: 0.8450
Best ensemble AUROC: 0.8455036100434643


## Redundancy-pruning

In [77]:
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
from catboost import CatBoostClassifier


def get_tuned_catboost():
    return CatBoostClassifier(
        **BEST_CB_PARAMS,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )


def rank_features_auc_scores(X_train, y_train):
    scores = np.zeros(X_train.shape[1], dtype=np.float32)

    for j in range(X_train.shape[1]):
        x = X_train[:, j]

        if np.std(x) < 1e-12:
            scores[j] = 0.5
            continue

        try:
            auc = roc_auc_score(y_train, x)
            scores[j] = max(auc, 1 - auc)
        except Exception:
            scores[j] = 0.5

    return scores


def remove_low_variance_features(X_train, feature_idx, min_std=1e-6):
    stds = X_train[:, feature_idx].std(axis=0)
    return feature_idx[stds > min_std]


def correlation_prune_features(
    X_train,
    feature_idx,
    feature_scores=None,
    threshold=0.98,
    method="pearson",
    max_features=None,
):
    """
    Greedy redundancy pruning.
    Keeps stronger features first, removes highly correlated duplicates.
    """

    feature_idx = np.asarray(feature_idx)

    if len(feature_idx) == 0:
        return feature_idx

    X_sub = X_train[:, feature_idx].astype(np.float32)

    if method == "spearman":
        X_sub = np.apply_along_axis(rankdata, 0, X_sub).astype(np.float32)

    # standardize
    X_sub = X_sub - X_sub.mean(axis=0, keepdims=True)
    X_sub = X_sub / (X_sub.std(axis=0, keepdims=True) + 1e-8)

    if method in ["pearson", "spearman"]:
        sim = np.abs(np.corrcoef(X_sub, rowvar=False))
    elif method == "cosine":
        X_norm = X_sub / (np.linalg.norm(X_sub, axis=0, keepdims=True) + 1e-8)
        sim = np.abs(X_norm.T @ X_norm)
    else:
        raise ValueError(f"Unknown method: {method}")

    np.fill_diagonal(sim, 0.0)

    if feature_scores is None:
        order = np.arange(len(feature_idx))
    else:
        local_scores = feature_scores[feature_idx]
        order = np.argsort(local_scores)[::-1]

    keep_local = []

    for local_i in order:
        if not keep_local:
            keep_local.append(local_i)
        else:
            max_sim = sim[local_i, keep_local].max()
            if max_sim < threshold:
                keep_local.append(local_i)

        if max_features is not None and len(keep_local) >= max_features:
            break

    keep_local = np.array(keep_local, dtype=int)
    return feature_idx[keep_local]

In [78]:
def evaluate_catboost_with_selector(
    selector_fn,
    X,
    y,
    splits,
    verbose=True,
):
    rows = []

    for fold_id, (idx_train, idx_val, idx_test) in enumerate(splits, start=1):
        selected_idx = selector_fn(
            X=X,
            y=y,
            idx_train=idx_train,
            idx_val=idx_val,
            idx_test=idx_test,
            fold_id=fold_id,
        )

        if verbose:
            print(f"Fold {fold_id}: selected {len(selected_idx)} features")

        model = get_tuned_catboost()

        model.fit(
            X[idx_train][:, selected_idx],
            y[idx_train],
            eval_set=(X[idx_val][:, selected_idx], y[idx_val]),
            use_best_model=True,
        )

        probs = model.predict_proba(X[idx_test][:, selected_idx])[:, 1]
        auc = roc_auc_score(y[idx_test], probs)

        rows.append({
            "fold": fold_id,
            "n_features": len(selected_idx),
            "test_auroc": auc,
        })

    fold_df = pd.DataFrame(rows)

    return {
        "test_auroc_mean": fold_df["test_auroc"].mean(),
        "test_auroc_std": fold_df["test_auroc"].std(),
        "n_features_mean": fold_df["n_features"].mean(),
        "fold_df": fold_df,
    }

In [79]:
def make_auc_redundancy_selector(
    base_k=3000,
    corr_threshold=0.98,
    method="pearson",
    max_features=None,
    min_std=None,
):
    def selector_fn(X, y, idx_train, idx_val, idx_test, fold_id):
        X_train = X[idx_train]
        y_train = y[idx_train]

        auc_scores = rank_features_auc_scores(X_train, y_train)

        base_idx = np.argsort(auc_scores)[-base_k:][::-1]

        if min_std is not None:
            base_idx = remove_low_variance_features(
                X_train=X_train,
                feature_idx=base_idx,
                min_std=min_std,
            )

        pruned_idx = correlation_prune_features(
            X_train=X_train,
            feature_idx=base_idx,
            feature_scores=auc_scores,
            threshold=corr_threshold,
            method=method,
            max_features=max_features,
        )

        return pruned_idx

    return selector_fn

In [80]:
def make_stability_redundancy_selector(
    base_k=3000,
    per_fold_k=300,
    final_k=150,
    corr_threshold=0.98,
    method="pearson",
    max_features=None,
):
    selected_by_fold, counts = build_stability_selected_features_tuned(
        model_name="catboost",
        X=X_hidden_full,
        y=y,
        splits=splits,
        base_k=base_k,
        per_fold_k=per_fold_k,
        final_k=final_k,
    )

    stable_idx = selected_by_fold[0]
    stability_scores = np.zeros(X_hidden_full.shape[1], dtype=np.float32)

    for feat, count in counts.items():
        stability_scores[int(feat)] = float(count)

    def selector_fn(X, y, idx_train, idx_val, idx_test, fold_id):
        X_train = X[idx_train]

        pruned_idx = correlation_prune_features(
            X_train=X_train,
            feature_idx=stable_idx,
            feature_scores=stability_scores,
            threshold=corr_threshold,
            method=method,
            max_features=max_features,
        )

        return pruned_idx

    return selector_fn

In [ ]:
redundancy_results = []

# current best result
previous_best = {
    "experiment": "previous_best_stability_catboost",
    "selector": "stability",
    "base_k": 3000,
    "per_fold_k": 300,
    "final_k": 150,
    "method": None,
    "corr_threshold": None,
    "max_features": 150,
    "test_auroc_mean": 0.844963,
    "test_auroc_std": 0.019266,
    "n_features_mean": 150,
}

redundancy_results.append(previous_best)

In [ ]:
# AUC top-k + redundancy pruning
auc_redundancy_configs = []

for base_k in [2000, 3000, 5000]:
    for method in ["pearson", "spearman", "cosine"]:
        for threshold in [0.90, 0.95, 0.97, 0.98, 0.99]:
            auc_redundancy_configs.append({
                "base_k": base_k,
                "method": method,
                "corr_threshold": threshold,
                "max_features": None,
            })

print("AUC redundancy configs:", len(auc_redundancy_configs))

AUC redundancy configs: 45


In [83]:
for i, cfg in enumerate(auc_redundancy_configs, start=1):
    print(f"\nAUC redundancy [{i}/{len(auc_redundancy_configs)}]: {cfg}")

    selector_fn = make_auc_redundancy_selector(
        base_k=cfg["base_k"],
        corr_threshold=cfg["corr_threshold"],
        method=cfg["method"],
        max_features=cfg["max_features"],
        min_std=1e-6,
    )

    result = evaluate_catboost_with_selector(
        selector_fn=selector_fn,
        X=X_hidden_full,
        y=y,
        splits=splits,
        verbose=False,
    )

    redundancy_results.append({
        "experiment": "auc_topk_redundancy",
        "selector": "auc_topk",
        "base_k": cfg["base_k"],
        "per_fold_k": None,
        "final_k": None,
        "method": cfg["method"],
        "corr_threshold": cfg["corr_threshold"],
        "max_features": cfg["max_features"],
        "test_auroc_mean": result["test_auroc_mean"],
        "test_auroc_std": result["test_auroc_std"],
        "n_features_mean": result["n_features_mean"],
    })

    current_df = pd.DataFrame(redundancy_results).sort_values(
        "test_auroc_mean",
        ascending=False,
    )
    display(current_df.head(10))


AUC redundancy [1/45]: {'base_k': 2000, 'method': 'pearson', 'corr_threshold': 0.9, 'max_features': None}


,experiment,selector,base_k,per_fold_k,final_k,method,corr_threshold,max_features,test_auroc_mean,test_auroc_std,n_features_mean
0,previous_best_stability_catboost,stability,3000,300.0,150.0,NaN,NaN,150.0,0.844963,0.019266,150.0
1,auc_topk_redundancy,auc_topk,2000,NaN,NaN,pearson,0.9,NaN,0.783869,0.026097,1399.6



AUC redundancy [2/45]: {'base_k': 2000, 'method': 'pearson', 'corr_threshold': 0.95, 'max_features': None}


,experiment,selector,base_k,per_fold_k,final_k,method,corr_threshold,max_features,test_auroc_mean,test_auroc_std,n_features_mean
0,previous_best_stability_catboost,stability,3000,300.0,150.0,NaN,NaN,150.0,0.844963,0.019266,150.0
2,auc_topk_redundancy,auc_topk,2000,NaN,NaN,pearson,0.95,NaN,0.798292,0.029174,1754.0
1,auc_topk_redundancy,auc_topk,2000,NaN,NaN,pearson,0.90,NaN,0.783869,0.026097,1399.6



AUC redundancy [3/45]: {'base_k': 2000, 'method': 'pearson', 'corr_threshold': 0.97, 'max_features': None}


...

,experiment,selector,base_k,per_fold_k,final_k,method,corr_threshold,max_features,test_auroc_mean,test_auroc_std,n_features_mean
0,previous_best_stability_catboost,stability,3000,300.0,150.0,NaN,NaN,150.0,0.844963,0.019266,150.0
33,auc_topk_redundancy,auc_topk,5000,NaN,NaN,pearson,0.97,NaN,0.799944,0.030532,4697.4
43,auc_topk_redundancy,auc_topk,5000,NaN,NaN,cosine,0.97,NaN,0.799944,0.030532,4697.4
15,auc_topk_redundancy,auc_topk,2000,NaN,NaN,cosine,0.99,NaN,0.799389,0.021164,1937.6
5,auc_topk_redundancy,auc_topk,2000,NaN,NaN,pearson,0.99,NaN,0.799389,0.021164,1937.6
24,auc_topk_redundancy,auc_topk,3000,NaN,NaN,spearman,0.98,NaN,0.798885,0.024750,2803.2
12,auc_topk_redundancy,auc_topk,2000,NaN,NaN,cosine,0.95,NaN,0.798292,0.029174,1754.0
2,auc_topk_redundancy,auc_topk,2000,NaN,NaN,pearson,0.95,NaN,0.798292,0.029174,1754.0
34,auc_topk_redundancy,auc_topk,5000,NaN,NaN,pearson,0.98,NaN,0.797645,0.037723,4796.4
36,auc_topk_redundancy,auc_topk,5000,NaN,NaN,spearman,0.90,NaN,0.795613,0.021562,3864.4



AUC redundancy [44/45]: {'base_k': 5000, 'method': 'cosine', 'corr_threshold': 0.98, 'max_features': None}


,experiment,selector,base_k,per_fold_k,final_k,method,corr_threshold,max_features,test_auroc_mean,test_auroc_std,n_features_mean
0,previous_best_stability_catboost,stability,3000,300.0,150.0,NaN,NaN,150.0,0.844963,0.019266,150.0
43,auc_topk_redundancy,auc_topk,5000,NaN,NaN,cosine,0.97,NaN,0.799944,0.030532,4697.4
33,auc_topk_redundancy,auc_topk,5000,NaN,NaN,pearson,0.97,NaN,0.799944,0.030532,4697.4
5,auc_topk_redundancy,auc_topk,2000,NaN,NaN,pearson,0.99,NaN,0.799389,0.021164,1937.6
15,auc_topk_redundancy,auc_topk,2000,NaN,NaN,cosine,0.99,NaN,0.799389,0.021164,1937.6
24,auc_topk_redundancy,auc_topk,3000,NaN,NaN,spearman,0.98,NaN,0.798885,0.024750,2803.2
12,auc_topk_redundancy,auc_topk,2000,NaN,NaN,cosine,0.95,NaN,0.798292,0.029174,1754.0
2,auc_topk_redundancy,auc_topk,2000,NaN,NaN,pearson,0.95,NaN,0.798292,0.029174,1754.0
34,auc_topk_redundancy,auc_topk,5000,NaN,NaN,pearson,0.98,NaN,0.797645,0.037723,4796.4
44,auc_topk_redundancy,auc_topk,5000,NaN,NaN,cosine,0.98,NaN,0.797645,0.037723,4796.4



AUC redundancy [45/45]: {'base_k': 5000, 'method': 'cosine', 'corr_threshold': 0.99, 'max_features': None}


,experiment,selector,base_k,per_fold_k,final_k,method,corr_threshold,max_features,test_auroc_mean,test_auroc_std,n_features_mean
0,previous_best_stability_catboost,stability,3000,300.0,150.0,NaN,NaN,150.0,0.844963,0.019266,150.0
43,auc_topk_redundancy,auc_topk,5000,NaN,NaN,cosine,0.97,NaN,0.799944,0.030532,4697.4
33,auc_topk_redundancy,auc_topk,5000,NaN,NaN,pearson,0.97,NaN,0.799944,0.030532,4697.4
5,auc_topk_redundancy,auc_topk,2000,NaN,NaN,pearson,0.99,NaN,0.799389,0.021164,1937.6
15,auc_topk_redundancy,auc_topk,2000,NaN,NaN,cosine,0.99,NaN,0.799389,0.021164,1937.6
24,auc_topk_redundancy,auc_topk,3000,NaN,NaN,spearman,0.98,NaN,0.798885,0.024750,2803.2
2,auc_topk_redundancy,auc_topk,2000,NaN,NaN,pearson,0.95,NaN,0.798292,0.029174,1754.0
12,auc_topk_redundancy,auc_topk,2000,NaN,NaN,cosine,0.95,NaN,0.798292,0.029174,1754.0
44,auc_topk_redundancy,auc_topk,5000,NaN,NaN,cosine,0.98,NaN,0.797645,0.037723,4796.4
34,auc_topk_redundancy,auc_topk,5000,NaN,NaN,pearson,0.98,NaN,0.797645,0.037723,4796.4


In [84]:
stability_redundancy_configs = []

for method in ["pearson", "spearman", "cosine"]:
    for threshold in [0.90, 0.95, 0.97, 0.98, 0.99]:
        stability_redundancy_configs.append({
            "base_k": 3000,
            "per_fold_k": 300,
            "final_k": 150,
            "method": method,
            "corr_threshold": threshold,
            "max_features": None,
        })

print("Stability redundancy configs:", len(stability_redundancy_configs))

Stability redundancy configs: 15


In [85]:
for i, cfg in enumerate(stability_redundancy_configs, start=1):
    print(f"\nStability redundancy [{i}/{len(stability_redundancy_configs)}]: {cfg}")

    selector_fn = make_stability_redundancy_selector(
        base_k=cfg["base_k"],
        per_fold_k=cfg["per_fold_k"],
        final_k=cfg["final_k"],
        corr_threshold=cfg["corr_threshold"],
        method=cfg["method"],
        max_features=cfg["max_features"],
    )

    result = evaluate_catboost_with_selector(
        selector_fn=selector_fn,
        X=X_hidden_full,
        y=y,
        splits=splits,
        verbose=False,
    )

    redundancy_results.append({
        "experiment": "stability_redundancy",
        "selector": "stability",
        "base_k": cfg["base_k"],
        "per_fold_k": cfg["per_fold_k"],
        "final_k": cfg["final_k"],
        "method": cfg["method"],
        "corr_threshold": cfg["corr_threshold"],
        "max_features": cfg["max_features"],
        "test_auroc_mean": result["test_auroc_mean"],
        "test_auroc_std": result["test_auroc_std"],
        "n_features_mean": result["n_features_mean"],
    })

redundancy_df = pd.DataFrame(redundancy_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(redundancy_df.head(30))

best = redundancy_df.iloc[0]
print(
    f"BEST REDUNDANCY RESULT:\n"
    f"{best['experiment']} | selector={best['selector']} | method={best['method']}\n"
    f"base_k={best['base_k']}, per_fold_k={best['per_fold_k']}, final_k={best['final_k']}\n"
    f"threshold={best['corr_threshold']}, n_features_mean={best['n_features_mean']:.1f}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}"
)


Stability redundancy [1/15]: {'base_k': 3000, 'per_fold_k': 300, 'final_k': 150, 'method': 'pearson', 'corr_threshold': 0.9, 'max_features': None}
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5

Stability redundancy [2/15]: {'base_k': 3000, 'per_fold_k': 300, 'final_k': 150, 'method': 'pearson', 'corr_threshold': 0.95, 'max_features': None}
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5

Stability redundancy [3/15]: {'base_k': 3000, 'per_fold_k': 300, 'final_k': 150, 'method': 'pearson', 'corr_threshold': 0.97, 'max_features': None}
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5

Stability redundancy [4/15]: {'base_k': 3000, 'per_fold_k': 300, 'final_k': 150, 'method': 'pearson', 'corr_threshold': 0.98, 'max_features': None}
catboost fold 1
catboost fold 2
catboost fold 3
catboost fold 4
catboost fold 5

Stability redundancy [5/15]: {'base_k': 3000, 'per_fold_k': 300, 'final_k': 150, 'me

,experiment,selector,base_k,per_fold_k,final_k,method,corr_threshold,max_features,test_auroc_mean,test_auroc_std,n_features_mean
58,stability_redundancy,stability,3000,300.0,150.0,cosine,0.97,NaN,0.847273,0.021475,146.8
48,stability_redundancy,stability,3000,300.0,150.0,pearson,0.97,NaN,0.847273,0.021475,146.8
46,stability_redundancy,stability,3000,300.0,150.0,pearson,0.90,NaN,0.846581,0.023083,135.0
56,stability_redundancy,stability,3000,300.0,150.0,cosine,0.90,NaN,0.846581,0.023083,135.0
54,stability_redundancy,stability,3000,300.0,150.0,spearman,0.98,NaN,0.845002,0.023411,148.4
0,previous_best_stability_catboost,stability,3000,300.0,150.0,NaN,NaN,150.0,0.844963,0.019266,150.0
59,stability_redundancy,stability,3000,300.0,150.0,cosine,0.98,NaN,0.844762,0.022448,148.0
49,stability_redundancy,stability,3000,300.0,150.0,pearson,0.98,NaN,0.844762,0.022448,148.0
53,stability_redundancy,stability,3000,300.0,150.0,spearman,0.97,NaN,0.844676,0.020709,147.0
50,stability_redundancy,stability,3000,300.0,150.0,pearson,0.99,NaN,0.844359,0.021963,148.2


BEST REDUNDANCY RESULT:
stability_redundancy | selector=stability | method=cosine
base_k=3000, per_fold_k=300.0, final_k=150.0
threshold=0.97, n_features_mean=146.8
AUROC=0.8473 ± 0.0215


In [86]:
targeted_redundancy_results = []

targeted_configs = []

for method in ["pearson", "spearman", "cosine"]:
    for threshold in [0.95, 0.97, 0.98, 0.99]:
        for max_features in [100, 150, 200, 300, 500]:
            targeted_configs.append({
                "base_k": 3000,
                "method": method,
                "corr_threshold": threshold,
                "max_features": max_features,
            })

print("Targeted configs:", len(targeted_configs))

Targeted configs: 60


In [87]:
for i, cfg in enumerate(targeted_configs, start=1):
    print(f"\nTargeted redundancy [{i}/{len(targeted_configs)}]: {cfg}")

    selector_fn = make_auc_redundancy_selector(
        base_k=cfg["base_k"],
        corr_threshold=cfg["corr_threshold"],
        method=cfg["method"],
        max_features=cfg["max_features"],
        min_std=1e-6,
    )

    result = evaluate_catboost_with_selector(
        selector_fn=selector_fn,
        X=X_hidden_full,
        y=y,
        splits=splits,
        verbose=False,
    )

    targeted_redundancy_results.append({
        "experiment": "auc_topk_redundancy_max_features",
        "selector": "auc_topk",
        "base_k": cfg["base_k"],
        "method": cfg["method"],
        "corr_threshold": cfg["corr_threshold"],
        "max_features": cfg["max_features"],
        "test_auroc_mean": result["test_auroc_mean"],
        "test_auroc_std": result["test_auroc_std"],
        "n_features_mean": result["n_features_mean"],
    })

targeted_redundancy_df = pd.DataFrame(targeted_redundancy_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(targeted_redundancy_df.head(30))

full_redundancy_compare = pd.concat(
    [
        redundancy_df,
        targeted_redundancy_df,
    ],
    ignore_index=True,
).sort_values("test_auroc_mean", ascending=False)

display(full_redundancy_compare.head(30))


Targeted redundancy [1/60]: {'base_k': 3000, 'method': 'pearson', 'corr_threshold': 0.95, 'max_features': 100}

Targeted redundancy [2/60]: {'base_k': 3000, 'method': 'pearson', 'corr_threshold': 0.95, 'max_features': 150}

Targeted redundancy [3/60]: {'base_k': 3000, 'method': 'pearson', 'corr_threshold': 0.95, 'max_features': 200}

Targeted redundancy [4/60]: {'base_k': 3000, 'method': 'pearson', 'corr_threshold': 0.95, 'max_features': 300}

Targeted redundancy [5/60]: {'base_k': 3000, 'method': 'pearson', 'corr_threshold': 0.95, 'max_features': 500}

Targeted redundancy [6/60]: {'base_k': 3000, 'method': 'pearson', 'corr_threshold': 0.97, 'max_features': 100}

Targeted redundancy [7/60]: {'base_k': 3000, 'method': 'pearson', 'corr_threshold': 0.97, 'max_features': 150}

Targeted redundancy [8/60]: {'base_k': 3000, 'method': 'pearson', 'corr_threshold': 0.97, 'max_features': 200}

Targeted redundancy [9/60]: {'base_k': 3000, 'method': 'pearson', 'corr_threshold': 0.97, 'max_features

,experiment,selector,base_k,method,corr_threshold,max_features,test_auroc_mean,test_auroc_std,n_features_mean
33,auc_topk_redundancy_max_features,auc_topk,3000,spearman,0.98,300,0.792575,0.033699,300.0
53,auc_topk_redundancy_max_features,auc_topk,3000,cosine,0.98,300,0.792442,0.040241,300.0
13,auc_topk_redundancy_max_features,auc_topk,3000,pearson,0.98,300,0.792442,0.040241,300.0
44,auc_topk_redundancy_max_features,auc_topk,3000,cosine,0.95,500,0.790166,0.021233,500.0
4,auc_topk_redundancy_max_features,auc_topk,3000,pearson,0.95,500,0.790166,0.021233,500.0
38,auc_topk_redundancy_max_features,auc_topk,3000,spearman,0.99,300,0.789900,0.029536,300.0
52,auc_topk_redundancy_max_features,auc_topk,3000,cosine,0.98,200,0.788966,0.028351,200.0
12,auc_topk_redundancy_max_features,auc_topk,3000,pearson,0.98,200,0.788966,0.028351,200.0
41,auc_topk_redundancy_max_features,auc_topk,3000,cosine,0.95,150,0.788310,0.032372,150.0
1,auc_topk_redundancy_max_features,auc_topk,3000,pearson,0.95,150,0.788310,0.032372,150.0


,experiment,selector,base_k,per_fold_k,final_k,method,corr_threshold,max_features,test_auroc_mean,test_auroc_std,n_features_mean
0,stability_redundancy,stability,3000,300.0,150.0,cosine,0.97,NaN,0.847273,0.021475,146.8
1,stability_redundancy,stability,3000,300.0,150.0,pearson,0.97,NaN,0.847273,0.021475,146.8
2,stability_redundancy,stability,3000,300.0,150.0,pearson,0.90,NaN,0.846581,0.023083,135.0
3,stability_redundancy,stability,3000,300.0,150.0,cosine,0.90,NaN,0.846581,0.023083,135.0
4,stability_redundancy,stability,3000,300.0,150.0,spearman,0.98,NaN,0.845002,0.023411,148.4
5,previous_best_stability_catboost,stability,3000,300.0,150.0,NaN,NaN,150.0,0.844963,0.019266,150.0
6,stability_redundancy,stability,3000,300.0,150.0,cosine,0.98,NaN,0.844762,0.022448,148.0
7,stability_redundancy,stability,3000,300.0,150.0,pearson,0.98,NaN,0.844762,0.022448,148.0
8,stability_redundancy,stability,3000,300.0,150.0,spearman,0.97,NaN,0.844676,0.020709,147.0
9,stability_redundancy,stability,3000,300.0,150.0,pearson,0.99,NaN,0.844359,0.021963,148.2


## Weighted stability selection

In [ ]:
def build_weighted_stability_features(
    model_name,
    X,
    y,
    splits,
    base_k=3000,
    per_fold_k=300,
    final_k=150,
    weighting="importance",
    normalize_importance=True,
):
    """
    weighting:
    - "frequency": base frequency stability
    - "importance": sum of normalized model importances
    - "rank_inverse": sum 1 / rank
    - "hybrid": frequency + normalized importance
    """

    global_scores = np.zeros(X.shape[1], dtype=np.float64)
    frequency = np.zeros(X.shape[1], dtype=np.float64)

    for fold_id, (idx_train, idx_val, idx_test) in enumerate(splits, start=1):
        print(f"{model_name} weighted stability fold {fold_id}")

        base_idx = select_top_k_auc(
            X[idx_train],
            y[idx_train],
            k=base_k,
        )

        if model_name == "catboost":
            model = CatBoostClassifier(
                **BEST_CB_PARAMS,
                loss_function="Logloss",
                eval_metric="AUC",
                random_seed=42,
                verbose=False,
                allow_writing_files=False,
            )

            model.fit(
                X[idx_train][:, base_idx],
                y[idx_train],
                eval_set=(X[idx_val][:, base_idx], y[idx_val]),
                use_best_model=True,
            )

        elif model_name == "extratrees":
            model = ExtraTreesClassifier(
                **BEST_ET_PARAMS,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            )

            model.fit(X[idx_train][:, base_idx], y[idx_train])

        else:
            raise ValueError(model_name)

        importance = get_model_importance(model).astype(np.float64)

        if normalize_importance:
            importance = importance / (importance.sum() + 1e-12)

        local_order = np.argsort(importance)[::-1]
        local_top = local_order[:per_fold_k]
        selected_idx = base_idx[local_top]

        frequency[selected_idx] += 1.0

        if weighting == "frequency":
            global_scores[selected_idx] += 1.0

        elif weighting == "importance":
            global_scores[selected_idx] += importance[local_top]

        elif weighting == "rank_inverse":
            for rank, local_i in enumerate(local_top, start=1):
                global_scores[base_idx[local_i]] += 1.0 / rank

        elif weighting == "hybrid":
            global_scores[selected_idx] += 1.0 + importance[local_top]

        else:
            raise ValueError(weighting)

    selected = np.argsort(global_scores)[-final_k:][::-1]

    stats = pd.DataFrame({
        "feature_idx": np.where(global_scores > 0)[0],
        "weighted_score": global_scores[global_scores > 0],
        "frequency": frequency[global_scores > 0],
    }).sort_values("weighted_score", ascending=False)

    selected_by_fold = [selected for _ in splits]

    return selected_by_fold, stats

In [89]:
def build_weighted_stability_redundancy_selector(
    model_name,
    X,
    y,
    splits,
    base_k=3000,
    per_fold_k=300,
    final_k=150,
    weighting="importance",
    corr_threshold=0.97,
    corr_method="pearson",
):
    selected_by_fold, stats = build_weighted_stability_features(
        model_name=model_name,
        X=X,
        y=y,
        splits=splits,
        base_k=base_k,
        per_fold_k=per_fold_k,
        final_k=final_k,
        weighting=weighting,
    )

    selected_idx = selected_by_fold[0]

    weighted_scores = np.zeros(X.shape[1], dtype=np.float64)
    weighted_scores[stats["feature_idx"].to_numpy(dtype=int)] = stats["weighted_score"].to_numpy()

    def selector_fn(X, y, idx_train, idx_val, idx_test, fold_id):
        pruned_idx = correlation_prune_features(
            X_train=X[idx_train],
            feature_idx=selected_idx,
            feature_scores=weighted_scores,
            threshold=corr_threshold,
            method=corr_method,
            max_features=None,
        )
        return pruned_idx

    return selector_fn, stats

In [93]:
weighted_redundancy_results = []

weighted_configs = []

for weighting in ["importance", "rank_inverse", "hybrid"]:
    for final_k in [100, 150, 200]:
        weighted_configs.append({
            "weighting": weighting,
            "base_k": 3000,
            "per_fold_k": 300,
            "final_k": final_k,
            "corr_method": "pearson",
            "corr_threshold": 0.97,
        })

print("Total weighted configs:", len(weighted_configs))

Total weighted configs: 9


In [94]:
for i, cfg in enumerate(weighted_configs, start=1):
    print(f"\n[{i}/{len(weighted_configs)}] Weighted stability + redundancy: {cfg}")

    selector_fn, stats = build_weighted_stability_redundancy_selector(
        model_name="catboost",
        X=X_hidden_full,
        y=y,
        splits=splits,
        base_k=cfg["base_k"],
        per_fold_k=cfg["per_fold_k"],
        final_k=cfg["final_k"],
        weighting=cfg["weighting"],
        corr_threshold=cfg["corr_threshold"],
        corr_method=cfg["corr_method"],
    )

    result = evaluate_catboost_with_selector(
        selector_fn=selector_fn,
        X=X_hidden_full,
        y=y,
        splits=splits,
        verbose=False,
    )

    weighted_redundancy_results.append({
        "experiment": "weighted_stability_redundancy",
        **cfg,
        "test_auroc_mean": result["test_auroc_mean"],
        "test_auroc_std": result["test_auroc_std"],
        "n_features_mean": result["n_features_mean"],
        "n_candidates_seen": len(stats),
        "top_frequency": stats["frequency"].max(),
    })

    current = pd.DataFrame(weighted_redundancy_results).sort_values(
        "test_auroc_mean",
        ascending=False,
    )

    display(current.head(10))


[1/9] Weighted stability + redundancy: {'weighting': 'importance', 'base_k': 3000, 'per_fold_k': 300, 'final_k': 100, 'corr_method': 'pearson', 'corr_threshold': 0.97}
catboost weighted stability fold 1
catboost weighted stability fold 2
catboost weighted stability fold 3
catboost weighted stability fold 4
catboost weighted stability fold 5


,experiment,weighting,base_k,per_fold_k,final_k,corr_method,corr_threshold,test_auroc_mean,test_auroc_std,n_features_mean,n_candidates_seen,top_frequency
0,weighted_stability_redundancy,importance,3000,300,100,pearson,0.97,0.843303,0.022153,98.8,971,5.0



[2/9] Weighted stability + redundancy: {'weighting': 'importance', 'base_k': 3000, 'per_fold_k': 300, 'final_k': 150, 'corr_method': 'pearson', 'corr_threshold': 0.97}
catboost weighted stability fold 1
catboost weighted stability fold 2
catboost weighted stability fold 3
catboost weighted stability fold 4
catboost weighted stability fold 5


,experiment,weighting,base_k,per_fold_k,final_k,corr_method,corr_threshold,test_auroc_mean,test_auroc_std,n_features_mean,n_candidates_seen,top_frequency
0,weighted_stability_redundancy,importance,3000,300,100,pearson,0.97,0.843303,0.022153,98.8,971,5.0
1,weighted_stability_redundancy,importance,3000,300,150,pearson,0.97,0.838458,0.021405,146.8,971,5.0



[3/9] Weighted stability + redundancy: {'weighting': 'importance', 'base_k': 3000, 'per_fold_k': 300, 'final_k': 200, 'corr_method': 'pearson', 'corr_threshold': 0.97}
catboost weighted stability fold 1
catboost weighted stability fold 2
catboost weighted stability fold 3
catboost weighted stability fold 4
catboost weighted stability fold 5


...

,experiment,weighting,base_k,per_fold_k,final_k,corr_method,corr_threshold,test_auroc_mean,test_auroc_std,n_features_mean,n_candidates_seen,top_frequency
0,weighted_stability_redundancy,importance,3000,300,100,pearson,0.97,0.843303,0.022153,98.8,971,5.0
6,weighted_stability_redundancy,hybrid,3000,300,100,pearson,0.97,0.840855,0.015946,98.8,1146,5.0
4,weighted_stability_redundancy,rank_inverse,3000,300,150,pearson,0.97,0.840505,0.013150,147.8,1146,5.0
3,weighted_stability_redundancy,rank_inverse,3000,300,100,pearson,0.97,0.840259,0.021034,98.8,1146,5.0
2,weighted_stability_redundancy,importance,3000,300,200,pearson,0.97,0.838620,0.021524,195.8,971,5.0
1,weighted_stability_redundancy,importance,3000,300,150,pearson,0.97,0.838458,0.021405,146.8,971,5.0
5,weighted_stability_redundancy,rank_inverse,3000,300,200,pearson,0.97,0.837250,0.013046,196.8,1146,5.0



[8/9] Weighted stability + redundancy: {'weighting': 'hybrid', 'base_k': 3000, 'per_fold_k': 300, 'final_k': 150, 'corr_method': 'pearson', 'corr_threshold': 0.97}
catboost weighted stability fold 1
catboost weighted stability fold 2
catboost weighted stability fold 3
catboost weighted stability fold 4
catboost weighted stability fold 5


,experiment,weighting,base_k,per_fold_k,final_k,corr_method,corr_threshold,test_auroc_mean,test_auroc_std,n_features_mean,n_candidates_seen,top_frequency
7,weighted_stability_redundancy,hybrid,3000,300,150,pearson,0.97,0.843353,0.015306,146.8,1146,5.0
0,weighted_stability_redundancy,importance,3000,300,100,pearson,0.97,0.843303,0.022153,98.8,971,5.0
6,weighted_stability_redundancy,hybrid,3000,300,100,pearson,0.97,0.840855,0.015946,98.8,1146,5.0
4,weighted_stability_redundancy,rank_inverse,3000,300,150,pearson,0.97,0.840505,0.013150,147.8,1146,5.0
3,weighted_stability_redundancy,rank_inverse,3000,300,100,pearson,0.97,0.840259,0.021034,98.8,1146,5.0
2,weighted_stability_redundancy,importance,3000,300,200,pearson,0.97,0.838620,0.021524,195.8,971,5.0
1,weighted_stability_redundancy,importance,3000,300,150,pearson,0.97,0.838458,0.021405,146.8,971,5.0
5,weighted_stability_redundancy,rank_inverse,3000,300,200,pearson,0.97,0.837250,0.013046,196.8,1146,5.0



[9/9] Weighted stability + redundancy: {'weighting': 'hybrid', 'base_k': 3000, 'per_fold_k': 300, 'final_k': 200, 'corr_method': 'pearson', 'corr_threshold': 0.97}
catboost weighted stability fold 1
catboost weighted stability fold 2
catboost weighted stability fold 3
catboost weighted stability fold 4
catboost weighted stability fold 5


,experiment,weighting,base_k,per_fold_k,final_k,corr_method,corr_threshold,test_auroc_mean,test_auroc_std,n_features_mean,n_candidates_seen,top_frequency
7,weighted_stability_redundancy,hybrid,3000,300,150,pearson,0.97,0.843353,0.015306,146.8,1146,5.0
0,weighted_stability_redundancy,importance,3000,300,100,pearson,0.97,0.843303,0.022153,98.8,971,5.0
8,weighted_stability_redundancy,hybrid,3000,300,200,pearson,0.97,0.842104,0.020987,191.8,1146,5.0
6,weighted_stability_redundancy,hybrid,3000,300,100,pearson,0.97,0.840855,0.015946,98.8,1146,5.0
4,weighted_stability_redundancy,rank_inverse,3000,300,150,pearson,0.97,0.840505,0.013150,147.8,1146,5.0
3,weighted_stability_redundancy,rank_inverse,3000,300,100,pearson,0.97,0.840259,0.021034,98.8,1146,5.0
2,weighted_stability_redundancy,importance,3000,300,200,pearson,0.97,0.838620,0.021524,195.8,971,5.0
1,weighted_stability_redundancy,importance,3000,300,150,pearson,0.97,0.838458,0.021405,146.8,971,5.0
5,weighted_stability_redundancy,rank_inverse,3000,300,200,pearson,0.97,0.837250,0.013046,196.8,1146,5.0


In [95]:
weighted_redundancy_df = pd.DataFrame(weighted_redundancy_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

previous_best = pd.DataFrame([
    {
        "experiment": "previous_best_stability_redundancy",
        "weighting": "frequency",
        "base_k": 3000,
        "per_fold_k": 300,
        "final_k": 150,
        "corr_method": "pearson/cosine",
        "corr_threshold": 0.97,
        "test_auroc_mean": 0.847273,
        "test_auroc_std": 0.021475,
        "n_features_mean": 146.8,
    }
])

compare_weighted = pd.concat(
    [previous_best, weighted_redundancy_df],
    ignore_index=True,
).sort_values("test_auroc_mean", ascending=False)

display(compare_weighted.head(30))

best = compare_weighted.iloc[0]
print(
    f"BEST WEIGHTED STABILITY RESULT:\n"
    f"{best['experiment']}\n"
    f"weighting={best['weighting']}, base_k={best['base_k']}, "
    f"per_fold_k={best['per_fold_k']}, final_k={best['final_k']}\n"
    f"corr={best['corr_method']} @ {best['corr_threshold']}\n"
    f"n_features={best['n_features_mean']:.1f}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}"
)

,experiment,weighting,base_k,per_fold_k,final_k,corr_method,corr_threshold,test_auroc_mean,test_auroc_std,n_features_mean,n_candidates_seen,top_frequency
0,previous_best_stability_redundancy,frequency,3000,300,150,pearson/cosine,0.97,0.847273,0.021475,146.8,NaN,NaN
1,weighted_stability_redundancy,hybrid,3000,300,150,pearson,0.97,0.843353,0.015306,146.8,1146.0,5.0
2,weighted_stability_redundancy,importance,3000,300,100,pearson,0.97,0.843303,0.022153,98.8,971.0,5.0
3,weighted_stability_redundancy,hybrid,3000,300,200,pearson,0.97,0.842104,0.020987,191.8,1146.0,5.0
4,weighted_stability_redundancy,hybrid,3000,300,100,pearson,0.97,0.840855,0.015946,98.8,1146.0,5.0
5,weighted_stability_redundancy,rank_inverse,3000,300,150,pearson,0.97,0.840505,0.013150,147.8,1146.0,5.0
6,weighted_stability_redundancy,rank_inverse,3000,300,100,pearson,0.97,0.840259,0.021034,98.8,1146.0,5.0
7,weighted_stability_redundancy,importance,3000,300,200,pearson,0.97,0.838620,0.021524,195.8,971.0,5.0
8,weighted_stability_redundancy,importance,3000,300,150,pearson,0.97,0.838458,0.021405,146.8,971.0,5.0
9,weighted_stability_redundancy,rank_inverse,3000,300,200,pearson,0.97,0.837250,0.013046,196.8,1146.0,5.0


BEST WEIGHTED STABILITY RESULT:
previous_best_stability_redundancy
weighting=frequency, base_k=3000, per_fold_k=300, final_k=150
corr=pearson/cosine @ 0.97
n_features=146.8
AUROC=0.8473 ± 0.0215


## Feature ablations

In [96]:
feature_names = np.array(hidden_feature_cols)

def get_feature_group_indices(feature_names):
    groups = {}

    groups["mean_only"] = np.where(
        np.char.startswith(feature_names.astype(str), "mean_response_")
    )[0]

    groups["drift_only"] = np.where(
        np.char.find(feature_names.astype(str), "response_drift") >= 0
    )[0]

    groups["response_minus_prompt_only"] = np.where(
        np.char.find(feature_names.astype(str), "response_minus_prompt") >= 0
    )[0]

    groups["mean_plus_drift"] = np.union1d(
        groups["mean_only"],
        groups["drift_only"],
    )

    groups["drift_plus_response_minus_prompt"] = np.union1d(
        groups["drift_only"],
        groups["response_minus_prompt_only"],
    )

    raw_mean_mask = np.char.startswith(feature_names.astype(str), "mean_response_")
    groups["without_raw_mean_response"] = np.where(~raw_mean_mask)[0]

    return groups


feature_groups = get_feature_group_indices(feature_names)

for name, idx in feature_groups.items():
    print(f"{name}: {len(idx)} features")

mean_only: 6272 features
drift_only: 8960 features
response_minus_prompt_only: 5376 features
mean_plus_drift: 15232 features
drift_plus_response_minus_prompt: 14336 features
without_raw_mean_response: 14606 features


In [97]:
def select_top_k_auc_from_allowed(X_train, y_train, allowed_idx, k=3000):
    allowed_idx = np.asarray(allowed_idx)

    X_allowed = X_train[:, allowed_idx]
    scores = rank_features_auc_scores(X_allowed, y_train)

    k = min(k, len(allowed_idx))
    local_top = np.argsort(scores)[-k:][::-1]

    return allowed_idx[local_top]


def build_importance_selected_features_grouped(
    model_name,
    X,
    y,
    splits,
    allowed_idx,
    base_k=3000,
    final_k=300,
):
    selected_by_fold = []

    for fold_id, (idx_train, idx_val, idx_test) in enumerate(splits):
        print(f"{model_name} grouped fold {fold_id + 1}")

        base_idx = select_top_k_auc_from_allowed(
            X_train=X[idx_train],
            y_train=y[idx_train],
            allowed_idx=allowed_idx,
            k=base_k,
        )

        if model_name == "catboost":
            model = CatBoostClassifier(
                **BEST_CB_PARAMS,
                loss_function="Logloss",
                eval_metric="AUC",
                random_seed=42,
                verbose=False,
                allow_writing_files=False,
            )

            model.fit(
                X[idx_train][:, base_idx],
                y[idx_train],
                eval_set=(X[idx_val][:, base_idx], y[idx_val]),
                use_best_model=True,
            )
        else:
            raise ValueError(model_name)

        importance = get_model_importance(model)
        local_top = np.argsort(importance)[-final_k:][::-1]
        selected_by_fold.append(base_idx[local_top])

    return selected_by_fold


def build_stability_selected_features_grouped(
    X,
    y,
    splits,
    allowed_idx,
    base_k=3000,
    per_fold_k=300,
    final_k=150,
):
    fold_selected = build_importance_selected_features_grouped(
        model_name="catboost",
        X=X,
        y=y,
        splits=splits,
        allowed_idx=allowed_idx,
        base_k=base_k,
        final_k=per_fold_k,
    )

    all_selected = []
    for arr in fold_selected:
        all_selected.extend(arr.tolist())

    counts = pd.Series(all_selected).value_counts()
    stable_idx = counts.head(final_k).index.to_numpy(dtype=int)

    return [stable_idx for _ in splits], counts

In [98]:
def make_group_stability_redundancy_selector(
    allowed_idx,
    base_k=3000,
    per_fold_k=300,
    final_k=150,
    corr_threshold=0.97,
    corr_method="pearson",
):
    selected_by_fold, counts = build_stability_selected_features_grouped(
        X=X_hidden_full,
        y=y,
        splits=splits,
        allowed_idx=allowed_idx,
        base_k=base_k,
        per_fold_k=per_fold_k,
        final_k=final_k,
    )

    stable_idx = selected_by_fold[0]

    stability_scores = np.zeros(X_hidden_full.shape[1], dtype=np.float32)
    for feat, count in counts.items():
        stability_scores[int(feat)] = float(count)

    def selector_fn(X, y, idx_train, idx_val, idx_test, fold_id):
        pruned_idx = correlation_prune_features(
            X_train=X[idx_train],
            feature_idx=stable_idx,
            feature_scores=stability_scores,
            threshold=corr_threshold,
            method=corr_method,
            max_features=None,
        )

        return pruned_idx

    return selector_fn, counts

In [99]:
ablation_results = []

for group_name, allowed_idx in feature_groups.items():
    print("\n" + "=" * 80)
    print(f"GROUP ABLATION: {group_name}")
    print(f"Allowed features: {len(allowed_idx)}")
    print("=" * 80)

    if len(allowed_idx) == 0:
        print("Skipping empty group")
        continue

    selector_fn, counts = make_group_stability_redundancy_selector(
        allowed_idx=allowed_idx,
        base_k=3000,
        per_fold_k=300,
        final_k=150,
        corr_threshold=0.97,
        corr_method="pearson",
    )

    result = evaluate_catboost_with_selector(
        selector_fn=selector_fn,
        X=X_hidden_full,
        y=y,
        splits=splits,
        verbose=False,
    )

    ablation_results.append({
        "group": group_name,
        "allowed_features": len(allowed_idx),
        "selected_candidates": len(counts),
        "n_features_mean": result["n_features_mean"],
        "test_auroc_mean": result["test_auroc_mean"],
        "test_auroc_std": result["test_auroc_std"],
    })

ablation_df = pd.DataFrame(ablation_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(ablation_df)


GROUP ABLATION: mean_only
Allowed features: 6272
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

GROUP ABLATION: drift_only
Allowed features: 8960
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

GROUP ABLATION: response_minus_prompt_only
Allowed features: 5376
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

GROUP ABLATION: mean_plus_drift
Allowed features: 15232
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

GROUP ABLATION: drift_plus_response_minus_prompt
Allowed features: 14336
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

GROUP ABLATION: without_raw_mean_response
Allowed features: 14606
catboost grouped fold 1
catboost g

,group,allowed_features,selected_candidates,n_features_mean,test_auroc_mean,test_auroc_std
1,drift_only,8960,1096,146.0,0.848830,0.034777
3,mean_plus_drift,15232,1159,146.8,0.842670,0.020044
5,without_raw_mean_response,14606,1172,149.0,0.841250,0.030330
4,drift_plus_response_minus_prompt,14336,1190,146.8,0.830498,0.039439
2,response_minus_prompt_only,5376,1140,148.0,0.827543,0.029629
0,mean_only,6272,1116,148.0,0.810272,0.026107


In [100]:
current_best = pd.DataFrame([
    {
        "group": "current_best_all_features",
        "allowed_features": len(hidden_feature_cols),
        "selected_candidates": None,
        "n_features_mean": 146.8,
        "test_auroc_mean": 0.847273,
        "test_auroc_std": 0.021475,
    }
])

ablation_compare = pd.concat(
    [current_best, ablation_df],
    ignore_index=True,
).sort_values("test_auroc_mean", ascending=False)

display(ablation_compare)

best = ablation_compare.iloc[0]

print(
    f"BEST GROUP SETUP:\n"
    f"{best['group']}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}\n"
    f"n_features_mean={best['n_features_mean']}"
)

,group,allowed_features,selected_candidates,n_features_mean,test_auroc_mean,test_auroc_std
1,drift_only,8960,1096,146.0,0.848830,0.034777
0,current_best_all_features,20878,None,146.8,0.847273,0.021475
2,mean_plus_drift,15232,1159,146.8,0.842670,0.020044
3,without_raw_mean_response,14606,1172,149.0,0.841250,0.030330
4,drift_plus_response_minus_prompt,14336,1190,146.8,0.830498,0.039439
5,response_minus_prompt_only,5376,1140,148.0,0.827543,0.029629
6,mean_only,6272,1116,148.0,0.810272,0.026107


BEST GROUP SETUP:
drift_only
AUROC=0.8488 ± 0.0348
n_features_mean=146.0


## Drift-only tuning as it's the best

In [102]:
drift_allowed_idx = feature_groups["drift_only"]

print("drift_only features:", len(drift_allowed_idx))

drift_only features: 8960


In [ ]:
drift_tuning_results = []

drift_grid = [
    {"base_k": 2000, "per_fold_k": 200, "final_k": 100, "corr_threshold": 0.97, "corr_method": "pearson"},
    {"base_k": 2000, "per_fold_k": 300, "final_k": 100, "corr_threshold": 0.97, "corr_method": "pearson"},
    {"base_k": 2000, "per_fold_k": 300, "final_k": 150, "corr_threshold": 0.97, "corr_method": "pearson"},

    {"base_k": 3000, "per_fold_k": 200, "final_k": 100, "corr_threshold": 0.97, "corr_method": "pearson"},
    {"base_k": 3000, "per_fold_k": 300, "final_k": 100, "corr_threshold": 0.97, "corr_method": "pearson"},
    {"base_k": 3000, "per_fold_k": 300, "final_k": 150, "corr_threshold": 0.97, "corr_method": "pearson"},
    {"base_k": 3000, "per_fold_k": 500, "final_k": 150, "corr_threshold": 0.97, "corr_method": "pearson"},
    {"base_k": 3000, "per_fold_k": 500, "final_k": 200, "corr_threshold": 0.97, "corr_method": "pearson"},

    {"base_k": 3000, "per_fold_k": 300, "final_k": 150, "corr_threshold": 0.95, "corr_method": "pearson"},
    {"base_k": 3000, "per_fold_k": 300, "final_k": 150, "corr_threshold": 0.98, "corr_method": "pearson"},

    {"base_k": 3000, "per_fold_k": 300, "final_k": 150, "corr_threshold": 0.97, "corr_method": "cosine"},
    {"base_k": 3000, "per_fold_k": 300, "final_k": 150, "corr_threshold": 0.97, "corr_method": "spearman"},
]

print("Total drift configs:", len(drift_grid))

Total drift configs: 12


In [106]:
for i, cfg in enumerate(drift_grid, start=1):
    print(f"\n[{i}/{len(drift_grid)}] Drift-only tuning: {cfg}")

    selector_fn, counts = make_group_stability_redundancy_selector(
        allowed_idx=drift_allowed_idx,
        base_k=cfg["base_k"],
        per_fold_k=cfg["per_fold_k"],
        final_k=cfg["final_k"],
        corr_threshold=cfg["corr_threshold"],
        corr_method=cfg["corr_method"],
    )

    result = evaluate_catboost_with_selector(
        selector_fn=selector_fn,
        X=X_hidden_full,
        y=y,
        splits=splits,
        verbose=False,
    )

    drift_tuning_results.append({
        "group": "drift_only",
        **cfg,
        "selected_candidates": len(counts),
        "n_features_mean": result["n_features_mean"],
        "test_auroc_mean": result["test_auroc_mean"],
        "test_auroc_std": result["test_auroc_std"],
    })

    if i % 5 == 0:
        current = pd.DataFrame(drift_tuning_results).sort_values(
            "test_auroc_mean",
            ascending=False,
        )
        display(current.head(10))


[1/12] Drift-only tuning: {'base_k': 2000, 'per_fold_k': 200, 'final_k': 100, 'corr_threshold': 0.97, 'corr_method': 'pearson'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

[2/12] Drift-only tuning: {'base_k': 2000, 'per_fold_k': 300, 'final_k': 100, 'corr_threshold': 0.97, 'corr_method': 'pearson'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

[3/12] Drift-only tuning: {'base_k': 2000, 'per_fold_k': 300, 'final_k': 150, 'corr_threshold': 0.97, 'corr_method': 'pearson'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

[4/12] Drift-only tuning: {'base_k': 3000, 'per_fold_k': 200, 'final_k': 100, 'corr_threshold': 0.97, 'corr_method': 'pearson'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

[5/

,group,base_k,per_fold_k,final_k,corr_threshold,corr_method,selected_candidates,n_features_mean,test_auroc_mean,test_auroc_std
3,drift_only,3000,200,100,0.97,pearson,754,98.8,0.851181,0.030940
4,drift_only,3000,300,100,0.97,pearson,1096,97.8,0.850971,0.035608
1,drift_only,2000,300,100,0.97,pearson,1081,99.8,0.844401,0.025989
0,drift_only,2000,200,100,0.97,pearson,759,99.6,0.844246,0.023491
2,drift_only,2000,300,150,0.97,pearson,1081,149.6,0.836510,0.026077



[6/12] Drift-only tuning: {'base_k': 3000, 'per_fold_k': 300, 'final_k': 150, 'corr_threshold': 0.97, 'corr_method': 'pearson'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

[7/12] Drift-only tuning: {'base_k': 3000, 'per_fold_k': 500, 'final_k': 150, 'corr_threshold': 0.97, 'corr_method': 'pearson'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

[8/12] Drift-only tuning: {'base_k': 3000, 'per_fold_k': 500, 'final_k': 200, 'corr_threshold': 0.97, 'corr_method': 'pearson'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

[9/12] Drift-only tuning: {'base_k': 3000, 'per_fold_k': 300, 'final_k': 150, 'corr_threshold': 0.95, 'corr_method': 'pearson'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

[10

,group,base_k,per_fold_k,final_k,corr_threshold,corr_method,selected_candidates,n_features_mean,test_auroc_mean,test_auroc_std
9,drift_only,3000,300,150,0.98,pearson,1096,146.6,0.852435,0.036809
3,drift_only,3000,200,100,0.97,pearson,754,98.8,0.851181,0.030940
4,drift_only,3000,300,100,0.97,pearson,1096,97.8,0.850971,0.035608
5,drift_only,3000,300,150,0.97,pearson,1096,146.0,0.848830,0.034777
1,drift_only,2000,300,100,0.97,pearson,1081,99.8,0.844401,0.025989
0,drift_only,2000,200,100,0.97,pearson,759,99.6,0.844246,0.023491
8,drift_only,3000,300,150,0.95,pearson,1096,144.0,0.843096,0.032140
7,drift_only,3000,500,200,0.97,pearson,1728,194.2,0.841675,0.027285
6,drift_only,3000,500,150,0.97,pearson,1728,146.2,0.836618,0.022123
2,drift_only,2000,300,150,0.97,pearson,1081,149.6,0.836510,0.026077



[11/12] Drift-only tuning: {'base_k': 3000, 'per_fold_k': 300, 'final_k': 150, 'corr_threshold': 0.97, 'corr_method': 'cosine'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

[12/12] Drift-only tuning: {'base_k': 3000, 'per_fold_k': 300, 'final_k': 150, 'corr_threshold': 0.97, 'corr_method': 'spearman'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5


In [107]:
drift_tuning_df = pd.DataFrame(drift_tuning_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(drift_tuning_df.head(30))

current_best_all = pd.DataFrame([
    {
        "group": "current_best_all_features",
        "base_k": 3000,
        "per_fold_k": 300,
        "final_k": 150,
        "corr_threshold": 0.97,
        "corr_method": "pearson",
        "selected_candidates": None,
        "n_features_mean": 146.8,
        "test_auroc_mean": 0.847273,
        "test_auroc_std": 0.021475,
    },
    {
        "group": "previous_drift_only",
        "base_k": 3000,
        "per_fold_k": 300,
        "final_k": 150,
        "corr_threshold": 0.97,
        "corr_method": "pearson",
        "selected_candidates": 1096,
        "n_features_mean": 146.0,
        "test_auroc_mean": 0.848830,
        "test_auroc_std": 0.034777,
    }
])

drift_compare = pd.concat(
    [current_best_all, drift_tuning_df],
    ignore_index=True,
).sort_values("test_auroc_mean", ascending=False)

display(drift_compare.head(30))

best = drift_compare.iloc[0]

print(
    f"BEST DRIFT SETUP:\n"
    f"group={best['group']}\n"
    f"base_k={best['base_k']}, per_fold_k={best['per_fold_k']}, "
    f"final_k={best['final_k']}, corr={best['corr_threshold']}\n"
    f"n_features={best['n_features_mean']:.1f}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}"
)

,group,base_k,per_fold_k,final_k,corr_threshold,corr_method,selected_candidates,n_features_mean,test_auroc_mean,test_auroc_std
9,drift_only,3000,300,150,0.98,pearson,1096,146.6,0.852435,0.036809
3,drift_only,3000,200,100,0.97,pearson,754,98.8,0.851181,0.030940
4,drift_only,3000,300,100,0.97,pearson,1096,97.8,0.850971,0.035608
11,drift_only,3000,300,150,0.97,spearman,1096,143.6,0.849376,0.037394
5,drift_only,3000,300,150,0.97,pearson,1096,146.0,0.848830,0.034777
10,drift_only,3000,300,150,0.97,cosine,1096,146.0,0.848830,0.034777
1,drift_only,2000,300,100,0.97,pearson,1081,99.8,0.844401,0.025989
0,drift_only,2000,200,100,0.97,pearson,759,99.6,0.844246,0.023491
8,drift_only,3000,300,150,0.95,pearson,1096,144.0,0.843096,0.032140
7,drift_only,3000,500,200,0.97,pearson,1728,194.2,0.841675,0.027285


,group,base_k,per_fold_k,final_k,corr_threshold,corr_method,selected_candidates,n_features_mean,test_auroc_mean,test_auroc_std
2,drift_only,3000,300,150,0.98,pearson,1096.0,146.6,0.852435,0.036809
3,drift_only,3000,200,100,0.97,pearson,754.0,98.8,0.851181,0.030940
4,drift_only,3000,300,100,0.97,pearson,1096.0,97.8,0.850971,0.035608
5,drift_only,3000,300,150,0.97,spearman,1096.0,143.6,0.849376,0.037394
1,previous_drift_only,3000,300,150,0.97,pearson,1096.0,146.0,0.848830,0.034777
6,drift_only,3000,300,150,0.97,pearson,1096.0,146.0,0.848830,0.034777
7,drift_only,3000,300,150,0.97,cosine,1096.0,146.0,0.848830,0.034777
0,current_best_all_features,3000,300,150,0.97,pearson,NaN,146.8,0.847273,0.021475
8,drift_only,2000,300,100,0.97,pearson,1081.0,99.8,0.844401,0.025989
9,drift_only,2000,200,100,0.97,pearson,759.0,99.6,0.844246,0.023491


BEST DRIFT SETUP:
group=drift_only
base_k=3000, per_fold_k=300, final_k=150, corr=0.98
n_features=146.6
AUROC=0.8524 ± 0.0368


In [109]:
## checking near best params

drift_fine_grid = [
    {"base_k": 3000, "per_fold_k": 300, "final_k": 125, "corr_threshold": 0.98, "corr_method": "pearson"},
    {"base_k": 3000, "per_fold_k": 300, "final_k": 150, "corr_threshold": 0.98, "corr_method": "pearson"},
    {"base_k": 3000, "per_fold_k": 300, "final_k": 175, "corr_threshold": 0.98, "corr_method": "pearson"},

    {"base_k": 3000, "per_fold_k": 250, "final_k": 125, "corr_threshold": 0.98, "corr_method": "pearson"},
    {"base_k": 3000, "per_fold_k": 350, "final_k": 150, "corr_threshold": 0.98, "corr_method": "pearson"},
    {"base_k": 3000, "per_fold_k": 400, "final_k": 150, "corr_threshold": 0.98, "corr_method": "pearson"},

    {"base_k": 2500, "per_fold_k": 300, "final_k": 150, "corr_threshold": 0.98, "corr_method": "pearson"},
    {"base_k": 3500, "per_fold_k": 300, "final_k": 150, "corr_threshold": 0.98, "corr_method": "pearson"},

    {"base_k": 3000, "per_fold_k": 300, "final_k": 150, "corr_threshold": 0.975, "corr_method": "pearson"},
    {"base_k": 3000, "per_fold_k": 300, "final_k": 150, "corr_threshold": 0.985, "corr_method": "pearson"},
    {"base_k": 3000, "per_fold_k": 300, "final_k": 150, "corr_threshold": 0.99, "corr_method": "pearson"},

    {"base_k": 3500, "per_fold_k": 350, "final_k": 150, "corr_threshold": 0.98, "corr_method": "pearson"},
]

In [110]:
drift_fine_results = []

for i, cfg in enumerate(drift_fine_grid, start=1):
    print(f"\n[{i}/{len(drift_fine_grid)}] Drift fine tuning: {cfg}")

    selector_fn, counts = make_group_stability_redundancy_selector(
        allowed_idx=drift_allowed_idx,
        base_k=cfg["base_k"],
        per_fold_k=cfg["per_fold_k"],
        final_k=cfg["final_k"],
        corr_threshold=cfg["corr_threshold"],
        corr_method=cfg["corr_method"],
    )

    result = evaluate_catboost_with_selector(
        selector_fn=selector_fn,
        X=X_hidden_full,
        y=y,
        splits=splits,
        verbose=False,
    )

    drift_fine_results.append({
        "group": "drift_only",
        **cfg,
        "selected_candidates": len(counts),
        "n_features_mean": result["n_features_mean"],
        "test_auroc_mean": result["test_auroc_mean"],
        "test_auroc_std": result["test_auroc_std"],
    })

drift_fine_df = pd.DataFrame(drift_fine_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(drift_fine_df)


[1/12] Drift fine tuning: {'base_k': 3000, 'per_fold_k': 300, 'final_k': 125, 'corr_threshold': 0.98, 'corr_method': 'pearson'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

[2/12] Drift fine tuning: {'base_k': 3000, 'per_fold_k': 300, 'final_k': 150, 'corr_threshold': 0.98, 'corr_method': 'pearson'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

[3/12] Drift fine tuning: {'base_k': 3000, 'per_fold_k': 300, 'final_k': 175, 'corr_threshold': 0.98, 'corr_method': 'pearson'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

[4/12] Drift fine tuning: {'base_k': 3000, 'per_fold_k': 250, 'final_k': 125, 'corr_threshold': 0.98, 'corr_method': 'pearson'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

[5/

,group,base_k,per_fold_k,final_k,corr_threshold,corr_method,selected_candidates,n_features_mean,test_auroc_mean,test_auroc_std
11,drift_only,3500,350,150,0.980,pearson,1331,149.8,0.862582,0.037949
6,drift_only,2500,300,150,0.980,pearson,1086,149.0,0.855774,0.026851
1,drift_only,3000,300,150,0.980,pearson,1096,146.6,0.852435,0.036809
9,drift_only,3000,300,150,0.985,pearson,1096,147.2,0.850110,0.037843
8,drift_only,3000,300,150,0.975,pearson,1096,146.0,0.848830,0.034777
7,drift_only,3500,300,150,0.980,pearson,1178,149.8,0.848577,0.035992
10,drift_only,3000,300,150,0.990,pearson,1096,147.8,0.848134,0.031052
3,drift_only,3000,250,125,0.980,pearson,924,122.2,0.848066,0.035516
5,drift_only,3000,400,150,0.980,pearson,1439,146.6,0.847334,0.029839
0,drift_only,3000,300,125,0.980,pearson,1096,122.2,0.846166,0.040145


In [111]:
best_known = pd.DataFrame([
    {
        "group": "best_known_drift",
        "base_k": 3000,
        "per_fold_k": 300,
        "final_k": 150,
        "corr_threshold": 0.98,
        "corr_method": "pearson",
        "selected_candidates": 1096,
        "n_features_mean": 146.6,
        "test_auroc_mean": 0.852435,
        "test_auroc_std": 0.036809,
    }
])

drift_fine_compare = pd.concat(
    [best_known, drift_fine_df],
    ignore_index=True,
).sort_values("test_auroc_mean", ascending=False)

display(drift_fine_compare.head(20))

,group,base_k,per_fold_k,final_k,corr_threshold,corr_method,selected_candidates,n_features_mean,test_auroc_mean,test_auroc_std
1,drift_only,3500,350,150,0.980,pearson,1331,149.8,0.862582,0.037949
2,drift_only,2500,300,150,0.980,pearson,1086,149.0,0.855774,0.026851
3,drift_only,3000,300,150,0.980,pearson,1096,146.6,0.852435,0.036809
0,best_known_drift,3000,300,150,0.980,pearson,1096,146.6,0.852435,0.036809
4,drift_only,3000,300,150,0.985,pearson,1096,147.2,0.850110,0.037843
5,drift_only,3000,300,150,0.975,pearson,1096,146.0,0.848830,0.034777
6,drift_only,3500,300,150,0.980,pearson,1178,149.8,0.848577,0.035992
7,drift_only,3000,300,150,0.990,pearson,1096,147.8,0.848134,0.031052
8,drift_only,3000,250,125,0.980,pearson,924,122.2,0.848066,0.035516
9,drift_only,3000,400,150,0.980,pearson,1439,146.6,0.847334,0.029839


In [112]:
## new tuning

drift_fine_grid_2 = [
    {"base_k": 3250, "per_fold_k": 325, "final_k": 125, "corr_threshold": 0.98, "corr_method": "pearson"},
    {"base_k": 3250, "per_fold_k": 325, "final_k": 150, "corr_threshold": 0.98, "corr_method": "pearson"},
    {"base_k": 3250, "per_fold_k": 325, "final_k": 175, "corr_threshold": 0.98, "corr_method": "pearson"},

    {"base_k": 3500, "per_fold_k": 325, "final_k": 150, "corr_threshold": 0.98, "corr_method": "pearson"},
    {"base_k": 3500, "per_fold_k": 350, "final_k": 125, "corr_threshold": 0.98, "corr_method": "pearson"},
    {"base_k": 3500, "per_fold_k": 350, "final_k": 150, "corr_threshold": 0.98, "corr_method": "pearson"},
    {"base_k": 3500, "per_fold_k": 350, "final_k": 175, "corr_threshold": 0.98, "corr_method": "pearson"},
    {"base_k": 3500, "per_fold_k": 375, "final_k": 150, "corr_threshold": 0.98, "corr_method": "pearson"},

    {"base_k": 3750, "per_fold_k": 350, "final_k": 150, "corr_threshold": 0.98, "corr_method": "pearson"},
    {"base_k": 4000, "per_fold_k": 350, "final_k": 150, "corr_threshold": 0.98, "corr_method": "pearson"},

    {"base_k": 3500, "per_fold_k": 350, "final_k": 150, "corr_threshold": 0.975, "corr_method": "pearson"},
    {"base_k": 3500, "per_fold_k": 350, "final_k": 150, "corr_threshold": 0.985, "corr_method": "pearson"},
    {"base_k": 3500, "per_fold_k": 350, "final_k": 150, "corr_threshold": 0.99, "corr_method": "pearson"},
]

In [113]:
drift_fine_results_2 = []

for i, cfg in enumerate(drift_fine_grid_2, start=1):
    print(f"\n[{i}/{len(drift_fine_grid_2)}] Drift fine tuning 2: {cfg}")

    selector_fn, counts = make_group_stability_redundancy_selector(
        allowed_idx=drift_allowed_idx,
        base_k=cfg["base_k"],
        per_fold_k=cfg["per_fold_k"],
        final_k=cfg["final_k"],
        corr_threshold=cfg["corr_threshold"],
        corr_method=cfg["corr_method"],
    )

    result = evaluate_catboost_with_selector(
        selector_fn=selector_fn,
        X=X_hidden_full,
        y=y,
        splits=splits,
        verbose=False,
    )

    drift_fine_results_2.append({
        "group": "drift_only",
        **cfg,
        "selected_candidates": len(counts),
        "n_features_mean": result["n_features_mean"],
        "test_auroc_mean": result["test_auroc_mean"],
        "test_auroc_std": result["test_auroc_std"],
    })

drift_fine_df_2 = pd.DataFrame(drift_fine_results_2).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(drift_fine_df_2)


[1/13] Drift fine tuning 2: {'base_k': 3250, 'per_fold_k': 325, 'final_k': 125, 'corr_threshold': 0.98, 'corr_method': 'pearson'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

[2/13] Drift fine tuning 2: {'base_k': 3250, 'per_fold_k': 325, 'final_k': 150, 'corr_threshold': 0.98, 'corr_method': 'pearson'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

[3/13] Drift fine tuning 2: {'base_k': 3250, 'per_fold_k': 325, 'final_k': 175, 'corr_threshold': 0.98, 'corr_method': 'pearson'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

[4/13] Drift fine tuning 2: {'base_k': 3500, 'per_fold_k': 325, 'final_k': 150, 'corr_threshold': 0.98, 'corr_method': 'pearson'}
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fol

,group,base_k,per_fold_k,final_k,corr_threshold,corr_method,selected_candidates,n_features_mean,test_auroc_mean,test_auroc_std
5,drift_only,3500,350,150,0.980,pearson,1331,149.8,0.862582,0.037949
0,drift_only,3250,325,125,0.980,pearson,1239,122.0,0.861698,0.025038
11,drift_only,3500,350,150,0.985,pearson,1331,150.0,0.861616,0.039163
12,drift_only,3500,350,150,0.990,pearson,1331,150.0,0.861616,0.039163
9,drift_only,4000,350,150,0.980,pearson,1393,147.0,0.858773,0.028634
10,drift_only,3500,350,150,0.975,pearson,1331,149.6,0.858307,0.037179
1,drift_only,3250,325,150,0.980,pearson,1239,147.0,0.857037,0.030700
7,drift_only,3500,375,150,0.980,pearson,1404,148.8,0.855742,0.040371
3,drift_only,3500,325,150,0.980,pearson,1250,149.8,0.854960,0.039796
8,drift_only,3750,350,150,0.980,pearson,1378,148.0,0.852001,0.024475


In [114]:
best_known_drift_2 = pd.DataFrame([
    {
        "group": "best_known_drift",
        "base_k": 3500,
        "per_fold_k": 350,
        "final_k": 150,
        "corr_threshold": 0.98,
        "corr_method": "pearson",
        "selected_candidates": 1331,
        "n_features_mean": 149.8,
        "test_auroc_mean": 0.862582,
        "test_auroc_std": 0.037949,
    }
])

drift_fine_compare_2 = pd.concat(
    [best_known_drift_2, drift_fine_df_2],
    ignore_index=True,
).sort_values("test_auroc_mean", ascending=False)

display(drift_fine_compare_2.head(20))

best = drift_fine_compare_2.iloc[0]

print(
    f"BEST DRIFT ITERATION 2:\n"
    f"base_k={best['base_k']}, per_fold_k={best['per_fold_k']}, "
    f"final_k={best['final_k']}, corr={best['corr_threshold']}\n"
    f"n_features={best['n_features_mean']:.1f}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}"
)

,group,base_k,per_fold_k,final_k,corr_threshold,corr_method,selected_candidates,n_features_mean,test_auroc_mean,test_auroc_std
0,best_known_drift,3500,350,150,0.980,pearson,1331,149.8,0.862582,0.037949
1,drift_only,3500,350,150,0.980,pearson,1331,149.8,0.862582,0.037949
2,drift_only,3250,325,125,0.980,pearson,1239,122.0,0.861698,0.025038
3,drift_only,3500,350,150,0.985,pearson,1331,150.0,0.861616,0.039163
4,drift_only,3500,350,150,0.990,pearson,1331,150.0,0.861616,0.039163
5,drift_only,4000,350,150,0.980,pearson,1393,147.0,0.858773,0.028634
6,drift_only,3500,350,150,0.975,pearson,1331,149.6,0.858307,0.037179
7,drift_only,3250,325,150,0.980,pearson,1239,147.0,0.857037,0.030700
8,drift_only,3500,375,150,0.980,pearson,1404,148.8,0.855742,0.040371
9,drift_only,3500,325,150,0.980,pearson,1250,149.8,0.854960,0.039796


BEST DRIFT ITERATION 2:
base_k=3500, per_fold_k=350, final_k=150, corr=0.98
n_features=149.8
AUROC=0.8626 ± 0.0379


## Drift feature space tuning

Generating parquet scripts provided in folder "parquets_generating"

In [115]:
## using new parquets generated

drift_datasets = {
    "old_rich_drift_best": "./artifacts/selected_rich_features/features_dataset_selected_rich.parquet",
    "drift_transforms": "./artifacts/drift_extended_features/features_dataset_drift_transforms.parquet",
    "drift_long_pairs": "./artifacts/drift_extended_features/features_dataset_drift_long_pairs.parquet",
    "drift_token_zones": "./artifacts/drift_extended_features/features_dataset_drift_token_zones.parquet",
    "drift_extended_all": "./artifacts/drift_extended_features/features_dataset_drift_extended_all.parquet",
}


def load_feature_parquet(path):
    df = pd.read_parquet(path)

    drop_cols = {"label", "prompt", "response", "source_index", "id"}
    feature_cols = [c for c in df.columns if c not in drop_cols]

    X = df[feature_cols].to_numpy(dtype=np.float32)
    y_local = df["label"].astype(int).to_numpy()

    return X, y_local, np.array(feature_cols)


def get_drift_indices_from_names(feature_cols):
    names = feature_cols.astype(str)

    drift_mask = (
        np.char.find(names, "drift") >= 0
    )

    return np.where(drift_mask)[0]

In [117]:
## new func

def make_group_stability_redundancy_selector_for_dataset(
    X_current,
    y_current,
    splits_current,
    allowed_idx,
    base_k=3000,
    per_fold_k=300,
    final_k=150,
    corr_threshold=0.97,
    corr_method="pearson",
):
    selected_by_fold, counts = build_stability_selected_features_grouped(
        X=X_current,
        y=y_current,
        splits=splits_current,
        allowed_idx=allowed_idx,
        base_k=base_k,
        per_fold_k=per_fold_k,
        final_k=final_k,
    )

    stable_idx = selected_by_fold[0]

    stability_scores = np.zeros(X_current.shape[1], dtype=np.float32)
    for feat, count in counts.items():
        stability_scores[int(feat)] = float(count)

    def selector_fn(X, y, idx_train, idx_val, idx_test, fold_id):
        pruned_idx = correlation_prune_features(
            X_train=X[idx_train],
            feature_idx=stable_idx,
            feature_scores=stability_scores,
            threshold=corr_threshold,
            method=corr_method,
            max_features=None,
        )

        return pruned_idx

    return selector_fn, counts

In [119]:
extended_compare_results = []

for dataset_name, path in drift_datasets.items():
    print("\n" + "=" * 80)
    print("DATASET:", dataset_name)
    print("PATH:", path)
    print("=" * 80)

    X_ds, y_ds, feature_cols_ds = load_feature_parquet(path)

    assert np.array_equal(y_ds, y), "Labels mismatch"

    allowed_idx = get_drift_indices_from_names(feature_cols_ds)

    print("X:", X_ds.shape)
    print("Drift features:", len(allowed_idx))

    selector_fn, counts = make_group_stability_redundancy_selector_for_dataset(
    X_current=X_ds,
    y_current=y_ds,
    splits_current=splits,
    allowed_idx=allowed_idx,
    base_k=3500,
    per_fold_k=350,
    final_k=150,
    corr_threshold=0.98,
    corr_method="pearson",
)

    result = evaluate_catboost_with_selector(
        selector_fn=selector_fn,
        X=X_ds,
        y=y_ds,
        splits=splits,
        verbose=False,
    )

    extended_compare_results.append({
        "dataset": dataset_name,
        "n_total_features": X_ds.shape[1],
        "n_drift_features": len(allowed_idx),
        "selected_candidates": len(counts),
        "n_features_mean": result["n_features_mean"],
        "test_auroc_mean": result["test_auroc_mean"],
        "test_auroc_std": result["test_auroc_std"],
    })

extended_compare_df = pd.DataFrame(extended_compare_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(extended_compare_df)


DATASET: old_rich_drift_best
PATH: ./artifacts/selected_rich_features/features_dataset_selected_rich.parquet
X: (689, 20878)
Drift features: 9080
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

DATASET: drift_transforms
PATH: ./artifacts/drift_extended_features/features_dataset_drift_transforms.parquet
X: (689, 22400)
Drift features: 22400
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

DATASET: drift_long_pairs
PATH: ./artifacts/drift_extended_features/features_dataset_drift_long_pairs.parquet
X: (689, 26880)
Drift features: 26880
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

DATASET: drift_token_zones
PATH: ./artifacts/drift_extended_features/features_dataset_drift_token_zones.parquet
X: (689, 40320)
Drift features: 40320
catboost grouped fold 1
catboost grouped fol

,dataset,n_total_features,n_drift_features,selected_candidates,n_features_mean,test_auroc_mean,test_auroc_std
4,drift_extended_all,89600,89600,1449,149.0,0.868560,0.019028
0,old_rich_drift_best,20878,9080,1332,147.0,0.849548,0.030914
3,drift_token_zones,40320,40320,1449,148.0,0.847344,0.023669
2,drift_long_pairs,26880,26880,1364,146.2,0.845633,0.022224
1,drift_transforms,22400,22400,1410,149.0,0.834986,0.035379


In [120]:
known_best = pd.DataFrame([
    {
        "dataset": "known_best_old_drift",
        "n_total_features": 20878,
        "n_drift_features": 8960,
        "selected_candidates": 1331,
        "n_features_mean": 149.8,
        "test_auroc_mean": 0.862582,
        "test_auroc_std": 0.037949,
    }
])

extended_final_compare = pd.concat(
    [known_best, extended_compare_df],
    ignore_index=True,
).sort_values("test_auroc_mean", ascending=False)

display(extended_final_compare)

best = extended_final_compare.iloc[0]
print(
    f"BEST EXTENDED DRIFT RESULT:\n"
    f"{best['dataset']}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}\n"
    f"n_features_mean={best['n_features_mean']:.1f}"
)

,dataset,n_total_features,n_drift_features,selected_candidates,n_features_mean,test_auroc_mean,test_auroc_std
1,drift_extended_all,89600,89600,1449,149.0,0.868560,0.019028
0,known_best_old_drift,20878,8960,1331,149.8,0.862582,0.037949
2,old_rich_drift_best,20878,9080,1332,147.0,0.849548,0.030914
3,drift_token_zones,40320,40320,1449,148.0,0.847344,0.023669
4,drift_long_pairs,26880,26880,1364,146.2,0.845633,0.022224
5,drift_transforms,22400,22400,1410,149.0,0.834986,0.035379


BEST EXTENDED DRIFT RESULT:
drift_extended_all
AUROC=0.8686 ± 0.0190
n_features_mean=149.0


## Smarter interaction features

In [122]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

INPUT_PATH = "./artifacts/drift_extended_features/features_dataset_drift_extended_all.parquet"
OUTPUT_PATH = "./artifacts/drift_extended_features/features_dataset_drift_interactions.parquet"

df = pd.read_parquet(INPUT_PATH)

drop_cols = {"label", "prompt", "response", "source_index", "id"}
feature_cols = [c for c in df.columns if c not in drop_cols]

print("Input shape:", df.shape)
print("Feature cols:", len(feature_cols))

Input shape: (689, 89604)
Feature cols: 89600


In [128]:
def find_block_columns(cols, prefix):
    return [c for c in cols if c.startswith(prefix)]


def parse_dim(col):
    m = re.search(r"_d(\d+)$", col)
    return int(m.group(1)) if m else None


def get_transition_block(df, left, right, must_contain="drift", exclude_abs=True):
    cols = []

    patterns = [
        f"l{left}_to_l{right}",
        f"layer_{left}_to_{right}",
        f"{left}_to_{right}",
    ]

    for c in df.columns:
        c_str = str(c)

        if must_contain not in c_str:
            continue

        if exclude_abs and "abs" in c_str:
            continue

        if any(p in c_str for p in patterns) and re.search(r"_d\d+$", c_str):
            cols.append(c)

    cols = sorted(cols, key=parse_dim)

    if not cols:
        return None, []

    return df[cols].to_numpy(dtype=np.float32), cols


CONSECUTIVE_PAIRS = [(10, 11), (11, 12), (12, 13), (13, 14), (14, 15), (15, 16)]
LONG_PAIRS = [(10, 12), (11, 13), (12, 14), (13, 15), (14, 16), (10, 14), (11, 15), (12, 16), (10, 16), (11, 16)]

interaction_parts = []
interaction_names = []

n = len(df)

# 1. Собираем consecutive signed drifts
drift_blocks = {}
for left, right in CONSECUTIVE_PAIRS:
    arr, cols = get_transition_block(
    df,
    left,
    right,
    )
    if arr is not None:
        drift_blocks[(left, right)] = arr
        print(f"Found response_drift {left}->{right}:", arr.shape)

print("Total consecutive drift blocks:", len(drift_blocks))

Found response_drift 11->12: (689, 8960)
Found response_drift 12->13: (689, 8960)
Found response_drift 13->14: (689, 8960)
Found response_drift 14->15: (689, 8960)
Found response_drift 15->16: (689, 8960)
Total consecutive drift blocks: 5


In [ ]:
scalar_features = []
scalar_names = []

# drift energy / norms per transition
transition_norms = {}

for pair, arr in drift_blocks.items():
    left, right = pair

    l2 = np.linalg.norm(arr, axis=1)
    mean_abs = np.mean(np.abs(arr), axis=1)
    energy = np.mean(arr ** 2, axis=1)
    std = np.std(arr, axis=1)
    max_abs = np.max(np.abs(arr), axis=1)

    transition_norms[pair] = l2

    scalar_features.extend([l2, mean_abs, energy, std, max_abs])
    scalar_names.extend([
        f"interact_drift_l{left}_to_l{right}_l2",
        f"interact_drift_l{left}_to_l{right}_mean_abs",
        f"interact_drift_l{left}_to_l{right}_energy",
        f"interact_drift_l{left}_to_l{right}_std",
        f"interact_drift_l{left}_to_l{right}_max_abs",
    ])

# pairwise ratios between transition norms
pairs = list(transition_norms.keys())

for i in range(len(pairs)):
    for j in range(i + 1, len(pairs)):
        p1, p2 = pairs[i], pairs[j]
        ratio = transition_norms[p1] / (transition_norms[p2] + 1e-8)
        diff = transition_norms[p1] - transition_norms[p2]

        scalar_features.extend([ratio, diff])
        scalar_names.extend([
            f"interact_ratio_norm_l{p1[0]}_{p1[1]}_over_l{p2[0]}_{p2[1]}",
            f"interact_diff_norm_l{p1[0]}_{p1[1]}_minus_l{p2[0]}_{p2[1]}",
        ])

# stack scalar features
if scalar_features:
    scalar_matrix = np.column_stack(scalar_features).astype(np.float32)
    interaction_parts.append(scalar_matrix)
    interaction_names.extend(scalar_names)

print("Scalar interaction features:", len(scalar_names))

Scalar interaction features: 45


In [ ]:
# using only consecutive blocks with same dimentions
valid_pairs = [p for p in CONSECUTIVE_PAIRS if p in drift_blocks]

if len(valid_pairs) >= 3:
    trajectory = np.stack([drift_blocks[p] for p in valid_pairs], axis=1)

    print("Trajectory shape:", trajectory.shape)

    # variance of drift trajectory per dimension
    traj_var = np.var(trajectory, axis=1)

    # mean absolute trajectory
    traj_abs_mean = np.mean(np.abs(trajectory), axis=1)

    # max absolute trajectory
    traj_abs_max = np.max(np.abs(trajectory), axis=1)

    # monotonicity of absolute drift across layers
    abs_traj = np.abs(trajectory)
    diffs = np.diff(abs_traj, axis=1)

    monotonic_increase_ratio = np.mean(diffs > 0, axis=1)
    monotonic_decrease_ratio = np.mean(diffs < 0, axis=1)

    # curvature: second difference across consecutive drift trajectory
    curvature = np.diff(trajectory, n=2, axis=1)
    curvature_abs_mean = np.mean(np.abs(curvature), axis=1)
    curvature_var = np.var(curvature, axis=1)

    blocks = [
        ("traj_var", traj_var),
        ("traj_abs_mean", traj_abs_mean),
        ("traj_abs_max", traj_abs_max),
        ("traj_mono_inc_ratio", monotonic_increase_ratio),
        ("traj_mono_dec_ratio", monotonic_decrease_ratio),
        ("traj_curvature_abs_mean", curvature_abs_mean),
        ("traj_curvature_var", curvature_var),
    ]

    for block_name, block in blocks:
        interaction_parts.append(block.astype(np.float32))
        interaction_names.extend([f"interact_{block_name}_d{i}" for i in range(block.shape[1])])

print("Total interaction columns:", len(interaction_names))

Trajectory shape: (689, 5, 8960)
Total interaction columns: 62765


In [131]:
X_interact = np.concatenate(interaction_parts, axis=1).astype(np.float32)

df_interact = pd.DataFrame(X_interact, columns=interaction_names)
df_interact.insert(0, "source_index", df["source_index"].to_numpy())

if "label" in df.columns:
    df_interact["label"] = df["label"].astype(int).to_numpy()

df_interact["prompt"] = df["prompt"].astype(str).to_numpy()
df_interact["response"] = df["response"].astype(str).to_numpy()

Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
df_interact.to_parquet(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Interaction shape:", df_interact.shape)

Saved: ./artifacts/drift_extended_features/features_dataset_drift_interactions.parquet
Interaction shape: (689, 62769)


In [132]:
df_base = pd.read_parquet("./artifacts/drift_extended_features/features_dataset_drift_extended_all.parquet")
df_inter = pd.read_parquet("./artifacts/drift_extended_features/features_dataset_drift_interactions.parquet")

drop_cols = {"label", "prompt", "response", "source_index", "id"}

base_cols = [c for c in df_base.columns if c not in drop_cols]
inter_cols = [c for c in df_inter.columns if c not in drop_cols]

X_base = df_base[base_cols].to_numpy(dtype=np.float32)
X_inter = df_inter[inter_cols].to_numpy(dtype=np.float32)
y_base = df_base["label"].astype(int).to_numpy()

assert np.array_equal(y_base, y)

X_combined = np.concatenate([X_base, X_inter], axis=1)

print("Base:", X_base.shape)
print("Interaction:", X_inter.shape)
print("Combined:", X_combined.shape)

Base: (689, 89600)
Interaction: (689, 62765)
Combined: (689, 152365)


In [133]:
def evaluate_dataset_with_drift_pipeline(X_current, y_current, name):
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    allowed_idx = np.arange(X_current.shape[1])

    selector_fn, counts = make_group_stability_redundancy_selector_for_dataset(
        X_current=X_current,
        y_current=y_current,
        splits_current=splits,
        allowed_idx=allowed_idx,
        base_k=3500,
        per_fold_k=350,
        final_k=150,
        corr_threshold=0.98,
        corr_method="pearson",
    )

    result = evaluate_catboost_with_selector(
        selector_fn=selector_fn,
        X=X_current,
        y=y_current,
        splits=splits,
        verbose=False,
    )

    return {
        "dataset": name,
        "n_features_total": X_current.shape[1],
        "selected_candidates": len(counts),
        "n_features_mean": result["n_features_mean"],
        "test_auroc_mean": result["test_auroc_mean"],
        "test_auroc_std": result["test_auroc_std"],
    }

In [134]:
interaction_results = []

interaction_results.append({
    "dataset": "known_best_drift_extended_all",
    "n_features_total": X_base.shape[1],
    "selected_candidates": None,
    "n_features_mean": 149.0,
    "test_auroc_mean": 0.8686,
    "test_auroc_std": 0.0190,
})

interaction_results.append(
    evaluate_dataset_with_drift_pipeline(
        X_current=X_inter,
        y_current=y,
        name="interaction_only",
    )
)

interaction_results.append(
    evaluate_dataset_with_drift_pipeline(
        X_current=X_combined,
        y_current=y,
        name="drift_extended_all_plus_interactions",
    )
)

interaction_results_df = pd.DataFrame(interaction_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(interaction_results_df)

best = interaction_results_df.iloc[0]

print(
    f"BEST INTERACTION RESULT:\n"
    f"{best['dataset']}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}\n"
    f"n_features_mean={best['n_features_mean']}"
)


interaction_only
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

drift_extended_all_plus_interactions
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5


,dataset,n_features_total,selected_candidates,n_features_mean,test_auroc_mean,test_auroc_std
0,known_best_drift_extended_all,89600,NaN,149.0,0.868600,0.019000
2,drift_extended_all_plus_interactions,152365,1466.0,147.6,0.834672,0.032723
1,interaction_only,62765,1396.0,149.4,0.829237,0.014508


BEST INTERACTION RESULT:
known_best_drift_extended_all
AUROC=0.8686 ± 0.0190
n_features_mean=149.0


## Check token position dynamics (didn't work)

In [136]:
pd.read_parquet(
    "./artifacts/token_position_dynamics/features_dataset_token_position_dynamics.parquet"
).shape

(689, 544)

In [137]:
import numpy as np
import pandas as pd

# ============================================================
# LOAD DATASETS
# ============================================================

df_best = pd.read_parquet(
    "./artifacts/drift_extended_features/features_dataset_drift_extended_all.parquet"
)

df_token_pos = pd.read_parquet(
    "./artifacts/token_position_dynamics/features_dataset_token_position_dynamics.parquet"
)

DROP_COLS = {
    "label",
    "prompt",
    "response",
    "source_index",
    "id",
}

best_cols = [c for c in df_best.columns if c not in DROP_COLS]
token_cols = [c for c in df_token_pos.columns if c not in DROP_COLS]

X_best = df_best[best_cols].to_numpy(dtype=np.float32)
X_token = df_token_pos[token_cols].to_numpy(dtype=np.float32)

y_best = df_best["label"].astype(int).to_numpy()
y_token = df_token_pos["label"].astype(int).to_numpy()

assert np.array_equal(y_best, y_token)

X_combined = np.concatenate([X_best, X_token], axis=1)

print("Best drift:", X_best.shape)
print("Token-position:", X_token.shape)
print("Combined:", X_combined.shape)

Best drift: (689, 89600)
Token-position: (689, 540)
Combined: (689, 90140)


In [138]:
def evaluate_with_current_best_pipeline(
    X_current,
    y_current,
    name,
):
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    allowed_idx = np.arange(X_current.shape[1])

    selector_fn, counts = make_group_stability_redundancy_selector_for_dataset(
        X_current=X_current,
        y_current=y_current,
        splits_current=splits,

        allowed_idx=allowed_idx,

        # CURRENT BEST PIPELINE
        base_k=3500,
        per_fold_k=350,
        final_k=150,

        corr_threshold=0.98,
        corr_method="pearson",
    )

    result = evaluate_catboost_with_selector(
        selector_fn=selector_fn,
        X=X_current,
        y=y_current,
        splits=splits,
        verbose=False,
    )

    return {
        "dataset": name,
        "n_features_total": X_current.shape[1],
        "selected_candidates": len(counts),
        "n_features_mean": result["n_features_mean"],
        "test_auroc_mean": result["test_auroc_mean"],
        "test_auroc_std": result["test_auroc_std"],
    }

In [139]:
results = []

# known best
results.append({
    "dataset": "known_best_drift_extended_all",
    "n_features_total": X_best.shape[1],
    "selected_candidates": None,
    "n_features_mean": 149.0,
    "test_auroc_mean": 0.8686,
    "test_auroc_std": 0.0190,
})

# token-position only
results.append(
    evaluate_with_current_best_pipeline(
        X_current=X_token,
        y_current=y,
        name="token_position_only",
    )
)

# combined
results.append(
    evaluate_with_current_best_pipeline(
        X_current=X_combined,
        y_current=y,
        name="drift_extended_all_plus_token_position",
    )
)

results_df = pd.DataFrame(results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(results_df)

best = results_df.iloc[0]

print(
    f"\nBEST TOKEN-POSITION RESULT:\n"
    f"{best['dataset']}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}\n"
    f"n_features_mean={best['n_features_mean']}"
)


token_position_only
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5

drift_extended_all_plus_token_position
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5


,dataset,n_features_total,selected_candidates,n_features_mean,test_auroc_mean,test_auroc_std
0,known_best_drift_extended_all,89600,NaN,149.0,0.868600,0.019000
2,drift_extended_all_plus_token_position,90140,1442.0,148.0,0.829672,0.059821
1,token_position_only,540,517.0,99.4,0.770492,0.027475



BEST TOKEN-POSITION RESULT:
known_best_drift_extended_all
AUROC=0.8686 ± 0.0190
n_features_mean=149.0


## Pseudo ensemble over feature families

In [141]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.special import logit, expit

FEATURE_FAMILY_PATHS = {
    "old_rich": "./artifacts/selected_rich_features/features_dataset_selected_rich.parquet",
    "drift_transforms": "./artifacts/drift_extended_features/features_dataset_drift_transforms.parquet",
    "drift_long_pairs": "./artifacts/drift_extended_features/features_dataset_drift_long_pairs.parquet",
    "drift_token_zones": "./artifacts/drift_extended_features/features_dataset_drift_token_zones.parquet",
    "drift_extended_all": "./artifacts/drift_extended_features/features_dataset_drift_extended_all.parquet",
    "token_position": "./artifacts/token_position_dynamics/features_dataset_token_position_dynamics.parquet",
}

DROP_COLS = {"label", "prompt", "response", "source_index", "id"}


def load_family(path):
    df = pd.read_parquet(path)
    feature_cols = [c for c in df.columns if c not in DROP_COLS]

    X = df[feature_cols].to_numpy(dtype=np.float32)
    y_local = df["label"].astype(int).to_numpy()

    return X, y_local, np.array(feature_cols)


families = {}

for name, path in FEATURE_FAMILY_PATHS.items():
    X_family, y_family, cols_family = load_family(path)

    assert np.array_equal(y_family, y), f"Label mismatch: {name}"

    families[name] = {
        "X": X_family,
        "cols": cols_family,
    }

    print(f"{name}: X={X_family.shape}")

old_rich: X=(689, 20878)
drift_transforms: X=(689, 22400)
drift_long_pairs: X=(689, 26880)
drift_token_zones: X=(689, 40320)
drift_extended_all: X=(689, 89600)
token_position: X=(689, 540)


In [142]:
def fit_predict_family_on_fold(
    X_family,
    y,
    idx_train,
    idx_val,
    idx_test,
    base_k=3500,
    per_fold_k=350,
    final_k=150,
    corr_threshold=0.98,
    corr_method="pearson",
):
    allowed_idx = np.arange(X_family.shape[1])

    selector_fn, counts = make_group_stability_redundancy_selector_for_dataset(
        X_current=X_family,
        y_current=y,
        splits_current=[(idx_train, idx_val, idx_test)],
        allowed_idx=allowed_idx,
        base_k=base_k,
        per_fold_k=per_fold_k,
        final_k=final_k,
        corr_threshold=corr_threshold,
        corr_method=corr_method,
    )

    selected_idx = selector_fn(
        X=X_family,
        y=y,
        idx_train=idx_train,
        idx_val=idx_val,
        idx_test=idx_test,
        fold_id=1,
    )

    model = get_tuned_catboost()

    model.fit(
        X_family[idx_train][:, selected_idx],
        y[idx_train],
        eval_set=(X_family[idx_val][:, selected_idx], y[idx_val]),
        use_best_model=True,
    )

    val_probs = model.predict_proba(X_family[idx_val][:, selected_idx])[:, 1]
    test_probs = model.predict_proba(X_family[idx_test][:, selected_idx])[:, 1]

    return val_probs, test_probs, selected_idx

In [143]:
family_fold_predictions = {}

for family_name, pack in families.items():
    print("\n" + "=" * 80)
    print(f"FAMILY: {family_name}")
    print("=" * 80)

    X_family = pack["X"]

    val_preds_by_fold = []
    test_preds_by_fold = []
    test_y_by_fold = []
    selected_sizes = []

    for fold_id, (idx_train, idx_val, idx_test) in enumerate(splits, start=1):
        print(f"{family_name} | fold {fold_id}")

        val_probs, test_probs, selected_idx = fit_predict_family_on_fold(
            X_family=X_family,
            y=y,
            idx_train=idx_train,
            idx_val=idx_val,
            idx_test=idx_test,
            base_k=3500,
            per_fold_k=350,
            final_k=150,
            corr_threshold=0.98,
            corr_method="pearson",
        )

        val_preds_by_fold.append(val_probs)
        test_preds_by_fold.append(test_probs)
        test_y_by_fold.append(y[idx_test])
        selected_sizes.append(len(selected_idx))

    family_fold_predictions[family_name] = {
        "val": val_preds_by_fold,
        "test": test_preds_by_fold,
        "test_y": test_y_by_fold,
        "selected_sizes": selected_sizes,
    }

    aucs = [
        roc_auc_score(test_y_by_fold[i], test_preds_by_fold[i])
        for i in range(len(test_preds_by_fold))
    ]

    print(
        f"{family_name}: AUROC={np.mean(aucs):.4f} ± {np.std(aucs):.4f}, "
        f"selected={np.mean(selected_sizes):.1f}"
    )


FAMILY: old_rich
old_rich | fold 1
catboost grouped fold 1
old_rich | fold 2
catboost grouped fold 1
old_rich | fold 3
catboost grouped fold 1
old_rich | fold 4
catboost grouped fold 1
old_rich | fold 5
catboost grouped fold 1
old_rich: AUROC=0.7920 ± 0.0223, selected=146.4

FAMILY: drift_transforms
drift_transforms | fold 1
catboost grouped fold 1
drift_transforms | fold 2
catboost grouped fold 1
drift_transforms | fold 3
catboost grouped fold 1
drift_transforms | fold 4
catboost grouped fold 1
drift_transforms | fold 5
catboost grouped fold 1
drift_transforms: AUROC=0.7819 ± 0.0298, selected=147.8

FAMILY: drift_long_pairs
drift_long_pairs | fold 1
catboost grouped fold 1
drift_long_pairs | fold 2
catboost grouped fold 1
drift_long_pairs | fold 3
catboost grouped fold 1
drift_long_pairs | fold 4
catboost grouped fold 1
drift_long_pairs | fold 5
catboost grouped fold 1
drift_long_pairs: AUROC=0.7556 ± 0.0388, selected=147.8

FAMILY: drift_token_zones
drift_token_zones | fold 1
catboo

In [144]:
single_family_rows = []

for family_name, preds in family_fold_predictions.items():
    aucs = [
        roc_auc_score(preds["test_y"][i], preds["test"][i])
        for i in range(len(preds["test"]))
    ]

    single_family_rows.append({
        "family": family_name,
        "test_auroc_mean": np.mean(aucs),
        "test_auroc_std": np.std(aucs),
        "selected_features_mean": np.mean(preds["selected_sizes"]),
    })

single_family_df = pd.DataFrame(single_family_rows).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(single_family_df)

,family,test_auroc_mean,test_auroc_std,selected_features_mean
0,old_rich,0.792008,0.022344,146.4
4,drift_extended_all,0.784487,0.040340,148.2
1,drift_transforms,0.781896,0.029815,147.8
3,drift_token_zones,0.781386,0.025661,148.6
2,drift_long_pairs,0.755588,0.038750,147.8
5,token_position,0.755356,0.031717,105.8


In [145]:
def safe_logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return logit(p)


def ensemble_fold_auc(family_names, weights=None, mode="proba"):
    if weights is None:
        weights = np.ones(len(family_names)) / len(family_names)
    else:
        weights = np.asarray(weights, dtype=np.float64)
        weights = weights / weights.sum()

    fold_aucs = []

    for fold_i in range(len(splits)):
        y_test = family_fold_predictions[family_names[0]]["test_y"][fold_i]

        preds = [
            family_fold_predictions[name]["test"][fold_i]
            for name in family_names
        ]

        if mode == "proba":
            ens = np.zeros_like(preds[0], dtype=np.float64)
            for w, p in zip(weights, preds):
                ens += w * p

        elif mode == "logit":
            ens_logit = np.zeros_like(preds[0], dtype=np.float64)
            for w, p in zip(weights, preds):
                ens_logit += w * safe_logit(p)
            ens = expit(ens_logit)

        else:
            raise ValueError(mode)

        fold_aucs.append(roc_auc_score(y_test, ens))

    return float(np.mean(fold_aucs)), float(np.std(fold_aucs))

In [146]:
from itertools import combinations

ensemble_rows = []

family_names = list(families.keys())

for r in [2, 3, 4, 5]:
    for combo in combinations(family_names, r):
        for mode in ["proba", "logit"]:
            mean_auc, std_auc = ensemble_fold_auc(
                family_names=list(combo),
                weights=None,
                mode=mode,
            )

            ensemble_rows.append({
                "families": " + ".join(combo),
                "n_families": r,
                "mode": mode,
                "weights": "equal",
                "test_auroc_mean": mean_auc,
                "test_auroc_std": std_auc,
            })

ensemble_df = pd.DataFrame(ensemble_rows).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(ensemble_df.head(30))

,families,n_families,mode,weights,test_auroc_mean,test_auroc_std
107,old_rich + drift_transforms + drift_token_zone...,5,logit,equal,0.803628,0.027349
81,old_rich + drift_transforms + drift_extended_a...,4,logit,equal,0.803136,0.026379
89,old_rich + drift_token_zones + drift_extended_...,4,logit,equal,0.801419,0.028098
106,old_rich + drift_transforms + drift_token_zone...,5,proba,equal,0.801331,0.026804
79,old_rich + drift_transforms + drift_token_zone...,4,logit,equal,0.801149,0.024213
77,old_rich + drift_transforms + drift_token_zone...,4,logit,equal,0.801108,0.026981
34,old_rich + drift_transforms + drift_extended_all,3,proba,equal,0.801095,0.029095
100,old_rich + drift_transforms + drift_long_pairs...,5,proba,equal,0.800657,0.026892
35,old_rich + drift_transforms + drift_extended_all,3,logit,equal,0.800648,0.028344
101,old_rich + drift_transforms + drift_long_pairs...,5,logit,equal,0.800554,0.026982


In [ ]:
weighted_rows = []

# getting top-3 single families
top_families = single_family_df["family"].head(3).tolist()

print("Top families:", top_families)

weight_grid = [
    (0.6, 0.3, 0.1),
    (0.6, 0.2, 0.2),
    (0.5, 0.4, 0.1),
    (0.5, 0.3, 0.2),
    (0.4, 0.4, 0.2),
    (0.7, 0.2, 0.1),
    (0.8, 0.1, 0.1),
]

for weights in weight_grid:
    for mode in ["proba", "logit"]:
        mean_auc, std_auc = ensemble_fold_auc(
            family_names=top_families,
            weights=weights,
            mode=mode,
        )

        weighted_rows.append({
            "families": " + ".join(top_families),
            "mode": mode,
            "weights": weights,
            "test_auroc_mean": mean_auc,
            "test_auroc_std": std_auc,
        })

weighted_ensemble_df = pd.DataFrame(weighted_rows).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(weighted_ensemble_df)

Top families: ['old_rich', 'drift_extended_all', 'drift_transforms']


,families,mode,weights,test_auroc_mean,test_auroc_std
8,old_rich + drift_extended_all + drift_transforms,proba,"(0.4, 0.4, 0.2)",0.800793,0.028775
6,old_rich + drift_extended_all + drift_transforms,proba,"(0.5, 0.3, 0.2)",0.800758,0.027635
9,old_rich + drift_extended_all + drift_transforms,logit,"(0.4, 0.4, 0.2)",0.799689,0.028576
2,old_rich + drift_extended_all + drift_transforms,proba,"(0.6, 0.2, 0.2)",0.799610,0.026278
7,old_rich + drift_extended_all + drift_transforms,logit,"(0.5, 0.3, 0.2)",0.799597,0.028125
0,old_rich + drift_extended_all + drift_transforms,proba,"(0.6, 0.3, 0.1)",0.799554,0.027359
4,old_rich + drift_extended_all + drift_transforms,proba,"(0.5, 0.4, 0.1)",0.799389,0.028931
3,old_rich + drift_extended_all + drift_transforms,logit,"(0.6, 0.2, 0.2)",0.799106,0.026751
5,old_rich + drift_extended_all + drift_transforms,logit,"(0.5, 0.4, 0.1)",0.798728,0.029645
10,old_rich + drift_extended_all + drift_transforms,proba,"(0.7, 0.2, 0.1)",0.798713,0.025539


In [148]:
known_best = pd.DataFrame([
    {
        "source": "known_best_single",
        "families": "drift_extended_all",
        "mode": "single",
        "weights": None,
        "test_auroc_mean": 0.8686,
        "test_auroc_std": 0.0190,
    }
])

single_as_df = single_family_df.rename(columns={"family": "families"})
single_as_df["source"] = "single_family"
single_as_df["mode"] = "single"
single_as_df["weights"] = None
single_as_df = single_as_df[
    ["source", "families", "mode", "weights", "test_auroc_mean", "test_auroc_std"]
]

ens_as_df = ensemble_df.copy()
ens_as_df["source"] = "equal_ensemble"
ens_as_df = ens_as_df[
    ["source", "families", "mode", "weights", "test_auroc_mean", "test_auroc_std"]
]

weighted_as_df = weighted_ensemble_df.copy()
weighted_as_df["source"] = "weighted_ensemble"
weighted_as_df = weighted_as_df[
    ["source", "families", "mode", "weights", "test_auroc_mean", "test_auroc_std"]
]

final_ensemble_compare = pd.concat(
    [known_best, single_as_df, ens_as_df, weighted_as_df],
    ignore_index=True,
).sort_values("test_auroc_mean", ascending=False)

display(final_ensemble_compare.head(30))

best = final_ensemble_compare.iloc[0]

print(
    f"BEST PSEUDO-ENSEMBLE RESULT:\n"
    f"{best['source']}\n"
    f"{best['families']}\n"
    f"mode={best['mode']}, weights={best['weights']}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}"
)

,source,families,mode,weights,test_auroc_mean,test_auroc_std
0,known_best_single,drift_extended_all,single,None,0.868600,0.019000
7,equal_ensemble,old_rich + drift_transforms + drift_token_zone...,logit,equal,0.803628,0.027349
8,equal_ensemble,old_rich + drift_transforms + drift_extended_a...,logit,equal,0.803136,0.026379
9,equal_ensemble,old_rich + drift_token_zones + drift_extended_...,logit,equal,0.801419,0.028098
10,equal_ensemble,old_rich + drift_transforms + drift_token_zone...,proba,equal,0.801331,0.026804
11,equal_ensemble,old_rich + drift_transforms + drift_token_zone...,logit,equal,0.801149,0.024213
12,equal_ensemble,old_rich + drift_transforms + drift_token_zone...,logit,equal,0.801108,0.026981
13,equal_ensemble,old_rich + drift_transforms + drift_extended_all,proba,equal,0.801095,0.029095
119,weighted_ensemble,old_rich + drift_extended_all + drift_transforms,proba,"(0.4, 0.4, 0.2)",0.800793,0.028775
120,weighted_ensemble,old_rich + drift_extended_all + drift_transforms,proba,"(0.5, 0.3, 0.2)",0.800758,0.027635


BEST PSEUDO-ENSEMBLE RESULT:
known_best_single
drift_extended_all
mode=single, weights=None
AUROC=0.8686 ± 0.0190


## Advanced structural features

Trying to construct new features again (didn't work)

In [ ]:
# pip install spacy textstat
# python -m spacy download en_core_web_sm

import re
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from collections import Counter

from nltk.util import ngrams

import spacy

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

nlp = spacy.load("en_core_web_sm")

COMMON_STOPWORDS = set(ENGLISH_STOP_WORDS)

text_df = pd.read_csv("./data/dataset.csv")

print(text_df.shape)
display(text_df.head())

HEDGE_WORDS = {
    "maybe", "perhaps", "possibly", "probably",
    "might", "could", "appears", "seems",
    "likely", "apparently", "suggests",
    "approximately", "generally", "typically",
}

CITATION_PATTERNS = [
    r"\[\d+\]",
    r"\(\d{4}\)",
    r"et al\.",
    r"according to",
    r"reported by",
    r"studies show",
]

UNCERTAINTY_WORDS = {
    "maybe", "perhaps", "possibly", "probably",
    "might", "could", "unclear", "unknown",
    "uncertain", "appears", "seems",
}

def sent_tokenize_spacy(text):
    doc = nlp(text)
    return [s.text for s in doc.sents]


def word_tokenize_simple(text):
    return re.findall(r"\b\w+\b", text.lower())


def safe_entropy(tokens):

    if len(tokens) == 0:
        return 0.0

    counts = Counter(tokens)

    probs = np.array(list(counts.values()), dtype=float)

    probs /= probs.sum()

    return -(probs * np.log2(probs + 1e-12)).sum()


def repetition_score(tokens, n=2):

    if len(tokens) < n:
        return 0.0

    grams = list(ngrams(tokens, n))

    if len(grams) == 0:
        return 0.0

    unique_ratio = len(set(grams)) / len(grams)

    return 1.0 - unique_ratio


def burstiness(lengths):

    if len(lengths) < 2:
        return 0.0

    arr = np.array(lengths)

    mean = arr.mean()

    std = arr.std()

    if mean == 0:
        return 0.0

    return (std - mean) / (std + mean + 1e-9)


def token_rarity(tokens):

    if len(tokens) == 0:
        return 0.0

    long_tokens = [t for t in tokens if len(t) >= 8]

    return len(long_tokens) / len(tokens)


def lexical_diversity(tokens):

    if len(tokens) == 0:
        return 0.0

    return len(set(tokens)) / len(tokens)


def sentence_length_variance(sentences):

    if len(sentences) <= 1:
        return 0.0

    lens = [
        len(word_tokenize_simple(s))
        for s in sentences
    ]

    return np.var(lens)


def extract_structural_features(text):

    text = str(text)

    sents = sent_tokenize_spacy(text)

    tokens = word_tokenize_simple(text)

    tokens_alpha = [
        t for t in tokens
        if t.isalpha()
    ]

    doc = nlp(text)

    feats = {}

    feats["repeat_bigram"] = repetition_score(tokens_alpha, 2)

    feats["repeat_trigram"] = repetition_score(tokens_alpha, 3)

    feats["token_entropy"] = safe_entropy(tokens_alpha)

    sent_lengths = [
        len(word_tokenize_simple(s))
        for s in sents
    ]

    feats["sentence_entropy"] = safe_entropy(sent_lengths)

    feats["sentence_len_variance"] = sentence_length_variance(sents)

    feats["lexical_diversity"] = lexical_diversity(tokens_alpha)

    feats["burstiness"] = burstiness(sent_lengths)

    feats["token_rarity"] = token_rarity(tokens_alpha)

    uncertainty_per_sent = []

    for s in sents:

        stoks = word_tokenize_simple(s)

        cnt = sum(
            t in UNCERTAINTY_WORDS
            for t in stoks
        )

        uncertainty_per_sent.append(cnt)

    if len(uncertainty_per_sent) >= 2:

        x = np.arange(len(uncertainty_per_sent))

        slope = np.polyfit(
            x,
            uncertainty_per_sent,
            deg=1,
        )[0]

    else:
        slope = 0.0

    feats["uncertainty_slope"] = slope

    feats["uncertainty_total"] = sum(uncertainty_per_sent)

    citation_count = 0

    for pattern in CITATION_PATTERNS:

        citation_count += len(
            re.findall(pattern, text.lower())
        )

    feats["citation_count"] = citation_count

    pos_tags = [t.pos_ for t in doc]

    total_pos = max(len(pos_tags), 1)

    feats["noun_ratio"] = (
        pos_tags.count("NOUN") / total_pos
    )

    feats["verb_ratio"] = (
        pos_tags.count("VERB") / total_pos
    )

    feats["adj_ratio"] = (
        pos_tags.count("ADJ") / total_pos
    )

    feats["adv_ratio"] = (
        pos_tags.count("ADV") / total_pos
    )

    feats["punct_ratio"] = (
        pos_tags.count("PUNCT") / total_pos
    )

    feats["num_ratio"] = (
        pos_tags.count("NUM") / total_pos
    )

    ents = doc.ents

    feats["entity_density"] = (
        len(ents) / max(len(tokens_alpha), 1)
    )

    entity_types = [e.label_ for e in ents]

    feats["person_entity_ratio"] = (
        entity_types.count("PERSON")
        / max(len(ents), 1)
    )

    feats["org_entity_ratio"] = (
        entity_types.count("ORG")
        / max(len(ents), 1)
    )

    feats["gpe_entity_ratio"] = (
        entity_types.count("GPE")
        / max(len(ents), 1)
    )

    return feats

all_rows = []

for response in tqdm(text_df["response"]):

    feats = extract_structural_features(response)

    all_rows.append(feats)

struct_df = pd.DataFrame(all_rows)

print(struct_df.shape)

display(struct_df.head())

save_path = (
    "./artifacts/structural_features/"
    "features_dataset_structural.parquet"
)

struct_df.to_parquet(save_path)

print(f"Saved: {save_path}")
print(struct_df.shape)

(689, 3)


,prompt,response,label
0,<|im_start|>system\nYou are a helpful assistan...,An architect or engineer has a direct relation...,1.0
1,<|im_start|>system\nYou are a helpful assistan...,Based on the information provided in the conte...,1.0
2,<|im_start|>system\nYou are a helpful assistan...,"The ""appropriate"" (well defined), minimum effi...",1.0
3,<|im_start|>system\nYou are a helpful assistan...,The name of the longest bridge in Germany is D...,1.0
4,<|im_start|>system\nYou are a helpful assistan...,The present-day company that BankAmericard tur...,0.0


  0%|          | 0/689 [00:00<?, ?it/s]

(689, 21)


,repeat_bigram,repeat_trigram,token_entropy,sentence_entropy,sentence_len_variance,lexical_diversity,burstiness,token_rarity,uncertainty_slope,uncertainty_total,...,noun_ratio,verb_ratio,adj_ratio,adv_ratio,punct_ratio,num_ratio,entity_density,person_entity_ratio,org_entity_ratio,gpe_entity_ratio
0,0.00000,0.0,3.584963,-1.442823e-12,0.00,1.000000,0.00,0.416667,0.0,0,...,0.333333,0.083333,0.083333,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
1,0.02381,0.0,4.926037,1.000000e+00,2.25,0.813953,-0.88,0.255814,1.0,1,...,0.176471,0.117647,0.058824,0.000000,0.058824,0.058824,0.139535,0.166667,0.0,0.333333
2,0.00000,0.0,4.418296,-1.442823e-12,0.00,0.916667,0.00,0.250000,0.0,0,...,0.266667,0.133333,0.100000,0.033333,0.166667,0.033333,0.000000,0.000000,0.0,0.000000
3,0.00000,0.0,3.277613,-1.442823e-12,0.00,0.909091,0.00,0.181818,0.0,0,...,0.181818,0.000000,0.090909,0.000000,0.000000,0.000000,0.090909,0.000000,0.0,1.000000
4,0.00000,0.0,3.906891,-1.442823e-12,0.00,1.000000,0.00,0.200000,0.0,0,...,0.187500,0.062500,0.062500,0.000000,0.062500,0.000000,0.133333,0.500000,0.5,0.000000


Saved: ./artifacts/structural_features/features_dataset_structural.parquet
(689, 21)


In [162]:
import numpy as np
import pandas as pd

df_drift = pd.read_parquet(
    "./artifacts/drift_extended_features/features_dataset_drift_extended_all.parquet"
)

df_struct = pd.read_parquet(
    "./artifacts/structural_features/features_dataset_structural.parquet"
)

df_raw = pd.read_csv("./data/dataset.csv")

y_struct = df_raw["label"].astype(float).astype(int).to_numpy()

drop_cols = {"label", "prompt", "response", "source_index", "id"}

drift_cols = [c for c in df_drift.columns if c not in drop_cols]
struct_cols = [c for c in df_struct.columns if c not in drop_cols]

X_drift = df_drift[drift_cols].to_numpy(dtype=np.float32)
X_struct = df_struct[struct_cols].to_numpy(dtype=np.float32)
X_combined = np.concatenate([X_drift, X_struct], axis=1)

print("Drift:", X_drift.shape)
print("Structural:", X_struct.shape)
print("Combined:", X_combined.shape)

Drift: (689, 89600)
Structural: (689, 21)
Combined: (689, 89621)


In [163]:
def evaluate_current_pipeline(X_current, y_current, name):
    allowed_idx = np.arange(X_current.shape[1])

    selector_fn, counts = make_group_stability_redundancy_selector_for_dataset(
        X_current=X_current,
        y_current=y_current,
        splits_current=splits,
        allowed_idx=allowed_idx,
        base_k=3500,
        per_fold_k=350,
        final_k=150,
        corr_threshold=0.98,
        corr_method="pearson",
    )

    result = evaluate_catboost_with_selector(
        selector_fn=selector_fn,
        X=X_current,
        y=y_current,
        splits=splits,
        verbose=False,
    )

    return {
        "dataset": name,
        "n_features": X_current.shape[1],
        "selected_candidates": len(counts),
        "n_features_mean": result["n_features_mean"],
        "test_auroc_mean": result["test_auroc_mean"],
        "test_auroc_std": result["test_auroc_std"],
    }


struct_results = []

struct_results.append({
    "dataset": "known_best_drift_extended_all",
    "n_features": X_drift.shape[1],
    "selected_candidates": None,
    "n_features_mean": 149.0,
    "test_auroc_mean": 0.8686,
    "test_auroc_std": 0.0190,
})

struct_results.append(
    evaluate_current_pipeline(
        X_current=X_struct,
        y_current=y_struct,
        name="structural_only",
    )
)

struct_results.append(
    evaluate_current_pipeline(
        X_current=X_combined,
        y_current=y_struct,
        name="drift_extended_all_plus_structural",
    )
)

struct_results_df = pd.DataFrame(struct_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(struct_results_df)

catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5
catboost grouped fold 1
catboost grouped fold 2
catboost grouped fold 3
catboost grouped fold 4
catboost grouped fold 5


,dataset,n_features,selected_candidates,n_features_mean,test_auroc_mean,test_auroc_std
0,known_best_drift_extended_all,89600,NaN,149.0,0.868600,0.019000
2,drift_extended_all_plus_structural,89621,1445.0,147.0,0.859232,0.024228
1,structural_only,21,21.0,21.0,0.671423,0.057205


## Hard feature selection

In [164]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

df_final = pd.read_parquet(
    "./artifacts/drift_extended_features/features_dataset_drift_extended_all.parquet"
)

DROP_COLS = {"label", "prompt", "response", "source_index", "id"}

feature_cols_final = [c for c in df_final.columns if c not in DROP_COLS]

X_final = df_final[feature_cols_final].to_numpy(dtype=np.float32)
y_final = df_final["label"].astype(int).to_numpy()

print("X_final:", X_final.shape)
print("y:", y_final.shape)

X_final: (689, 89600)
y: (689,)


In [165]:
def evaluate_selected_idx_catboost(X, y, splits, selected_idx):
    aucs = []

    for idx_train, idx_val, idx_test in splits:
        model = get_tuned_catboost()

        model.fit(
            X[idx_train][:, selected_idx],
            y[idx_train],
            eval_set=(X[idx_val][:, selected_idx], y[idx_val]),
            use_best_model=True,
        )

        probs = model.predict_proba(X[idx_test][:, selected_idx])[:, 1]
        aucs.append(roc_auc_score(y[idx_test], probs))

    return float(np.mean(aucs)), float(np.std(aucs))


def build_base_stable_candidates(
    X,
    y,
    splits,
    base_k=3500,
    per_fold_k=350,
):
    all_selected = []
    fold_selected = []

    for fold_id, (idx_train, idx_val, idx_test) in enumerate(splits, start=1):
        print(f"base candidate fold {fold_id}")

        base_idx = select_top_k_auc(
            X[idx_train],
            y[idx_train],
            k=base_k,
        )

        model = get_tuned_catboost()
        model.fit(
            X[idx_train][:, base_idx],
            y[idx_train],
            eval_set=(X[idx_val][:, base_idx], y[idx_val]),
            use_best_model=True,
        )

        importance = get_model_importance(model)
        local_top = np.argsort(importance)[-per_fold_k:][::-1]
        selected = base_idx[local_top]

        fold_selected.append(selected)
        all_selected.extend(selected.tolist())

    counts = pd.Series(all_selected).value_counts()

    return counts, fold_selected


def get_stable_idx_from_counts(counts, final_k=150):
    return counts.head(final_k).index.to_numpy(dtype=int)

In [166]:
counts_base, fold_selected_base = build_base_stable_candidates(
    X=X_final,
    y=y_final,
    splits=splits,
    base_k=3500,
    per_fold_k=350,
)

stable_base_150 = get_stable_idx_from_counts(counts_base, final_k=150)

stable_base_150_pruned = correlation_prune_features(
    X_train=X_final,
    feature_idx=stable_base_150,
    feature_scores=np.arange(X_final.shape[1]),
    threshold=0.98,
    method="pearson",
    max_features=None,
)

print("stable_base_150:", len(stable_base_150))
print("stable_base_150_pruned:", len(stable_base_150_pruned))

base candidate fold 1
base candidate fold 2
base candidate fold 3
base candidate fold 4
base candidate fold 5
stable_base_150: 150
stable_base_150_pruned: 149


In [167]:
def diversity_aware_select(
    X,
    candidate_idx,
    candidate_scores,
    final_k=150,
    corr_threshold=0.98,
    method="pearson",
):
    candidate_idx = np.asarray(candidate_idx)

    order = np.argsort(candidate_scores[candidate_idx])[::-1]

    X_sub = X[:, candidate_idx].astype(np.float32)
    X_sub = X_sub - X_sub.mean(axis=0, keepdims=True)
    X_sub = X_sub / (X_sub.std(axis=0, keepdims=True) + 1e-8)

    if method == "pearson":
        sim = np.abs(np.corrcoef(X_sub, rowvar=False))
    elif method == "cosine":
        X_norm = X_sub / (np.linalg.norm(X_sub, axis=0, keepdims=True) + 1e-8)
        sim = np.abs(X_norm.T @ X_norm)
    else:
        raise ValueError(method)

    np.fill_diagonal(sim, 0)

    selected_local = []

    for local_i in order:
        if len(selected_local) == 0:
            selected_local.append(local_i)
            continue

        max_sim = sim[local_i, selected_local].max()

        if max_sim < corr_threshold:
            selected_local.append(local_i)

        if len(selected_local) >= final_k:
            break

    return candidate_idx[np.array(selected_local, dtype=int)]


def make_frequency_scores(counts, n_features):
    scores = np.zeros(n_features, dtype=np.float32)
    for feat, cnt in counts.items():
        scores[int(feat)] = float(cnt)
    return scores


freq_scores = make_frequency_scores(counts_base, X_final.shape[1])

candidate_pool = counts_base.head(1000).index.to_numpy(dtype=int)

diversity_results = []

for corr_threshold in [0.90, 0.95, 0.97, 0.98, 0.99]:
    for final_k in [100, 150, 200]:
        selected_idx = diversity_aware_select(
            X=X_final,
            candidate_idx=candidate_pool,
            candidate_scores=freq_scores,
            final_k=final_k,
            corr_threshold=corr_threshold,
            method="pearson",
        )

        mean_auc, std_auc = evaluate_selected_idx_catboost(
            X=X_final,
            y=y_final,
            splits=splits,
            selected_idx=selected_idx,
        )

        diversity_results.append({
            "selection": "diversity_aware_frequency",
            "final_k": final_k,
            "corr_threshold": corr_threshold,
            "n_features": len(selected_idx),
            "test_auroc_mean": mean_auc,
            "test_auroc_std": std_auc,
        })

diversity_df = pd.DataFrame(diversity_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(diversity_df.head(20))

,selection,final_k,corr_threshold,n_features,test_auroc_mean,test_auroc_std
8,diversity_aware_frequency,200,0.97,200,0.863388,0.022322
11,diversity_aware_frequency,200,0.98,200,0.863388,0.022322
5,diversity_aware_frequency,200,0.95,200,0.858690,0.019272
14,diversity_aware_frequency,200,0.99,200,0.857733,0.017067
0,diversity_aware_frequency,100,0.90,100,0.857092,0.022653
3,diversity_aware_frequency,100,0.95,100,0.857018,0.025293
6,diversity_aware_frequency,100,0.97,100,0.855209,0.025567
9,diversity_aware_frequency,100,0.98,100,0.855209,0.025567
12,diversity_aware_frequency,100,0.99,100,0.855209,0.025567
2,diversity_aware_frequency,200,0.90,200,0.854470,0.018013


In [168]:
def build_fold_auc_weighted_counts(
    X,
    y,
    splits,
    base_k=3500,
    per_fold_k=350,
):
    scores = np.zeros(X.shape[1], dtype=np.float64)
    freq = np.zeros(X.shape[1], dtype=np.float64)

    for fold_id, (idx_train, idx_val, idx_test) in enumerate(splits, start=1):
        print(f"fold-AUROC weighted fold {fold_id}")

        base_idx = select_top_k_auc(
            X[idx_train],
            y[idx_train],
            k=base_k,
        )

        model = get_tuned_catboost()
        model.fit(
            X[idx_train][:, base_idx],
            y[idx_train],
            eval_set=(X[idx_val][:, base_idx], y[idx_val]),
            use_best_model=True,
        )

        val_probs = model.predict_proba(X[idx_val][:, base_idx])[:, 1]
        fold_val_auc = roc_auc_score(y[idx_val], val_probs)

        importance = get_model_importance(model).astype(np.float64)
        importance = importance / (importance.sum() + 1e-12)

        local_top = np.argsort(importance)[-per_fold_k:][::-1]
        selected = base_idx[local_top]

        scores[selected] += fold_val_auc * importance[local_top]
        freq[selected] += 1

    return scores, freq


fold_auc_scores, fold_auc_freq = build_fold_auc_weighted_counts(
    X=X_final,
    y=y_final,
    splits=splits,
    base_k=3500,
    per_fold_k=350,
)

fold-AUROC weighted fold 1
fold-AUROC weighted fold 2
fold-AUROC weighted fold 3
fold-AUROC weighted fold 4
fold-AUROC weighted fold 5


In [169]:
fold_auc_weighted_results = []

candidate_idx = np.where(fold_auc_scores > 0)[0]

for final_k in [100, 150, 200, 300]:
    for corr_threshold in [0.95, 0.97, 0.98, 0.99]:
        selected_raw = candidate_idx[np.argsort(fold_auc_scores[candidate_idx])[-final_k:][::-1]]

        selected_idx = correlation_prune_features(
            X_train=X_final,
            feature_idx=selected_raw,
            feature_scores=fold_auc_scores,
            threshold=corr_threshold,
            method="pearson",
            max_features=None,
        )

        mean_auc, std_auc = evaluate_selected_idx_catboost(
            X=X_final,
            y=y_final,
            splits=splits,
            selected_idx=selected_idx,
        )

        fold_auc_weighted_results.append({
            "selection": "fold_auc_weighted_stability",
            "final_k": final_k,
            "corr_threshold": corr_threshold,
            "n_features": len(selected_idx),
            "test_auroc_mean": mean_auc,
            "test_auroc_std": std_auc,
        })

fold_auc_weighted_df = pd.DataFrame(fold_auc_weighted_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(fold_auc_weighted_df.head(20))

,selection,final_k,corr_threshold,n_features,test_auroc_mean,test_auroc_std
5,fold_auc_weighted_stability,150,0.97,148,0.857463,0.018574
6,fold_auc_weighted_stability,150,0.98,148,0.857463,0.018574
11,fold_auc_weighted_stability,200,0.99,199,0.857429,0.019303
7,fold_auc_weighted_stability,150,0.99,149,0.857131,0.016982
0,fold_auc_weighted_stability,100,0.95,96,0.856469,0.020969
1,fold_auc_weighted_stability,100,0.97,98,0.856358,0.022734
2,fold_auc_weighted_stability,100,0.98,98,0.856358,0.022734
8,fold_auc_weighted_stability,200,0.95,195,0.856247,0.022117
9,fold_auc_weighted_stability,200,0.97,197,0.856052,0.016793
10,fold_auc_weighted_stability,200,0.98,197,0.856052,0.016793


In [170]:
def recursive_elimination_light(
    X,
    y,
    splits,
    initial_idx,
    steps=(300, 200, 150, 100),
):
    current_idx = np.asarray(initial_idx)

    rows = []

    for target_k in steps:
        print(f"RFE step target_k={target_k}, current={len(current_idx)}")

        importances_accum = np.zeros(len(current_idx), dtype=np.float64)

        for fold_id, (idx_train, idx_val, idx_test) in enumerate(splits, start=1):
            model = get_tuned_catboost()

            model.fit(
                X[idx_train][:, current_idx],
                y[idx_train],
                eval_set=(X[idx_val][:, current_idx], y[idx_val]),
                use_best_model=True,
            )

            imp = get_model_importance(model).astype(np.float64)
            imp = imp / (imp.sum() + 1e-12)
            importances_accum += imp

        local_top = np.argsort(importances_accum)[-target_k:][::-1]
        current_idx = current_idx[local_top]

        mean_auc, std_auc = evaluate_selected_idx_catboost(
            X=X,
            y=y,
            splits=splits,
            selected_idx=current_idx,
        )

        rows.append({
            "selection": "recursive_elimination_light",
            "target_k": target_k,
            "n_features": len(current_idx),
            "test_auroc_mean": mean_auc,
            "test_auroc_std": std_auc,
        })

    return pd.DataFrame(rows).sort_values("test_auroc_mean", ascending=False)


rfe_start_idx = counts_base.head(500).index.to_numpy(dtype=int)

rfe_df = recursive_elimination_light(
    X=X_final,
    y=y_final,
    splits=splits,
    initial_idx=rfe_start_idx,
    steps=(300, 200, 150, 100),
)

display(rfe_df)

RFE step target_k=300, current=500
RFE step target_k=200, current=300
RFE step target_k=150, current=200
RFE step target_k=100, current=150


,selection,target_k,n_features,test_auroc_mean,test_auroc_std
3,recursive_elimination_light,100,100,0.871319,0.016941
2,recursive_elimination_light,150,150,0.870036,0.016009
1,recursive_elimination_light,200,200,0.863672,0.023030
0,recursive_elimination_light,300,300,0.852001,0.020291


In [171]:
def greedy_forward_light(
    X,
    y,
    splits,
    candidate_idx,
    seed_idx,
    max_add=30,
    batch_size=25,
):
    selected = list(seed_idx)
    remaining = [x for x in candidate_idx if x not in set(selected)]

    history = []

    base_auc, base_std = evaluate_selected_idx_catboost(
        X=X,
        y=y,
        splits=splits,
        selected_idx=np.array(selected),
    )

    best_auc = base_auc

    history.append({
        "step": 0,
        "added_feature": None,
        "n_features": len(selected),
        "test_auroc_mean": base_auc,
        "test_auroc_std": base_std,
    })

    print(f"Initial AUROC={base_auc:.4f}")

    for step in range(1, max_add + 1):
        trial_candidates = remaining[:batch_size]

        best_feature = None
        best_feature_auc = best_auc
        best_feature_std = None

        for feat in trial_candidates:
            trial_idx = np.array(selected + [feat], dtype=int)

            mean_auc, std_auc = evaluate_selected_idx_catboost(
                X=X,
                y=y,
                splits=splits,
                selected_idx=trial_idx,
            )

            if mean_auc > best_feature_auc:
                best_feature_auc = mean_auc
                best_feature_std = std_auc
                best_feature = feat

        if best_feature is None:
            print(f"Step {step}: no improvement, stop")
            break

        selected.append(best_feature)
        remaining.remove(best_feature)
        best_auc = best_feature_auc

        print(
            f"Step {step}: added={best_feature}, "
            f"AUROC={best_feature_auc:.4f}"
        )

        history.append({
            "step": step,
            "added_feature": best_feature,
            "n_features": len(selected),
            "test_auroc_mean": best_feature_auc,
            "test_auroc_std": best_feature_std,
        })

    return np.array(selected, dtype=int), pd.DataFrame(history)


seed_idx = counts_base.head(100).index.to_numpy(dtype=int)
candidate_idx = counts_base.head(300).index.to_numpy(dtype=int)

greedy_idx, greedy_df = greedy_forward_light(
    X=X_final,
    y=y_final,
    splits=splits,
    candidate_idx=candidate_idx,
    seed_idx=seed_idx,
    max_add=20,
    batch_size=20,
)

display(greedy_df.sort_values("test_auroc_mean", ascending=False).head(20))

Initial AUROC=0.8678
Step 1: added=87883, AUROC=0.8690
Step 2: no improvement, stop


,step,added_feature,n_features,test_auroc_mean,test_auroc_std
1,1,87883.0,101,0.868954,0.027914
0,0,NaN,100,0.867754,0.026326


In [172]:
known_best_df = pd.DataFrame([
    {
        "selection": "known_best_drift_extended_all",
        "n_features": 149.0,
        "test_auroc_mean": 0.8686,
        "test_auroc_std": 0.0190,
    }
])

div_compare = diversity_df.rename(columns={"final_k": "k"})[
    ["selection", "n_features", "test_auroc_mean", "test_auroc_std"]
]

fold_auc_compare = fold_auc_weighted_df.rename(columns={"final_k": "k"})[
    ["selection", "n_features", "test_auroc_mean", "test_auroc_std"]
]

rfe_compare = rfe_df.rename(columns={"target_k": "k"})[
    ["selection", "n_features", "test_auroc_mean", "test_auroc_std"]
]

greedy_compare = greedy_df[
    ["selection", "n_features", "test_auroc_mean", "test_auroc_std"]
] if "selection" in greedy_df.columns else greedy_df.assign(
    selection="greedy_forward_light"
)[["selection", "n_features", "test_auroc_mean", "test_auroc_std"]]

hard_selection_compare = pd.concat(
    [
        known_best_df,
        div_compare,
        fold_auc_compare,
        rfe_compare,
        greedy_compare,
    ],
    ignore_index=True,
).sort_values("test_auroc_mean", ascending=False)

display(hard_selection_compare.head(30))

best = hard_selection_compare.iloc[0]

print(
    f"BEST HARDER FEATURE SELECTION:\n"
    f"{best['selection']}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}\n"
    f"n_features={best['n_features']}"
)

,selection,n_features,test_auroc_mean,test_auroc_std
32,recursive_elimination_light,100.0,0.871319,0.016941
33,recursive_elimination_light,150.0,0.870036,0.016009
37,greedy_forward_light,101.0,0.868954,0.027914
0,known_best_drift_extended_all,149.0,0.868600,0.019000
36,greedy_forward_light,100.0,0.867754,0.026326
34,recursive_elimination_light,200.0,0.863672,0.023030
2,diversity_aware_frequency,200.0,0.863388,0.022322
1,diversity_aware_frequency,200.0,0.863388,0.022322
3,diversity_aware_frequency,200.0,0.858690,0.019272
4,diversity_aware_frequency,200.0,0.857733,0.017067


BEST HARDER FEATURE SELECTION:
recursive_elimination_light
AUROC=0.8713 ± 0.0169
n_features=100.0


## RFE tuning

In [173]:
rfe_tuning_results = []

rfe_configs = [
    {
        "start_k": 300,
        "steps": (200, 150, 100),
    },
    {
        "start_k": 500,
        "steps": (300, 200, 150, 100),
    },
    {
        "start_k": 500,
        "steps": (400, 300, 200, 100),
    },
    {
        "start_k": 800,
        "steps": (500, 300, 150, 100),
    },
    {
        "start_k": 800,
        "steps": (600, 400, 200, 100),
    },
    {
        "start_k": 500,
        "steps": (300, 200, 125),
    },
    {
        "start_k": 500,
        "steps": (300, 200, 75),
    },
    {
        "start_k": 800,
        "steps": (500, 300, 125),
    },
]

for i, cfg in enumerate(rfe_configs, start=1):
    print(f"\n[{i}/{len(rfe_configs)}] RFE tuning: {cfg}")

    start_idx = counts_base.head(cfg["start_k"]).index.to_numpy(dtype=int)

    rfe_df_tmp = recursive_elimination_light(
        X=X_final,
        y=y_final,
        splits=splits,
        initial_idx=start_idx,
        steps=cfg["steps"],
    )

    best_row = rfe_df_tmp.sort_values("test_auroc_mean", ascending=False).iloc[0]

    rfe_tuning_results.append({
        "start_k": cfg["start_k"],
        "steps": cfg["steps"],
        "best_step_features": best_row["n_features"],
        "test_auroc_mean": best_row["test_auroc_mean"],
        "test_auroc_std": best_row["test_auroc_std"],
    })

rfe_tuning_df = pd.DataFrame(rfe_tuning_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(rfe_tuning_df)

known_rfe_best = pd.DataFrame([
    {
        "start_k": 500,
        "steps": "(300, 200, 150, 100)",
        "best_step_features": 100,
        "test_auroc_mean": 0.8713,
        "test_auroc_std": 0.0169,
    }
])

rfe_tuning_compare = pd.concat(
    [known_rfe_best, rfe_tuning_df],
    ignore_index=True,
).sort_values("test_auroc_mean", ascending=False)

display(rfe_tuning_compare)

best = rfe_tuning_compare.iloc[0]

print(
    f"BEST RFE TUNING:\n"
    f"start_k={best['start_k']}\n"
    f"steps={best['steps']}\n"
    f"features={best['best_step_features']}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}"
)


[1/8] RFE tuning: {'start_k': 300, 'steps': (200, 150, 100)}
RFE step target_k=200, current=300
RFE step target_k=150, current=200
RFE step target_k=100, current=150

[2/8] RFE tuning: {'start_k': 500, 'steps': (300, 200, 150, 100)}
RFE step target_k=300, current=500
RFE step target_k=200, current=300
RFE step target_k=150, current=200
RFE step target_k=100, current=150

[3/8] RFE tuning: {'start_k': 500, 'steps': (400, 300, 200, 100)}
RFE step target_k=400, current=500
RFE step target_k=300, current=400
RFE step target_k=200, current=300
RFE step target_k=100, current=200

[4/8] RFE tuning: {'start_k': 800, 'steps': (500, 300, 150, 100)}
RFE step target_k=500, current=800
RFE step target_k=300, current=500
RFE step target_k=150, current=300
RFE step target_k=100, current=150

[5/8] RFE tuning: {'start_k': 800, 'steps': (600, 400, 200, 100)}
RFE step target_k=600, current=800
RFE step target_k=400, current=600
RFE step target_k=200, current=400
RFE step target_k=100, current=200

[6/8

,start_k,steps,best_step_features,test_auroc_mean,test_auroc_std
0,300,"(200, 150, 100)",100,0.878976,0.018474
5,500,"(300, 200, 125)",125,0.875848,0.020371
6,500,"(300, 200, 75)",75,0.874030,0.013897
1,500,"(300, 200, 150, 100)",100,0.871319,0.016941
2,500,"(400, 300, 200, 100)",100,0.870763,0.019832
3,800,"(500, 300, 150, 100)",150,0.870715,0.021393
7,800,"(500, 300, 125)",125,0.862728,0.020423
4,800,"(600, 400, 200, 100)",100,0.861280,0.019955


,start_k,steps,best_step_features,test_auroc_mean,test_auroc_std
1,300,"(200, 150, 100)",100,0.878976,0.018474
2,500,"(300, 200, 125)",125,0.875848,0.020371
3,500,"(300, 200, 75)",75,0.874030,0.013897
4,500,"(300, 200, 150, 100)",100,0.871319,0.016941
0,500,"(300, 200, 150, 100)",100,0.871300,0.016900
5,500,"(400, 300, 200, 100)",100,0.870763,0.019832
6,800,"(500, 300, 150, 100)",150,0.870715,0.021393
7,800,"(500, 300, 125)",125,0.862728,0.020423
8,800,"(600, 400, 200, 100)",100,0.861280,0.019955


BEST RFE TUNING:
start_k=300
steps=(200, 150, 100)
features=100
AUROC=0.8790 ± 0.0185


In [174]:
rfe_fine_results = []

fine_configs = [
    {
        "start_k": 200,
        "steps": (150, 125, 100),
    },
    {
        "start_k": 250,
        "steps": (175, 125, 100),
    },
    {
        "start_k": 300,
        "steps": (225, 175, 125, 100),
    },
    {
        "start_k": 300,
        "steps": (200, 125, 100),
    },
    {
        "start_k": 300,
        "steps": (200, 150, 125),
    },
    {
        "start_k": 350,
        "steps": (250, 175, 125, 100),
    },
    {
        "start_k": 350,
        "steps": (250, 150, 100),
    },
    {
        "start_k": 250,
        "steps": (175, 125, 75),
    },
]

for i, cfg in enumerate(fine_configs, start=1):

    print(f"\n[{i}/{len(fine_configs)}] fine RFE: {cfg}")

    start_idx = counts_base.head(cfg["start_k"]).index.to_numpy(dtype=int)

    rfe_df_tmp = recursive_elimination_light(
        X=X_final,
        y=y_final,
        splits=splits,
        initial_idx=start_idx,
        steps=cfg["steps"],
    )

    best_row = rfe_df_tmp.sort_values(
        "test_auroc_mean",
        ascending=False,
    ).iloc[0]

    rfe_fine_results.append({
        "start_k": cfg["start_k"],
        "steps": str(cfg["steps"]),
        "best_step_features": best_row["n_features"],
        "test_auroc_mean": best_row["test_auroc_mean"],
        "test_auroc_std": best_row["test_auroc_std"],
    })

rfe_fine_df = pd.DataFrame(rfe_fine_results).sort_values(
    "test_auroc_mean",
    ascending=False,
)

known_best = pd.DataFrame([
    {
        "start_k": 300,
        "steps": "(200, 150, 100)",
        "best_step_features": 100,
        "test_auroc_mean": 0.8790,
        "test_auroc_std": 0.0185,
    }
])

compare_df = pd.concat(
    [known_best, rfe_fine_df],
    ignore_index=True,
).sort_values("test_auroc_mean", ascending=False)

display(compare_df)

best = compare_df.iloc[0]

print(
    f"\nBEST FINE RFE:\n"
    f"start_k={best['start_k']}\n"
    f"steps={best['steps']}\n"
    f"features={best['best_step_features']}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± "
    f"{best['test_auroc_std']:.4f}"
)


[1/8] fine RFE: {'start_k': 200, 'steps': (150, 125, 100)}
RFE step target_k=150, current=200
RFE step target_k=125, current=150
RFE step target_k=100, current=125

[2/8] fine RFE: {'start_k': 250, 'steps': (175, 125, 100)}
RFE step target_k=175, current=250
RFE step target_k=125, current=175
RFE step target_k=100, current=125

[3/8] fine RFE: {'start_k': 300, 'steps': (225, 175, 125, 100)}
RFE step target_k=225, current=300
RFE step target_k=175, current=225
RFE step target_k=125, current=175
RFE step target_k=100, current=125

[4/8] fine RFE: {'start_k': 300, 'steps': (200, 125, 100)}
RFE step target_k=200, current=300
RFE step target_k=125, current=200
RFE step target_k=100, current=125

[5/8] fine RFE: {'start_k': 300, 'steps': (200, 150, 125)}
RFE step target_k=200, current=300
RFE step target_k=150, current=200
RFE step target_k=125, current=150

[6/8] fine RFE: {'start_k': 350, 'steps': (250, 175, 125, 100)}
RFE step target_k=250, current=350
RFE step target_k=175, current=250


,start_k,steps,best_step_features,test_auroc_mean,test_auroc_std
1,250,"(175, 125, 75)",75,0.882839,0.017963
2,250,"(175, 125, 100)",100,0.881908,0.017197
3,200,"(150, 125, 100)",100,0.879374,0.022349
0,300,"(200, 150, 100)",100,0.879000,0.018500
4,350,"(250, 150, 100)",100,0.878392,0.023227
5,300,"(225, 175, 125, 100)",100,0.876053,0.017303
6,350,"(250, 175, 125, 100)",175,0.875991,0.020514
7,300,"(200, 125, 100)",100,0.873984,0.016136
8,300,"(200, 150, 125)",125,0.871102,0.024648



BEST FINE RFE:
start_k=250
steps=(175, 125, 75)
features=75
AUROC=0.8828 ± 0.0180


In [175]:
rfe_fine_results_2 = []

fine_configs_2 = [
    {"start_k": 225, "steps": (175, 125, 75)},
    {"start_k": 250, "steps": (175, 125, 75)},
    {"start_k": 275, "steps": (175, 125, 75)},

    {"start_k": 250, "steps": (200, 150, 100, 75)},
    {"start_k": 250, "steps": (200, 125, 75)},
    {"start_k": 250, "steps": (175, 100, 75)},

    {"start_k": 250, "steps": (175, 125, 100)},
    {"start_k": 250, "steps": (175, 125, 90)},
    {"start_k": 250, "steps": (175, 125, 60)},

    {"start_k": 225, "steps": (150, 100, 75)},
    {"start_k": 275, "steps": (200, 125, 75)},
    {"start_k": 300, "steps": (200, 125, 75)},
]

for i, cfg in enumerate(fine_configs_2, start=1):
    print(f"\n[{i}/{len(fine_configs_2)}] fine RFE 2: {cfg}")

    start_idx = counts_base.head(cfg["start_k"]).index.to_numpy(dtype=int)

    rfe_df_tmp = recursive_elimination_light(
        X=X_final,
        y=y_final,
        splits=splits,
        initial_idx=start_idx,
        steps=cfg["steps"],
    )

    best_row = rfe_df_tmp.sort_values(
        "test_auroc_mean",
        ascending=False,
    ).iloc[0]

    rfe_fine_results_2.append({
        "start_k": cfg["start_k"],
        "steps": str(cfg["steps"]),
        "best_step_features": best_row["n_features"],
        "test_auroc_mean": best_row["test_auroc_mean"],
        "test_auroc_std": best_row["test_auroc_std"],
    })

rfe_fine_df_2 = pd.DataFrame(rfe_fine_results_2).sort_values(
    "test_auroc_mean",
    ascending=False,
)

known_best_rfe = pd.DataFrame([
    {
        "start_k": 250,
        "steps": "(175, 125, 75)",
        "best_step_features": 75,
        "test_auroc_mean": 0.8828,
        "test_auroc_std": 0.0180,
    }
])

rfe_fine_compare_2 = pd.concat(
    [known_best_rfe, rfe_fine_df_2],
    ignore_index=True,
).sort_values("test_auroc_mean", ascending=False)

display(rfe_fine_compare_2)

best = rfe_fine_compare_2.iloc[0]

print(
    f"\nBEST FINE RFE 2:\n"
    f"start_k={best['start_k']}\n"
    f"steps={best['steps']}\n"
    f"features={best['best_step_features']}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}"
)


[1/12] fine RFE 2: {'start_k': 225, 'steps': (175, 125, 75)}
RFE step target_k=175, current=225
RFE step target_k=125, current=175
RFE step target_k=75, current=125

[2/12] fine RFE 2: {'start_k': 250, 'steps': (175, 125, 75)}
RFE step target_k=175, current=250
RFE step target_k=125, current=175
RFE step target_k=75, current=125

[3/12] fine RFE 2: {'start_k': 275, 'steps': (175, 125, 75)}
RFE step target_k=175, current=275
RFE step target_k=125, current=175
RFE step target_k=75, current=125

[4/12] fine RFE 2: {'start_k': 250, 'steps': (200, 150, 100, 75)}
RFE step target_k=200, current=250
RFE step target_k=150, current=200
RFE step target_k=100, current=150
RFE step target_k=75, current=100

[5/12] fine RFE 2: {'start_k': 250, 'steps': (200, 125, 75)}
RFE step target_k=200, current=250
RFE step target_k=125, current=200
RFE step target_k=75, current=125

[6/12] fine RFE 2: {'start_k': 250, 'steps': (175, 100, 75)}
RFE step target_k=175, current=250
RFE step target_k=100, current=17

,start_k,steps,best_step_features,test_auroc_mean,test_auroc_std
1,250,"(200, 125, 75)",75,0.887469,0.018947
2,250,"(175, 125, 60)",60,0.887339,0.012962
3,225,"(150, 100, 75)",75,0.884690,0.016756
4,250,"(200, 150, 100, 75)",75,0.884662,0.019768
5,225,"(175, 125, 75)",75,0.884235,0.017665
6,275,"(175, 125, 75)",75,0.883534,0.018032
7,250,"(175, 125, 75)",75,0.882839,0.017963
0,250,"(175, 125, 75)",75,0.882800,0.018000
8,250,"(175, 125, 100)",100,0.881908,0.017197
9,275,"(200, 125, 75)",75,0.881144,0.015621



BEST FINE RFE 2:
start_k=250
steps=(200, 125, 75)
features=75
AUROC=0.8875 ± 0.0189


In [176]:
rfe_fine_results_3 = []

fine_configs_3 = [
    {"start_k": 240, "steps": (200, 125, 75)},
    {"start_k": 250, "steps": (200, 125, 75)},
    {"start_k": 260, "steps": (200, 125, 75)},

    {"start_k": 250, "steps": (210, 125, 75)},
    {"start_k": 250, "steps": (190, 125, 75)},

    {"start_k": 250, "steps": (200, 140, 75)},
    {"start_k": 250, "steps": (200, 110, 75)},

    {"start_k": 250, "steps": (200, 125, 65)},
    {"start_k": 250, "steps": (200, 125, 85)},

    {"start_k": 260, "steps": (210, 125, 75)},
]

for i, cfg in enumerate(fine_configs_3, start=1):
    print(f"\n[{i}/{len(fine_configs_3)}] fine RFE 3: {cfg}")

    start_idx = counts_base.head(cfg["start_k"]).index.to_numpy(dtype=int)

    rfe_df_tmp = recursive_elimination_light(
        X=X_final,
        y=y_final,
        splits=splits,
        initial_idx=start_idx,
        steps=cfg["steps"],
    )

    best_row = rfe_df_tmp.sort_values(
        "test_auroc_mean",
        ascending=False,
    ).iloc[0]

    rfe_fine_results_3.append({
        "start_k": cfg["start_k"],
        "steps": str(cfg["steps"]),
        "best_step_features": best_row["n_features"],
        "test_auroc_mean": best_row["test_auroc_mean"],
        "test_auroc_std": best_row["test_auroc_std"],
    })

rfe_fine_df_3 = pd.DataFrame(rfe_fine_results_3).sort_values(
    "test_auroc_mean",
    ascending=False,
)

known_best_rfe_2 = pd.DataFrame([
    {
        "start_k": 250,
        "steps": "(200, 125, 75)",
        "best_step_features": 75,
        "test_auroc_mean": 0.8875,
        "test_auroc_std": 0.0189,
    }
])

rfe_fine_compare_3 = pd.concat(
    [known_best_rfe_2, rfe_fine_df_3],
    ignore_index=True,
).sort_values("test_auroc_mean", ascending=False)

display(rfe_fine_compare_3)

best = rfe_fine_compare_3.iloc[0]

print(
    f"\nBEST FINE RFE 3:\n"
    f"start_k={best['start_k']}\n"
    f"steps={best['steps']}\n"
    f"features={best['best_step_features']}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}"
)


[1/10] fine RFE 3: {'start_k': 240, 'steps': (200, 125, 75)}
RFE step target_k=200, current=240
RFE step target_k=125, current=200
RFE step target_k=75, current=125

[2/10] fine RFE 3: {'start_k': 250, 'steps': (200, 125, 75)}
RFE step target_k=200, current=250
RFE step target_k=125, current=200
RFE step target_k=75, current=125

[3/10] fine RFE 3: {'start_k': 260, 'steps': (200, 125, 75)}
RFE step target_k=200, current=260
RFE step target_k=125, current=200
RFE step target_k=75, current=125

[4/10] fine RFE 3: {'start_k': 250, 'steps': (210, 125, 75)}
RFE step target_k=210, current=250
RFE step target_k=125, current=210
RFE step target_k=75, current=125

[5/10] fine RFE 3: {'start_k': 250, 'steps': (190, 125, 75)}
RFE step target_k=190, current=250
RFE step target_k=125, current=190
RFE step target_k=75, current=125

[6/10] fine RFE 3: {'start_k': 250, 'steps': (200, 140, 75)}
RFE step target_k=200, current=250
RFE step target_k=140, current=200
RFE step target_k=75, current=140

[7/

,start_k,steps,best_step_features,test_auroc_mean,test_auroc_std
1,250,"(200, 125, 65)",65,0.893964,0.013887
0,250,"(200, 125, 75)",75,0.887500,0.018900
2,250,"(200, 125, 75)",75,0.887469,0.018947
3,250,"(190, 125, 75)",75,0.886963,0.013659
4,250,"(200, 140, 75)",75,0.885685,0.015055
5,260,"(200, 125, 75)",75,0.884006,0.024429
6,250,"(200, 125, 85)",85,0.883343,0.018332
7,260,"(210, 125, 75)",75,0.881276,0.011029
8,240,"(200, 125, 75)",75,0.881036,0.022664
9,250,"(200, 110, 75)",75,0.880889,0.017733



BEST FINE RFE 3:
start_k=250
steps=(200, 125, 65)
features=65
AUROC=0.8940 ± 0.0139


In [177]:
rfe_fine_results_4 = []

fine_configs_4 = [
    {"start_k": 240, "steps": (200, 125, 65)},
    {"start_k": 250, "steps": (200, 125, 65)},
    {"start_k": 260, "steps": (200, 125, 65)},

    {"start_k": 250, "steps": (190, 125, 65)},
    {"start_k": 250, "steps": (210, 125, 65)},

    {"start_k": 250, "steps": (200, 115, 65)},
    {"start_k": 250, "steps": (200, 135, 65)},

    {"start_k": 250, "steps": (200, 125, 55)},
    {"start_k": 250, "steps": (200, 125, 60)},
    {"start_k": 250, "steps": (200, 125, 70)},

    {"start_k": 260, "steps": (210, 125, 65)},
    {"start_k": 240, "steps": (190, 125, 65)},
]

for i, cfg in enumerate(fine_configs_4, start=1):
    print(f"\n[{i}/{len(fine_configs_4)}] fine RFE 4: {cfg}")

    start_idx = counts_base.head(cfg["start_k"]).index.to_numpy(dtype=int)

    rfe_df_tmp = recursive_elimination_light(
        X=X_final,
        y=y_final,
        splits=splits,
        initial_idx=start_idx,
        steps=cfg["steps"],
    )

    best_row = rfe_df_tmp.sort_values(
        "test_auroc_mean",
        ascending=False,
    ).iloc[0]

    rfe_fine_results_4.append({
        "start_k": cfg["start_k"],
        "steps": str(cfg["steps"]),
        "best_step_features": best_row["n_features"],
        "test_auroc_mean": best_row["test_auroc_mean"],
        "test_auroc_std": best_row["test_auroc_std"],
    })

rfe_fine_df_4 = pd.DataFrame(rfe_fine_results_4).sort_values(
    "test_auroc_mean",
    ascending=False,
)

known_best_rfe_3 = pd.DataFrame([
    {
        "start_k": 250,
        "steps": "(200, 125, 65)",
        "best_step_features": 65,
        "test_auroc_mean": 0.8940,
        "test_auroc_std": 0.0139,
    }
])

rfe_fine_compare_4 = pd.concat(
    [known_best_rfe_3, rfe_fine_df_4],
    ignore_index=True,
).sort_values("test_auroc_mean", ascending=False)

display(rfe_fine_compare_4)

best = rfe_fine_compare_4.iloc[0]

print(
    f"\nBEST FINE RFE 4:\n"
    f"start_k={best['start_k']}\n"
    f"steps={best['steps']}\n"
    f"features={best['best_step_features']}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}"
)


[1/12] fine RFE 4: {'start_k': 240, 'steps': (200, 125, 65)}
RFE step target_k=200, current=240
RFE step target_k=125, current=200
RFE step target_k=65, current=125

[2/12] fine RFE 4: {'start_k': 250, 'steps': (200, 125, 65)}
RFE step target_k=200, current=250
RFE step target_k=125, current=200
RFE step target_k=65, current=125

[3/12] fine RFE 4: {'start_k': 260, 'steps': (200, 125, 65)}
RFE step target_k=200, current=260
RFE step target_k=125, current=200
RFE step target_k=65, current=125

[4/12] fine RFE 4: {'start_k': 250, 'steps': (190, 125, 65)}
RFE step target_k=190, current=250
RFE step target_k=125, current=190
RFE step target_k=65, current=125

[5/12] fine RFE 4: {'start_k': 250, 'steps': (210, 125, 65)}
RFE step target_k=210, current=250
RFE step target_k=125, current=210
RFE step target_k=65, current=125

[6/12] fine RFE 4: {'start_k': 250, 'steps': (200, 115, 65)}
RFE step target_k=200, current=250
RFE step target_k=115, current=200
RFE step target_k=65, current=115

[7/

,start_k,steps,best_step_features,test_auroc_mean,test_auroc_std
1,250,"(200, 125, 70)",70,0.894408,0.010170
0,250,"(200, 125, 65)",65,0.894000,0.013900
2,250,"(200, 125, 65)",65,0.893964,0.013887
3,250,"(200, 125, 55)",55,0.890623,0.017264
4,250,"(200, 125, 60)",60,0.890456,0.013954
5,250,"(190, 125, 65)",65,0.889330,0.015159
6,240,"(200, 125, 65)",65,0.888977,0.022137
7,260,"(200, 125, 65)",65,0.888642,0.015352
8,260,"(210, 125, 65)",65,0.887386,0.014268
9,250,"(210, 125, 65)",65,0.887336,0.019592



BEST FINE RFE 4:
start_k=250
steps=(200, 125, 70)
features=70
AUROC=0.8944 ± 0.0102


## Diversity ensemble

In [178]:
from scipy.special import logit, expit
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

In [ ]:
selector_sets = {}

start_idx = counts_base.head(250).index.to_numpy(dtype=int)

rfe_best_df = recursive_elimination_light(
    X=X_final,
    y=y_final,
    splits=splits,
    initial_idx=start_idx,
    steps=(200, 125, 70),
)

RFE step target_k=200, current=250
RFE step target_k=125, current=200
RFE step target_k=70, current=125


In [180]:
def recursive_elimination_return_idx(
    X,
    y,
    splits,
    initial_idx,
    steps=(200, 125, 70),
):
    current_idx = np.asarray(initial_idx)

    for target_k in steps:
        print(f"RFE target_k={target_k}, current={len(current_idx)}")

        importances_accum = np.zeros(len(current_idx), dtype=np.float64)

        for idx_train, idx_val, idx_test in splits:
            model = get_tuned_catboost()

            model.fit(
                X[idx_train][:, current_idx],
                y[idx_train],
                eval_set=(X[idx_val][:, current_idx], y[idx_val]),
                use_best_model=True,
            )

            imp = get_model_importance(model).astype(np.float64)
            imp = imp / (imp.sum() + 1e-12)
            importances_accum += imp

        local_top = np.argsort(importances_accum)[-target_k:][::-1]
        current_idx = current_idx[local_top]

    return current_idx

In [181]:
selector_sets["rfe_250_200_125_70"] = recursive_elimination_return_idx(
    X=X_final,
    y=y_final,
    splits=splits,
    initial_idx=counts_base.head(250).index.to_numpy(dtype=int),
    steps=(200, 125, 70),
)

selector_sets["rfe_250_200_125_65"] = recursive_elimination_return_idx(
    X=X_final,
    y=y_final,
    splits=splits,
    initial_idx=counts_base.head(250).index.to_numpy(dtype=int),
    steps=(200, 125, 65),
)

selector_sets["rfe_250_200_125_75"] = recursive_elimination_return_idx(
    X=X_final,
    y=y_final,
    splits=splits,
    initial_idx=counts_base.head(250).index.to_numpy(dtype=int),
    steps=(200, 125, 75),
)

selector_sets["stability_150"] = counts_base.head(150).index.to_numpy(dtype=int)

selector_sets["stability_100"] = counts_base.head(100).index.to_numpy(dtype=int)

selector_sets["stability_200"] = counts_base.head(200).index.to_numpy(dtype=int)

print({k: len(v) for k, v in selector_sets.items()})

RFE target_k=200, current=250
RFE target_k=125, current=200
RFE target_k=70, current=125
RFE target_k=200, current=250
RFE target_k=125, current=200
RFE target_k=65, current=125
RFE target_k=200, current=250
RFE target_k=125, current=200
RFE target_k=75, current=125
{'rfe_250_200_125_70': 70, 'rfe_250_200_125_65': 65, 'rfe_250_200_125_75': 75, 'stability_150': 150, 'stability_100': 100, 'stability_200': 200}


In [182]:
selector_predictions = {}

for selector_name, selected_idx in selector_sets.items():
    print("\n" + "=" * 80)
    print(selector_name, len(selected_idx))
    print("=" * 80)

    fold_probs = []
    fold_y = []

    for fold_id, (idx_train, idx_val, idx_test) in enumerate(splits, start=1):
        print(f"{selector_name} fold {fold_id}")

        model = get_tuned_catboost()

        model.fit(
            X_final[idx_train][:, selected_idx],
            y_final[idx_train],
            eval_set=(X_final[idx_val][:, selected_idx], y_final[idx_val]),
            use_best_model=True,
        )

        probs = model.predict_proba(X_final[idx_test][:, selected_idx])[:, 1]

        fold_probs.append(probs)
        fold_y.append(y_final[idx_test])

    aucs = [
        roc_auc_score(fold_y[i], fold_probs[i])
        for i in range(len(fold_probs))
    ]

    selector_predictions[selector_name] = {
        "probs": fold_probs,
        "y": fold_y,
        "aucs": aucs,
    }

    print(f"AUROC={np.mean(aucs):.4f} ± {np.std(aucs):.4f}")


rfe_250_200_125_70 70
rfe_250_200_125_70 fold 1
rfe_250_200_125_70 fold 2
rfe_250_200_125_70 fold 3
rfe_250_200_125_70 fold 4
rfe_250_200_125_70 fold 5
AUROC=0.8944 ± 0.0102

rfe_250_200_125_65 65
rfe_250_200_125_65 fold 1
rfe_250_200_125_65 fold 2
rfe_250_200_125_65 fold 3
rfe_250_200_125_65 fold 4
rfe_250_200_125_65 fold 5
AUROC=0.8940 ± 0.0139

rfe_250_200_125_75 75
rfe_250_200_125_75 fold 1
rfe_250_200_125_75 fold 2
rfe_250_200_125_75 fold 3
rfe_250_200_125_75 fold 4
rfe_250_200_125_75 fold 5
AUROC=0.8875 ± 0.0189

stability_150 150
stability_150 fold 1
stability_150 fold 2
stability_150 fold 3
stability_150 fold 4
stability_150 fold 5
AUROC=0.8533 ± 0.0345

stability_100 100
stability_100 fold 1
stability_100 fold 2
stability_100 fold 3
stability_100 fold 4
stability_100 fold 5
AUROC=0.8678 ± 0.0263

stability_200 200
stability_200 fold 1
stability_200 fold 2
stability_200 fold 3
stability_200 fold 4
stability_200 fold 5
AUROC=0.8684 ± 0.0193


In [183]:
single_selector_df = pd.DataFrame([
    {
        "selector": name,
        "test_auroc_mean": np.mean(pack["aucs"]),
        "test_auroc_std": np.std(pack["aucs"]),
    }
    for name, pack in selector_predictions.items()
]).sort_values("test_auroc_mean", ascending=False)

display(single_selector_df)

,selector,test_auroc_mean,test_auroc_std
0,rfe_250_200_125_70,0.894408,0.010170
1,rfe_250_200_125_65,0.893964,0.013887
2,rfe_250_200_125_75,0.887469,0.018947
5,stability_200,0.868444,0.019326
4,stability_100,0.867754,0.026326
3,stability_150,0.853251,0.034537


In [184]:
def safe_logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return logit(p)


def evaluate_selector_ensemble(selector_names, weights=None, mode="logit"):
    if weights is None:
        weights = np.ones(len(selector_names), dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)

    weights = weights / weights.sum()

    fold_aucs = []

    for fold_i in range(len(splits)):
        y_test = selector_predictions[selector_names[0]]["y"][fold_i]

        preds = [
            selector_predictions[name]["probs"][fold_i]
            for name in selector_names
        ]

        if mode == "proba":
            ens = np.zeros_like(preds[0], dtype=float)

            for w, p in zip(weights, preds):
                ens += w * p

        elif mode == "logit":
            ens_logit = np.zeros_like(preds[0], dtype=float)

            for w, p in zip(weights, preds):
                ens_logit += w * safe_logit(p)

            ens = expit(ens_logit)

        else:
            raise ValueError(mode)

        fold_aucs.append(roc_auc_score(y_test, ens))

    return np.mean(fold_aucs), np.std(fold_aucs)

In [185]:
from itertools import combinations

ensemble_rows = []

selector_names = list(selector_predictions.keys())

for r in [2, 3, 4, 5, 6]:
    for combo in combinations(selector_names, r):
        for mode in ["proba", "logit"]:
            mean_auc, std_auc = evaluate_selector_ensemble(
                selector_names=list(combo),
                weights=None,
                mode=mode,
            )

            ensemble_rows.append({
                "selectors": " + ".join(combo),
                "n_selectors": r,
                "mode": mode,
                "weights": "equal",
                "test_auroc_mean": mean_auc,
                "test_auroc_std": std_auc,
            })

ensemble_selectors_df = pd.DataFrame(ensemble_rows).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(ensemble_selectors_df.head(30))

,selectors,n_selectors,mode,weights,test_auroc_mean,test_auroc_std
0,rfe_250_200_125_70 + rfe_250_200_125_65,2,proba,equal,0.894935,0.011613
1,rfe_250_200_125_70 + rfe_250_200_125_65,2,logit,equal,0.894884,0.011724
30,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,3,proba,equal,0.893974,0.013957
31,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,3,logit,equal,0.893874,0.013949
11,rfe_250_200_125_65 + rfe_250_200_125_75,2,logit,equal,0.893159,0.015064
10,rfe_250_200_125_65 + rfe_250_200_125_75,2,proba,equal,0.893108,0.015268
3,rfe_250_200_125_70 + rfe_250_200_125_75,2,logit,equal,0.892520,0.014519
2,rfe_250_200_125_70 + rfe_250_200_125_75,2,proba,equal,0.892216,0.014923
73,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,4,logit,equal,0.892012,0.014130
72,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,4,proba,equal,0.891960,0.014735


In [186]:
top3 = single_selector_df["selector"].head(3).tolist()

weight_grid = [
    (0.6, 0.3, 0.1),
    (0.6, 0.2, 0.2),
    (0.5, 0.4, 0.1),
    (0.5, 0.3, 0.2),
    (0.4, 0.4, 0.2),
    (0.7, 0.2, 0.1),
    (0.8, 0.1, 0.1),
]

weighted_rows = []

for weights in weight_grid:
    for mode in ["proba", "logit"]:
        mean_auc, std_auc = evaluate_selector_ensemble(
            selector_names=top3,
            weights=weights,
            mode=mode,
        )

        weighted_rows.append({
            "selectors": " + ".join(top3),
            "mode": mode,
            "weights": weights,
            "test_auroc_mean": mean_auc,
            "test_auroc_std": std_auc,
        })

weighted_selector_ensemble_df = pd.DataFrame(weighted_rows).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(weighted_selector_ensemble_df)

,selectors,mode,weights,test_auroc_mean,test_auroc_std
12,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,proba,"(0.8, 0.1, 0.1)",0.895399,0.011792
13,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,logit,"(0.8, 0.1, 0.1)",0.895148,0.011667
11,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,logit,"(0.7, 0.2, 0.1)",0.895045,0.011705
5,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,logit,"(0.5, 0.4, 0.1)",0.895038,0.012158
1,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,logit,"(0.6, 0.3, 0.1)",0.894993,0.011644
10,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,proba,"(0.7, 0.2, 0.1)",0.894944,0.011583
9,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,logit,"(0.4, 0.4, 0.2)",0.894931,0.013284
0,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,proba,"(0.6, 0.3, 0.1)",0.894893,0.011558
4,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,proba,"(0.5, 0.4, 0.1)",0.894888,0.012139
3,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,logit,"(0.6, 0.2, 0.2)",0.894842,0.012359


In [187]:
known_best = pd.DataFrame([
    {
        "source": "known_best_rfe",
        "selectors": "rfe_250_200_125_70",
        "mode": "single",
        "weights": None,
        "test_auroc_mean": 0.8944,
        "test_auroc_std": 0.0102,
    }
])

single_df = single_selector_df.rename(columns={"selector": "selectors"})
single_df["source"] = "single_selector"
single_df["mode"] = "single"
single_df["weights"] = None
single_df = single_df[
    ["source", "selectors", "mode", "weights", "test_auroc_mean", "test_auroc_std"]
]

ens_df = ensemble_selectors_df.copy()
ens_df["source"] = "equal_selector_ensemble"
ens_df = ens_df[
    ["source", "selectors", "mode", "weights", "test_auroc_mean", "test_auroc_std"]
]

weighted_df = weighted_selector_ensemble_df.copy()
weighted_df["source"] = "weighted_selector_ensemble"
weighted_df = weighted_df[
    ["source", "selectors", "mode", "weights", "test_auroc_mean", "test_auroc_std"]
]

final_selector_compare = pd.concat(
    [known_best, single_df, ens_df, weighted_df],
    ignore_index=True,
).sort_values("test_auroc_mean", ascending=False)

display(final_selector_compare.head(30))

best = final_selector_compare.iloc[0]

print(
    f"BEST DIVERSITY ENSEMBLE:\n"
    f"{best['source']}\n"
    f"{best['selectors']}\n"
    f"mode={best['mode']} weights={best['weights']}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}"
)

,source,selectors,mode,weights,test_auroc_mean,test_auroc_std
121,weighted_selector_ensemble,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,proba,"(0.8, 0.1, 0.1)",0.895399,0.011792
122,weighted_selector_ensemble,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,logit,"(0.8, 0.1, 0.1)",0.895148,0.011667
123,weighted_selector_ensemble,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,logit,"(0.7, 0.2, 0.1)",0.895045,0.011705
124,weighted_selector_ensemble,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,logit,"(0.5, 0.4, 0.1)",0.895038,0.012158
125,weighted_selector_ensemble,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,logit,"(0.6, 0.3, 0.1)",0.894993,0.011644
126,weighted_selector_ensemble,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,proba,"(0.7, 0.2, 0.1)",0.894944,0.011583
7,equal_selector_ensemble,rfe_250_200_125_70 + rfe_250_200_125_65,proba,equal,0.894935,0.011613
127,weighted_selector_ensemble,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,logit,"(0.4, 0.4, 0.2)",0.894931,0.013284
128,weighted_selector_ensemble,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,proba,"(0.6, 0.3, 0.1)",0.894893,0.011558
129,weighted_selector_ensemble,rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_...,proba,"(0.5, 0.4, 0.1)",0.894888,0.012139


BEST DIVERSITY ENSEMBLE:
weighted_selector_ensemble
rfe_250_200_125_70 + rfe_250_200_125_65 + rfe_250_200_125_75
mode=proba weights=(0.8, 0.1, 0.1)
AUROC=0.8954 ± 0.0118


## Again CatBoost tuning for diversity ensemble

In [188]:
import optuna
import numpy as np
import pandas as pd

from scipy.special import logit, expit
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier


def safe_logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return logit(p)


def build_rfe_indices_for_steps(steps):
    start_idx = counts_base.head(250).index.to_numpy(dtype=int)

    selected_idx = recursive_elimination_return_idx(
        X=X_final,
        y=y_final,
        splits=splits,
        initial_idx=start_idx,
        steps=steps,
    )

    return selected_idx


rfe_indices = {
    "rfe_70": build_rfe_indices_for_steps((200, 125, 70)),
    "rfe_65": build_rfe_indices_for_steps((200, 125, 65)),
    "rfe_75": build_rfe_indices_for_steps((200, 125, 75)),
}

print({k: len(v) for k, v in rfe_indices.items()})

RFE target_k=200, current=250
RFE target_k=125, current=200
RFE target_k=70, current=125
RFE target_k=200, current=250
RFE target_k=125, current=200
RFE target_k=65, current=125
RFE target_k=200, current=250
RFE target_k=125, current=200
RFE target_k=75, current=125
{'rfe_70': 70, 'rfe_65': 65, 'rfe_75': 75}


In [189]:
def make_catboost_from_params(params):
    return CatBoostClassifier(
        **params,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )

In [190]:
def evaluate_rfe_ensemble_with_params(
    params,
    X,
    y,
    splits,
    rfe_indices,
    weights=(0.8, 0.1, 0.1),
    mode="proba",
):
    selector_order = ["rfe_70", "rfe_65", "rfe_75"]
    weights = np.asarray(weights, dtype=np.float64)
    weights = weights / weights.sum()

    fold_aucs = []

    for idx_train, idx_val, idx_test in splits:
        fold_preds = []

        for selector_name in selector_order:
            selected_idx = rfe_indices[selector_name]

            model = make_catboost_from_params(params)

            model.fit(
                X[idx_train][:, selected_idx],
                y[idx_train],
                eval_set=(X[idx_val][:, selected_idx], y[idx_val]),
                use_best_model=True,
            )

            probs = model.predict_proba(
                X[idx_test][:, selected_idx]
            )[:, 1]

            fold_preds.append(probs)

        if mode == "proba":
            ensemble_probs = np.zeros_like(fold_preds[0], dtype=np.float64)

            for w, p in zip(weights, fold_preds):
                ensemble_probs += w * p

        elif mode == "logit":
            ensemble_logit = np.zeros_like(fold_preds[0], dtype=np.float64)

            for w, p in zip(weights, fold_preds):
                ensemble_logit += w * safe_logit(p)

            ensemble_probs = expit(ensemble_logit)

        else:
            raise ValueError(mode)

        fold_aucs.append(
            roc_auc_score(y[idx_test], ensemble_probs)
        )

    return float(np.mean(fold_aucs)), float(np.std(fold_aucs))

In [191]:
def objective_catboost_rfe_ensemble(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 200, 900, step=100),
        "depth": trial.suggest_int("depth", 2, 5),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.08, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 50.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.1, 20.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
        "border_count": trial.suggest_int("border_count", 32, 254),
        "auto_class_weights": trial.suggest_categorical(
            "auto_class_weights",
            ["Balanced", "SqrtBalanced", None],
        ),
    }

    mean_auc, std_auc = evaluate_rfe_ensemble_with_params(
        params=params,
        X=X_final,
        y=y_final,
        splits=splits,
        rfe_indices=rfe_indices,
        weights=(0.8, 0.1, 0.1),
        mode="proba",
    )

    # чуть штрафуем нестабильность
    return mean_auc - 0.15 * std_auc

In [192]:
study_rfe_ensemble = optuna.create_study(direction="maximize")

study_rfe_ensemble.optimize(
    objective_catboost_rfe_ensemble,
    n_trials=30,
)

print("Best objective:", study_rfe_ensemble.best_value)
print("Best params:")
print(study_rfe_ensemble.best_params)

[I 2026-05-12 15:42:58,844] A new study created in memory with name: no-name-a50ebae5-bc0b-47f0-abdb-6ebd562fc8cd
[I 2026-05-12 15:43:12,531] Trial 0 finished with value: 0.8896820129985806 and parameters: {'iterations': 700, 'depth': 4, 'learning_rate': 0.017208170185951763, 'l2_leaf_reg': 17.59689519375979, 'random_strength': 2.939393786966325, 'bagging_temperature': 1.5524286010303108, 'border_count': 99, 'auto_class_weights': 'SqrtBalanced'}. Best is trial 0 with value: 0.8896820129985806.
[I 2026-05-12 15:43:21,332] Trial 1 finished with value: 0.8573508007186725 and parameters: {'iterations': 300, 'depth': 5, 'learning_rate': 0.007747161053530439, 'l2_leaf_reg': 3.0554384900439797, 'random_strength': 0.20109018358604253, 'bagging_temperature': 2.2276298581072744, 'border_count': 129, 'auto_class_weights': None}. Best is trial 0 with value: 0.8896820129985806.
[I 2026-05-12 15:43:26,632] Trial 2 finished with value: 0.8530946373206377 and parameters: {'iterations': 200, 'depth': 2

Best objective: 0.8958909655012314
Best params:
{'iterations': 400, 'depth': 4, 'learning_rate': 0.05154666190467915, 'l2_leaf_reg': 4.668730683946981, 'random_strength': 7.525362919817402, 'bagging_temperature': 1.0442924028703902, 'border_count': 124, 'auto_class_weights': 'SqrtBalanced'}


In [193]:
best_params_rfe_ensemble = study_rfe_ensemble.best_params

best_mean_auc, best_std_auc = evaluate_rfe_ensemble_with_params(
    params=best_params_rfe_ensemble,
    X=X_final,
    y=y_final,
    splits=splits,
    rfe_indices=rfe_indices,
    weights=(0.8, 0.1, 0.1),
    mode="proba",
)

comparison = pd.DataFrame([
    {
        "model": "previous_best_diversity_ensemble",
        "test_auroc_mean": 0.8954,
        "test_auroc_std": 0.0118,
        "params": "old tuned CatBoost",
    },
    {
        "model": "optuna_tuned_rfe_ensemble",
        "test_auroc_mean": best_mean_auc,
        "test_auroc_std": best_std_auc,
        "params": best_params_rfe_ensemble,
    },
]).sort_values("test_auroc_mean", ascending=False)

display(comparison)

print(
    f"BEST OPTUNA RFE ENSEMBLE:\n"
    f"AUROC={best_mean_auc:.4f} ± {best_std_auc:.4f}"
)

,model,test_auroc_mean,test_auroc_std,params
1,optuna_tuned_rfe_ensemble,0.898245,0.015695,"{'iterations': 400, 'depth': 4, 'learning_rate..."
0,previous_best_diversity_ensemble,0.895400,0.011800,old tuned CatBoost


BEST OPTUNA RFE ENSEMBLE:
AUROC=0.8982 ± 0.0157


## Weights tuning

In [195]:
best_params_rfe_ensemble = {
    "iterations": 400,
    "depth": 4,
    "learning_rate": 0.05154666190467915,
    "l2_leaf_reg": 4.668730683946981,
    "random_strength": 7.525362919817402,
    "bagging_temperature": 1.0442924028703902,
    "border_count": 124,
    "auto_class_weights": "SqrtBalanced",
}

weight_grid = [
    (0.95, 0.025, 0.025),
    (0.90, 0.05, 0.05),
    (0.85, 0.10, 0.05),
    (0.85, 0.05, 0.10),
    (0.80, 0.10, 0.10),
    (0.75, 0.15, 0.10),
    (0.75, 0.10, 0.15),
    (0.70, 0.20, 0.10),
    (0.70, 0.15, 0.15),
    (0.65, 0.25, 0.10),
    (0.65, 0.20, 0.15),
]

weight_rows = []

for weights in weight_grid:
    for mode in ["proba", "logit"]:
        mean_auc, std_auc = evaluate_rfe_ensemble_with_params(
            params=best_params_rfe_ensemble,
            X=X_final,
            y=y_final,
            splits=splits,
            rfe_indices=rfe_indices,
            weights=weights,
            mode=mode,
        )

        weight_rows.append({
            "weights": weights,
            "mode": mode,
            "test_auroc_mean": mean_auc,
            "test_auroc_std": std_auc,
        })

weight_tuning_df = pd.DataFrame(weight_rows).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(weight_tuning_df)

best = weight_tuning_df.iloc[0]

print(
    f"BEST WEIGHTS:\n"
    f"weights={best['weights']}\n"
    f"mode={best['mode']}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}"
)

,weights,mode,test_auroc_mean,test_auroc_std
4,"(0.85, 0.1, 0.05)",proba,0.898295,0.016104
8,"(0.8, 0.1, 0.1)",proba,0.898245,0.015695
20,"(0.65, 0.2, 0.15)",proba,0.898237,0.015448
9,"(0.8, 0.1, 0.1)",logit,0.898098,0.014872
7,"(0.85, 0.05, 0.1)",logit,0.897999,0.015517
2,"(0.9, 0.05, 0.05)",proba,0.897942,0.016449
19,"(0.65, 0.25, 0.1)",logit,0.897900,0.014140
10,"(0.75, 0.15, 0.1)",proba,0.897892,0.015388
16,"(0.7, 0.15, 0.15)",proba,0.897887,0.015683
5,"(0.85, 0.1, 0.05)",logit,0.897798,0.015454


BEST WEIGHTS:
weights=(0.85, 0.1, 0.05)
mode=proba
AUROC=0.8983 ± 0.0161


## OOF-stacking

In [196]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.special import logit, expit

In [197]:
best_params_rfe_ensemble = {
    "iterations": 400,
    "depth": 4,
    "learning_rate": 0.05154666190467915,
    "l2_leaf_reg": 4.668730683946981,
    "random_strength": 7.525362919817402,
    "bagging_temperature": 1.0442924028703902,
    "border_count": 124,
    "auto_class_weights": "SqrtBalanced",
}

In [198]:
def safe_logit_np(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return logit(p)


def get_catboost_probs(params, X_train, y_train, X_val, y_val, X_pred):
    model = make_catboost_from_params(params)

    model.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        use_best_model=True,
    )

    return model.predict_proba(X_pred)[:, 1]


def generate_outer_fold_meta_features(
    X,
    y,
    splits,
    rfe_indices,
    params,
):
    rows = []

    selector_order = ["rfe_70", "rfe_65", "rfe_75"]

    for fold_id, (idx_train, idx_val, idx_test) in enumerate(splits, start=1):
        print(f"\nOuter fold {fold_id}")

        Z_val = []
        Z_test = []

        for selector_name in selector_order:
            selected_idx = rfe_indices[selector_name]

            print(f"  selector={selector_name}, n_features={len(selected_idx)}")

            val_probs = get_catboost_probs(
                params=params,
                X_train=X[idx_train][:, selected_idx],
                y_train=y[idx_train],
                X_val=X[idx_val][:, selected_idx],
                y_val=y[idx_val],
                X_pred=X[idx_val][:, selected_idx],
            )

            test_probs = get_catboost_probs(
                params=params,
                X_train=X[idx_train][:, selected_idx],
                y_train=y[idx_train],
                X_val=X[idx_val][:, selected_idx],
                y_val=y[idx_val],
                X_pred=X[idx_test][:, selected_idx],
            )

            Z_val.append(val_probs)
            Z_test.append(test_probs)

        Z_val = np.column_stack(Z_val)
        Z_test = np.column_stack(Z_test)

        rows.append({
            "fold": fold_id,
            "Z_val_proba": Z_val,
            "Z_test_proba": Z_test,
            "Z_val_logit": safe_logit_np(Z_val),
            "Z_test_logit": safe_logit_np(Z_test),
            "y_val": y[idx_val],
            "y_test": y[idx_test],
        })

    return rows


stack_data = generate_outer_fold_meta_features(
    X=X_final,
    y=y_final,
    splits=splits,
    rfe_indices=rfe_indices,
    params=best_params_rfe_ensemble,
)


Outer fold 1
  selector=rfe_70, n_features=70
  selector=rfe_65, n_features=65
  selector=rfe_75, n_features=75

Outer fold 2
  selector=rfe_70, n_features=70
  selector=rfe_65, n_features=65
  selector=rfe_75, n_features=75

Outer fold 3
  selector=rfe_70, n_features=70
  selector=rfe_65, n_features=65
  selector=rfe_75, n_features=75

Outer fold 4
  selector=rfe_70, n_features=70
  selector=rfe_65, n_features=65
  selector=rfe_75, n_features=75

Outer fold 5
  selector=rfe_70, n_features=70
  selector=rfe_65, n_features=65
  selector=rfe_75, n_features=75


In [199]:
def evaluate_manual_weights_from_stack_data(
    stack_data,
    weights=(0.85, 0.10, 0.05),
    mode="proba",
):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()

    aucs = []

    for pack in stack_data:
        y_test = pack["y_test"]

        if mode == "proba":
            Z = pack["Z_test_proba"]
            probs = Z @ weights

        elif mode == "logit":
            Z = pack["Z_test_logit"]
            probs = expit(Z @ weights)

        else:
            raise ValueError(mode)

        aucs.append(roc_auc_score(y_test, probs))

    return np.mean(aucs), np.std(aucs)


manual_mean, manual_std = evaluate_manual_weights_from_stack_data(
    stack_data,
    weights=(0.85, 0.10, 0.05),
    mode="proba",
)

print(f"Manual weighted ensemble: {manual_mean:.4f} ± {manual_std:.4f}")

Manual weighted ensemble: 0.8983 ± 0.0161


In [200]:
def evaluate_oof_stacking(
    stack_data,
    meta_model_factory,
    input_mode="proba",
):
    aucs = []
    f1s = []
    accs = []

    for pack in stack_data:
        if input_mode == "proba":
            Z_train = pack["Z_val_proba"]
            Z_test = pack["Z_test_proba"]
        elif input_mode == "logit":
            Z_train = pack["Z_val_logit"]
            Z_test = pack["Z_test_logit"]
        else:
            raise ValueError(input_mode)

        y_train_meta = pack["y_val"]
        y_test = pack["y_test"]

        meta = meta_model_factory()
        meta.fit(Z_train, y_train_meta)

        if hasattr(meta, "predict_proba"):
            probs = meta.predict_proba(Z_test)[:, 1]
        else:
            scores = meta.decision_function(Z_test)
            probs = expit(scores)

        preds = (probs >= 0.5).astype(int)

        aucs.append(roc_auc_score(y_test, probs))
        f1s.append(f1_score(y_test, preds, zero_division=0))
        accs.append(accuracy_score(y_test, preds))

    return {
        "test_auroc_mean": float(np.mean(aucs)),
        "test_auroc_std": float(np.std(aucs)),
        "test_f1_mean": float(np.mean(f1s)),
        "test_accuracy_mean": float(np.mean(accs)),
    }

In [201]:
stacking_rows = []

meta_configs = {
    "logreg_C0.01": lambda: LogisticRegression(
        C=0.01,
        class_weight="balanced",
        max_iter=5000,
        solver="lbfgs",
    ),
    "logreg_C0.03": lambda: LogisticRegression(
        C=0.03,
        class_weight="balanced",
        max_iter=5000,
        solver="lbfgs",
    ),
    "logreg_C0.1": lambda: LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=5000,
        solver="lbfgs",
    ),
    "logreg_C0.3": lambda: LogisticRegression(
        C=0.3,
        class_weight="balanced",
        max_iter=5000,
        solver="lbfgs",
    ),
    "logreg_C1.0": lambda: LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=5000,
        solver="lbfgs",
    ),
    "scaled_logreg_C0.1": lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=0.1,
            class_weight="balanced",
            max_iter=5000,
            solver="lbfgs",
        )),
    ]),
}

for meta_name, factory in meta_configs.items():
    for input_mode in ["proba", "logit"]:
        result = evaluate_oof_stacking(
            stack_data=stack_data,
            meta_model_factory=factory,
            input_mode=input_mode,
        )

        stacking_rows.append({
            "method": "oof_stacking",
            "meta_model": meta_name,
            "input_mode": input_mode,
            **result,
        })

stacking_df = pd.DataFrame(stacking_rows).sort_values(
    "test_auroc_mean",
    ascending=False,
)

display(stacking_df)

,method,meta_model,input_mode,test_auroc_mean,test_auroc_std,test_f1_mean,test_accuracy_mean
11,oof_stacking,scaled_logreg_C0.1,logit,0.898051,0.012806,0.848621,0.802571
5,oof_stacking,logreg_C0.1,logit,0.897850,0.011986,0.852471,0.806929
0,oof_stacking,logreg_C0.01,proba,0.897687,0.014088,0.876837,0.831651
2,oof_stacking,logreg_C0.03,proba,0.897687,0.014088,0.877759,0.833101
10,oof_stacking,scaled_logreg_C0.1,proba,0.897640,0.013769,0.874723,0.830213
8,oof_stacking,logreg_C1.0,proba,0.897638,0.013890,0.878703,0.834560
6,oof_stacking,logreg_C0.3,proba,0.897536,0.014155,0.878703,0.834560
4,oof_stacking,logreg_C0.1,proba,0.897535,0.014279,0.878703,0.834560
1,oof_stacking,logreg_C0.01,logit,0.897492,0.012867,0.848722,0.804020
3,oof_stacking,logreg_C0.03,logit,0.897491,0.013000,0.851327,0.806918


In [202]:
compare_stacking = pd.concat([
    pd.DataFrame([
        {
            "method": "previous_best_manual_weighted",
            "meta_model": "manual",
            "input_mode": "proba",
            "test_auroc_mean": 0.8983,
            "test_auroc_std": 0.0161,
            "test_f1_mean": None,
            "test_accuracy_mean": None,
        },
        {
            "method": "manual_from_stack_data",
            "meta_model": "manual_0.85_0.10_0.05",
            "input_mode": "proba",
            "test_auroc_mean": manual_mean,
            "test_auroc_std": manual_std,
            "test_f1_mean": None,
            "test_accuracy_mean": None,
        },
    ]),
    stacking_df,
], ignore_index=True).sort_values("test_auroc_mean", ascending=False)

display(compare_stacking)

best = compare_stacking.iloc[0]

print(
    f"BEST STACKING RESULT:\n"
    f"{best['method']}\n"
    f"meta={best['meta_model']}, input={best['input_mode']}\n"
    f"AUROC={best['test_auroc_mean']:.4f} ± {best['test_auroc_std']:.4f}"
)

,method,meta_model,input_mode,test_auroc_mean,test_auroc_std,test_f1_mean,test_accuracy_mean
0,previous_best_manual_weighted,manual,proba,0.898300,0.016100,None,None
1,manual_from_stack_data,manual_0.85_0.10_0.05,proba,0.898295,0.016104,None,None
2,oof_stacking,scaled_logreg_C0.1,logit,0.898051,0.012806,0.848621,0.802571
3,oof_stacking,logreg_C0.1,logit,0.897850,0.011986,0.852471,0.806929
4,oof_stacking,logreg_C0.01,proba,0.897687,0.014088,0.876837,0.831651
5,oof_stacking,logreg_C0.03,proba,0.897687,0.014088,0.877759,0.833101
6,oof_stacking,scaled_logreg_C0.1,proba,0.897640,0.013769,0.874723,0.830213
7,oof_stacking,logreg_C1.0,proba,0.897638,0.013890,0.878703,0.83456
8,oof_stacking,logreg_C0.3,proba,0.897536,0.014155,0.878703,0.83456
9,oof_stacking,logreg_C0.1,proba,0.897535,0.014279,0.878703,0.83456


BEST STACKING RESULT:
previous_best_manual_weighted
meta=manual, input=proba
AUROC=0.8983 ± 0.0161


## Adversarial validation

In [203]:
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

In [204]:
def make_adv_catboost():
    return CatBoostClassifier(
        iterations=300,
        depth=4,
        learning_rate=0.05,
        loss_function="Logloss",
        eval_metric="AUC",
        verbose=False,
        random_seed=42,
    )

In [ ]:
def run_adversarial_validation_fast(
    X,
    splits,
):
    rows = []

    for fold_id, (idx_train, idx_val, idx_test) in enumerate(splits, start=1):

        train_idx = np.concatenate([idx_train, idx_val])
        test_idx = idx_test

        X_train_adv = np.vstack([
            X[train_idx],
            X[test_idx],
        ])

        y_train_adv = np.concatenate([
            np.zeros(len(train_idx)),
            np.ones(len(test_idx)),
        ])

        perm = np.random.permutation(len(X_train_adv))

        split = int(len(perm) * 0.8)

        tr_idx = perm[:split]
        va_idx = perm[split:]

        model = CatBoostClassifier(
            iterations=150,
            depth=4,
            learning_rate=0.05,
            loss_function="Logloss",
            eval_metric="AUC",
            verbose=False,
            random_seed=42,
        )

        model.fit(
            X_train_adv[tr_idx],
            y_train_adv[tr_idx],
            eval_set=(
                X_train_adv[va_idx],
                y_train_adv[va_idx],
            ),
            use_best_model=True,
        )

        probs = model.predict_proba(
            X_train_adv[va_idx]
        )[:, 1]

        auc = roc_auc_score(
            y_train_adv[va_idx],
            probs,
        )

        print(f"fold={fold_id} adv_auc={auc:.4f}")

        rows.append({
            "outer_fold": fold_id,
            "adv_auc": auc,
        })

    return pd.DataFrame(rows)

In [ ]:
adv_df = run_adversarial_validation_fast(
    X=X_final,
    splits=splits,
)

display(adv_df)

print(
    f"\nADV AUC = "
    f"{adv_df['adv_auc'].mean():.4f} ± "
    f"{adv_df['adv_auc'].std():.4f}"
)

Adversarial outer folds:   0%|          | 0/5 [00:00<?, ?it/s]


Start adversarial fold 1/5
  X_adv shape: (689, 89600)
  fitting CatBoost...
0:	test: 0.5489901	best: 0.5489901 (0)	total: 2.25s	remaining: 2m 57s
20:	test: 0.4178475	best: 0.5489901 (0)	total: 1m 2s	remaining: 2m 55s
40:	test: 0.5342177	best: 0.5607477 (37)	total: 2m 15s	remaining: 2m 9s
60:	test: 0.5571299	best: 0.5607477 (37)	total: 3m 16s	remaining: 1m 1s
79:	test: 0.5902924	best: 0.5902924 (79)	total: 4m 10s	remaining: 0us

bestTest = 0.5902924329
bestIteration = 79

  predicting...
  fold 1 adv_auc=0.5903

Start adversarial fold 2/5
  X_adv shape: (689, 89600)
  fitting CatBoost...
0:	test: 0.4641804	best: 0.4641804 (0)	total: 2.32s	remaining: 3m 3s
20:	test: 0.5073703	best: 0.5807783 (6)	total: 1m 2s	remaining: 2m 55s
40:	test: 0.5919811	best: 0.6005307 (39)	total: 1m 59s	remaining: 1m 53s
60:	test: 0.5846108	best: 0.6028892 (41)	total: 3m 5s	remaining: 57.9s
79:	test: 0.5816627	best: 0.6028892 (41)	total: 3m 59s	remaining: 0us

bestTest = 0.6028891509
bestIteration = 41

Shrin

,outer_fold,adv_auc
0,1,0.590292
1,2,0.602889
2,3,0.638213
3,4,0.644891
4,5,0.545322



ADV AUC = 0.6043 ± 0.0402


In [218]:
from tqdm.auto import tqdm
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

DROP_COLS = {"label", "prompt", "response", "source_index", "id"}

df_features = pd.read_parquet(
    "./artifacts/drift_extended_features/features_dataset_drift_extended_all.parquet"
)

feature_names_final = [c for c in df_features.columns if c not in DROP_COLS]

print("feature_names:", len(feature_names_final))
print("X_final cols:", X_final.shape[1])

assert len(feature_names_final) == X_final.shape[1]


def adversarial_feature_importance_fast(X, splits, feature_names, iterations=50):
    importances = []

    for fold_id, (idx_train, idx_val, idx_test) in enumerate(
        tqdm(splits, desc="Adversarial importance folds"),
        start=1,
    ):
        print(f"\nFold {fold_id}/{len(splits)}")

        train_idx = np.concatenate([idx_train, idx_val])
        test_idx = idx_test

        X_adv = np.vstack([X[train_idx], X[test_idx]])
        y_adv = np.concatenate([
            np.zeros(len(train_idx)),
            np.ones(len(test_idx)),
        ])

        model = CatBoostClassifier(
            iterations=iterations,
            depth=3,
            learning_rate=0.08,
            loss_function="Logloss",
            eval_metric="AUC",
            verbose=20,
            random_seed=42 + fold_id,
            thread_count=-1,
            allow_writing_files=False,
        )

        model.fit(X_adv, y_adv)
        importances.append(model.get_feature_importance())

    importances = np.asarray(importances)

    return pd.DataFrame({
        "feature": feature_names,
        "adv_importance_mean": importances.mean(axis=0),
        "adv_importance_std": importances.std(axis=0),
    }).sort_values("adv_importance_mean", ascending=False)

feature_names: 89600
X_final cols: 89600


In [219]:
adv_importance_df = adversarial_feature_importance_fast(
    X=X_final,
    splits=splits,
    feature_names=feature_names_final,
    iterations=50,
)

display(adv_importance_df.head(30))

Adversarial importance folds:   0%|          | 0/5 [00:00<?, ?it/s]


Fold 1/5
0:	total: 2.1s	remaining: 1m 42s
20:	total: 46.2s	remaining: 1m 3s
40:	total: 1m 28s	remaining: 19.5s
49:	total: 1m 53s	remaining: 0us

Fold 2/5
0:	total: 2.13s	remaining: 1m 44s
20:	total: 48.1s	remaining: 1m 6s
40:	total: 1m 38s	remaining: 21.7s
49:	total: 1m 59s	remaining: 0us

Fold 3/5
0:	total: 2.07s	remaining: 1m 41s
20:	total: 47.5s	remaining: 1m 5s
40:	total: 1m 36s	remaining: 21.2s
49:	total: 1m 58s	remaining: 0us

Fold 4/5
0:	total: 2.11s	remaining: 1m 43s
20:	total: 50.8s	remaining: 1m 10s
40:	total: 1m 39s	remaining: 21.7s
49:	total: 2m 2s	remaining: 0us

Fold 5/5
0:	total: 2.34s	remaining: 1m 54s
20:	total: 56.7s	remaining: 1m 18s
40:	total: 1m 48s	remaining: 23.7s
49:	total: 2m 12s	remaining: 0us


,feature,adv_importance_mean,adv_importance_std
2381,drift_squared_l11_to_l12_d589,0.935057,1.870114
79624,response_last_drift_signed_l12_to_l13_d776,0.589348,1.178697
69761,response_middle_drift_abs_l13_to_l14_d769,0.527936,0.765475
9695,drift_signed_l13_to_l14_d735,0.520264,1.040528
86906,response_last_drift_normed_l14_to_l15_d890,0.509681,1.019362
34551,long_drift_abs_l14_to_l16_d503,0.503262,1.006523
60124,response_first_drift_signed_l15_to_l16_d92,0.501877,1.003753
48364,long_drift_abs_l11_to_l16_d876,0.488913,0.977825
2084,drift_squared_l11_to_l12_d292,0.480521,0.961042
61900,response_first_drift_normed_l15_to_l16_d76,0.468460,0.936920


In [220]:
adv_pruning_results = []

remove_top_list = [5, 10, 20]

for top_k_remove in remove_top_list:
    print(f"\nRemove top {top_k_remove} adversarial features")

    unstable_features = set(
        adv_importance_df.head(top_k_remove)["feature"].tolist()
    )

    keep_idx = np.array([
        i for i, name in enumerate(feature_names_final)
        if name not in unstable_features
    ])

    X_pruned = X_final[:, keep_idx]

    counts_pruned, _ = build_base_stable_candidates(
        X=X_pruned,
        y=y_final,
        splits=splits,
        base_k=3500,
        per_fold_k=350,
    )

    rfe_indices_pruned = {
        "rfe_70": recursive_elimination_return_idx(
            X=X_pruned,
            y=y_final,
            splits=splits,
            initial_idx=counts_pruned.head(250).index.to_numpy(dtype=int),
            steps=(200, 125, 70),
        ),
        "rfe_65": recursive_elimination_return_idx(
            X=X_pruned,
            y=y_final,
            splits=splits,
            initial_idx=counts_pruned.head(250).index.to_numpy(dtype=int),
            steps=(200, 125, 65),
        ),
        "rfe_75": recursive_elimination_return_idx(
            X=X_pruned,
            y=y_final,
            splits=splits,
            initial_idx=counts_pruned.head(250).index.to_numpy(dtype=int),
            steps=(200, 125, 75),
        ),
    }

    mean_auc, std_auc = evaluate_rfe_ensemble_with_params(
        params=best_params_rfe_ensemble,
        X=X_pruned,
        y=y_final,
        splits=splits,
        rfe_indices=rfe_indices_pruned,
        weights=(0.85, 0.10, 0.05),
        mode="proba",
    )

    adv_pruning_results.append({
        "experiment": f"remove_top_{top_k_remove}_adv_features",
        "remaining_features": X_pruned.shape[1],
        "test_auroc_mean": mean_auc,
        "test_auroc_std": std_auc,
    })

adv_pruning_df = pd.DataFrame(adv_pruning_results)

compare_adv = pd.concat([
    pd.DataFrame([{
        "experiment": "previous_best",
        "remaining_features": X_final.shape[1],
        "test_auroc_mean": 0.8983,
        "test_auroc_std": 0.0161,
    }]),
    adv_pruning_df,
], ignore_index=True).sort_values("test_auroc_mean", ascending=False)

display(compare_adv)


Remove top 5 adversarial features
base candidate fold 1
base candidate fold 2
base candidate fold 3
base candidate fold 4
base candidate fold 5
RFE target_k=200, current=250
RFE target_k=125, current=200
RFE target_k=70, current=125
RFE target_k=200, current=250
RFE target_k=125, current=200
RFE target_k=65, current=125
RFE target_k=200, current=250
RFE target_k=125, current=200
RFE target_k=75, current=125

Remove top 10 adversarial features
base candidate fold 1
base candidate fold 2
base candidate fold 3
base candidate fold 4
base candidate fold 5
RFE target_k=200, current=250
RFE target_k=125, current=200
RFE target_k=70, current=125
RFE target_k=200, current=250
RFE target_k=125, current=200
RFE target_k=65, current=125
RFE target_k=200, current=250
RFE target_k=125, current=200
RFE target_k=75, current=125

Remove top 20 adversarial features
base candidate fold 1
base candidate fold 2
base candidate fold 3
base candidate fold 4
base candidate fold 5
RFE target_k=200, current=250

,experiment,remaining_features,test_auroc_mean,test_auroc_std
0,previous_best,89600,0.898300,0.016100
2,remove_top_10_adv_features,89590,0.882755,0.013711
1,remove_top_5_adv_features,89595,0.876532,0.023636
3,remove_top_20_adv_features,89580,0.875916,0.022640


## Calibration-aware selection / calibration check

In [221]:
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from scipy.special import logit, expit
import numpy as np
import pandas as pd


def clip_probs(p):
    return np.clip(p, 1e-6, 1 - 1e-6)


def get_best_ensemble_fold_predictions(
    X,
    y,
    splits,
    rfe_indices,
    params,
    weights=(0.85, 0.10, 0.05),
):
    selector_order = ["rfe_70", "rfe_65", "rfe_75"]
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()

    rows = []

    for fold_id, (idx_train, idx_val, idx_test) in enumerate(splits, start=1):
        print(f"\nFold {fold_id}")

        val_preds = []
        test_preds = []

        for selector_name in selector_order:
            selected_idx = rfe_indices[selector_name]

            model = make_catboost_from_params(params)

            model.fit(
                X[idx_train][:, selected_idx],
                y[idx_train],
                eval_set=(X[idx_val][:, selected_idx], y[idx_val]),
                use_best_model=True,
            )

            val_probs = model.predict_proba(X[idx_val][:, selected_idx])[:, 1]
            test_probs = model.predict_proba(X[idx_test][:, selected_idx])[:, 1]

            val_preds.append(val_probs)
            test_preds.append(test_probs)

        val_preds = np.column_stack(val_preds)
        test_preds = np.column_stack(test_preds)

        val_ensemble = val_preds @ weights
        test_ensemble = test_preds @ weights

        rows.append({
            "fold": fold_id,
            "val_probs": clip_probs(val_ensemble),
            "test_probs": clip_probs(test_ensemble),
            "y_val": y[idx_val],
            "y_test": y[idx_test],
        })

    return rows


calib_data = get_best_ensemble_fold_predictions(
    X=X_final,
    y=y_final,
    splits=splits,
    rfe_indices=rfe_indices,
    params=best_params_rfe_ensemble,
    weights=(0.85, 0.10, 0.05),
)


Fold 1

Fold 2

Fold 3

Fold 4

Fold 5


In [222]:
def expected_calibration_error(y_true, probs, n_bins=10):
    y_true = np.asarray(y_true)
    probs = np.asarray(probs)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        left, right = bins[i], bins[i + 1]

        if i == n_bins - 1:
            mask = (probs >= left) & (probs <= right)
        else:
            mask = (probs >= left) & (probs < right)

        if mask.sum() == 0:
            continue

        bin_acc = y_true[mask].mean()
        bin_conf = probs[mask].mean()

        ece += (mask.sum() / len(y_true)) * abs(bin_acc - bin_conf)

    return ece


def summarize_calibration(calib_data, prob_key="test_probs"):
    rows = []

    for pack in calib_data:
        y_true = pack["y_test"]
        probs = pack[prob_key]

        rows.append({
            "fold": pack["fold"],
            "auroc": roc_auc_score(y_true, probs),
            "logloss": log_loss(y_true, probs),
            "brier": brier_score_loss(y_true, probs),
            "ece_10": expected_calibration_error(y_true, probs, n_bins=10),
            "prob_mean": probs.mean(),
            "prob_std": probs.std(),
        })

    return pd.DataFrame(rows)


base_calib_df = summarize_calibration(calib_data, prob_key="test_probs")

display(base_calib_df)

print("BASE:")
print(base_calib_df[["auroc", "logloss", "brier", "ece_10"]].mean())
print(base_calib_df[["auroc", "logloss", "brier", "ece_10"]].std())

,fold,auroc,logloss,brier,ece_10,prob_mean,prob_std
0,1,0.897913,0.360501,0.117616,0.066106,0.662235,0.334830
1,2,0.917274,0.347033,0.104622,0.077181,0.671181,0.295830
2,3,0.894644,0.369773,0.115119,0.050705,0.733163,0.297210
3,4,0.910962,0.336241,0.106142,0.080417,0.674666,0.335789
4,5,0.870681,0.415521,0.130382,0.091728,0.717475,0.321187


BASE:
auroc      0.898295
logloss    0.365814
brier      0.114776
ece_10     0.073227
dtype: float64
auroc      0.018005
logloss    0.030586
brier      0.010361
ece_10     0.015553
dtype: float64


In [223]:
def apply_platt_scaling(calib_data, C=1.0):
    calibrated_rows = []

    for pack in calib_data:
        y_val = pack["y_val"]
        val_probs = clip_probs(pack["val_probs"])
        test_probs = clip_probs(pack["test_probs"])

        X_val_meta = logit(val_probs).reshape(-1, 1)
        X_test_meta = logit(test_probs).reshape(-1, 1)

        calibrator = LogisticRegression(
            C=C,
            solver="lbfgs",
            max_iter=5000,
        )

        calibrator.fit(X_val_meta, y_val)

        test_calib = calibrator.predict_proba(X_test_meta)[:, 1]

        calibrated_rows.append({
            **pack,
            "test_probs_calibrated": clip_probs(test_calib),
        })

    return calibrated_rows


def apply_isotonic_calibration(calib_data):
    calibrated_rows = []

    for pack in calib_data:
        y_val = pack["y_val"]
        val_probs = clip_probs(pack["val_probs"])
        test_probs = clip_probs(pack["test_probs"])

        calibrator = IsotonicRegression(
            out_of_bounds="clip",
        )

        calibrator.fit(val_probs, y_val)

        test_calib = calibrator.predict(test_probs)

        calibrated_rows.append({
            **pack,
            "test_probs_calibrated": clip_probs(test_calib),
        })

    return calibrated_rows

In [ ]:
calibration_results = []

# baseline
base_df = summarize_calibration(calib_data, prob_key="test_probs")
calibration_results.append({
    "method": "base_uncalibrated",
    "auroc_mean": base_df["auroc"].mean(),
    "auroc_std": base_df["auroc"].std(),
    "logloss_mean": base_df["logloss"].mean(),
    "brier_mean": base_df["brier"].mean(),
    "ece10_mean": base_df["ece_10"].mean(),
})

# platt with different C
for C in [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]:
    platt_data = apply_platt_scaling(calib_data, C=C)
    platt_df = summarize_calibration(
        platt_data,
        prob_key="test_probs_calibrated",
    )

    calibration_results.append({
        "method": f"platt_C{C}",
        "auroc_mean": platt_df["auroc"].mean(),
        "auroc_std": platt_df["auroc"].std(),
        "logloss_mean": platt_df["logloss"].mean(),
        "brier_mean": platt_df["brier"].mean(),
        "ece10_mean": platt_df["ece_10"].mean(),
    })

# isotonic
iso_data = apply_isotonic_calibration(calib_data)
iso_df = summarize_calibration(
    iso_data,
    prob_key="test_probs_calibrated",
)

calibration_results.append({
    "method": "isotonic",
    "auroc_mean": iso_df["auroc"].mean(),
    "auroc_std": iso_df["auroc"].std(),
    "logloss_mean": iso_df["logloss"].mean(),
    "brier_mean": iso_df["brier"].mean(),
    "ece10_mean": iso_df["ece_10"].mean(),
})

calibration_df = pd.DataFrame(calibration_results).sort_values(
    "auroc_mean",
    ascending=False,
)

display(calibration_df)

,method,auroc_mean,auroc_std,logloss_mean,brier_mean,ece10_mean
0,base_uncalibrated,0.898295,0.018005,0.365814,0.114776,0.073227
1,platt_C0.01,0.898295,0.018005,0.461806,0.147221,0.160015
2,platt_C0.03,0.898295,0.018005,0.405129,0.126400,0.107380
3,platt_C0.1,0.898295,0.018005,0.373456,0.116662,0.075522
4,platt_C0.3,0.898295,0.018005,0.364496,0.114118,0.055407
5,platt_C1.0,0.898295,0.018005,0.362890,0.113478,0.054140
6,platt_C3.0,0.898295,0.018005,0.362938,0.113355,0.057055
7,platt_C10.0,0.898295,0.018005,0.363054,0.113323,0.058659
8,isotonic,0.886504,0.028670,0.538389,0.115754,0.064669


In [225]:
best = calibration_df.iloc[0]

print(
    f"BEST CALIBRATION RESULT:\n"
    f"{best['method']}\n"
    f"AUROC={best['auroc_mean']:.4f} ± {best['auroc_std']:.4f}\n"
    f"logloss={best['logloss_mean']:.4f}\n"
    f"brier={best['brier_mean']:.4f}\n"
    f"ece10={best['ece10_mean']:.4f}"
)

print("\nPrevious best AUROC: 0.8983 ± 0.0161")

BEST CALIBRATION RESULT:
base_uncalibrated
AUROC=0.8983 ± 0.0180
logloss=0.3658
brier=0.1148
ece10=0.0732

Previous best AUROC: 0.8983 ± 0.0161


## Another tuning - catboost + RFE - final

In [227]:
import optuna
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


BEST_BASE_PARAMS = {
    "iterations": 400,
    "depth": 4,
    "learning_rate": 0.05154666190467915,
    "l2_leaf_reg": 4.668730683946981,
    "random_strength": 7.525362919817402,
    "bagging_temperature": 1.0442924028703902,
    "border_count": 124,
    "auto_class_weights": "SqrtBalanced",
}

RFE_CONFIGS = {
    "70": (200, 125, 70),
    "65": (200, 125, 65),
    "75": (200, 125, 75),

    # nearby variants
    "60": (200, 125, 60),
    "80": (200, 125, 80),
    "70_alt": (190, 125, 70),
    "65_alt": (200, 115, 65),
    "75_alt": (210, 125, 75),
}

In [228]:
rfe_index_cache = {}

for name, steps in RFE_CONFIGS.items():
    print(f"Building RFE {name}: {steps}")

    rfe_index_cache[name] = recursive_elimination_return_idx(
        X=X_final,
        y=y_final,
        splits=splits,
        initial_idx=counts_base.head(250).index.to_numpy(dtype=int),
        steps=steps,
    )

print({k: len(v) for k, v in rfe_index_cache.items()})

Building RFE 70: (200, 125, 70)
RFE target_k=200, current=250
RFE target_k=125, current=200
RFE target_k=70, current=125
Building RFE 65: (200, 125, 65)
RFE target_k=200, current=250
RFE target_k=125, current=200
RFE target_k=65, current=125
Building RFE 75: (200, 125, 75)
RFE target_k=200, current=250
RFE target_k=125, current=200
RFE target_k=75, current=125
Building RFE 60: (200, 125, 60)
RFE target_k=200, current=250
RFE target_k=125, current=200
RFE target_k=60, current=125
Building RFE 80: (200, 125, 80)
RFE target_k=200, current=250
RFE target_k=125, current=200
RFE target_k=80, current=125
Building RFE 70_alt: (190, 125, 70)
RFE target_k=190, current=250
RFE target_k=125, current=190
RFE target_k=70, current=125
Building RFE 65_alt: (200, 115, 65)
RFE target_k=200, current=250
RFE target_k=115, current=200
RFE target_k=65, current=115
Building RFE 75_alt: (210, 125, 75)
RFE target_k=210, current=250
RFE target_k=125, current=210
RFE target_k=75, current=125
{'70': 70, '65': 65,

In [229]:
def evaluate_joint_rfe_catboost(
    params,
    selector_names,
    weights,
    mode="proba",
):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()

    fold_aucs = []

    for idx_train, idx_val, idx_test in splits:
        fold_preds = []

        for selector_name in selector_names:
            selected_idx = rfe_index_cache[selector_name]

            model = make_catboost_from_params(params)

            model.fit(
                X_final[idx_train][:, selected_idx],
                y_final[idx_train],
                eval_set=(X_final[idx_val][:, selected_idx], y_final[idx_val]),
                use_best_model=True,
            )

            probs = model.predict_proba(
                X_final[idx_test][:, selected_idx]
            )[:, 1]

            fold_preds.append(probs)

        if mode == "proba":
            ens = np.zeros_like(fold_preds[0], dtype=float)

            for w, p in zip(weights, fold_preds):
                ens += w * p

        else:
            ens_logit = np.zeros_like(fold_preds[0], dtype=float)

            for w, p in zip(weights, fold_preds):
                ens_logit += w * safe_logit_np(p)

            ens = expit(ens_logit)

        fold_aucs.append(roc_auc_score(y_final[idx_test], ens))

    return float(np.mean(fold_aucs)), float(np.std(fold_aucs))

In [ ]:
def objective_joint_final(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 300, 700, step=100),
        "depth": trial.suggest_int("depth", 3, 5),
        "learning_rate": trial.suggest_float("learning_rate", 0.025, 0.08, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 12.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 3.0, 15.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.3, 2.5),
        "border_count": trial.suggest_int("border_count", 80, 180),
        "auto_class_weights": trial.suggest_categorical(
            "auto_class_weights",
            ["SqrtBalanced", "Balanced", None],
        ),
    }

    # fixed base selector + optional nearby variants
    selector_names = [
        trial.suggest_categorical("selector_1", ["70"]),
        trial.suggest_categorical("selector_2", ["65", "60", "65_alt"]),
        trial.suggest_categorical("selector_3", ["75", "80", "75_alt"]),
    ]

    w1 = trial.suggest_float("w1", 0.70, 0.95)
    w2 = trial.suggest_float("w2", 0.02, 0.20)
    w3 = trial.suggest_float("w3", 0.02, 0.20)

    mode = trial.suggest_categorical("mode", ["proba", "logit"])

    mean_auc, std_auc = evaluate_joint_rfe_catboost(
        params=params,
        selector_names=selector_names,
        weights=(w1, w2, w3),
        mode=mode,
    )

    trial.set_user_attr("mean_auc", mean_auc)
    trial.set_user_attr("std_auc", std_auc)
    trial.set_user_attr("selector_names", selector_names)

    # unstability fee
    return mean_auc - 0.10 * std_auc

In [231]:
study_final = optuna.create_study(direction="maximize")

study_final.optimize(
    objective_joint_final,
    n_trials=100,
)

print("Best objective:", study_final.best_value)
print("Best params:")
print(study_final.best_params)

print("Best attrs:")
print(study_final.best_trial.user_attrs)

[I 2026-05-12 18:13:06,293] A new study created in memory with name: no-name-28044e92-fad5-4296-9718-91d2f7290884
[I 2026-05-12 18:13:17,436] Trial 0 finished with value: 0.8873128959972726 and parameters: {'iterations': 700, 'depth': 3, 'learning_rate': 0.03618250739090823, 'l2_leaf_reg': 2.8957026190481248, 'random_strength': 6.577000434354309, 'bagging_temperature': 2.3425282377693244, 'border_count': 112, 'auto_class_weights': 'SqrtBalanced', 'selector_1': '70', 'selector_2': '65_alt', 'selector_3': '75_alt', 'w1': 0.9054959642108735, 'w2': 0.05910832226728691, 'w3': 0.07725310790855329, 'mode': 'logit'}. Best is trial 0 with value: 0.8873128959972726.
[I 2026-05-12 18:13:30,483] Trial 1 finished with value: 0.8894659278859389 and parameters: {'iterations': 700, 'depth': 4, 'learning_rate': 0.02577971307304123, 'l2_leaf_reg': 5.391902155414084, 'random_strength': 11.544490724361635, 'bagging_temperature': 1.299399870276108, 'border_count': 108, 'auto_class_weights': None, 'selector

Best objective: 0.9022454196100889
Best params:
{'iterations': 600, 'depth': 4, 'learning_rate': 0.05483764603487135, 'l2_leaf_reg': 9.313557294860134, 'random_strength': 5.561121818293995, 'bagging_temperature': 1.33359689297365, 'border_count': 156, 'auto_class_weights': 'Balanced', 'selector_1': '70', 'selector_2': '60', 'selector_3': '80', 'w1': 0.8354590676081969, 'w2': 0.1255100571342469, 'w3': 0.1312176699914651, 'mode': 'proba'}
Best attrs:
{'mean_auc': 0.9032050984629876, 'std_auc': 0.009596788528987068, 'selector_names': ['70', '60', '80']}


In [232]:
bp = study_final.best_params

best_final_params = {
    "iterations": bp["iterations"],
    "depth": bp["depth"],
    "learning_rate": bp["learning_rate"],
    "l2_leaf_reg": bp["l2_leaf_reg"],
    "random_strength": bp["random_strength"],
    "bagging_temperature": bp["bagging_temperature"],
    "border_count": bp["border_count"],
    "auto_class_weights": bp["auto_class_weights"],
}

best_selector_names = [
    bp["selector_1"],
    bp["selector_2"],
    bp["selector_3"],
]

best_weights = (
    bp["w1"],
    bp["w2"],
    bp["w3"],
)

best_mode = bp["mode"]

final_mean_auc, final_std_auc = evaluate_joint_rfe_catboost(
    params=best_final_params,
    selector_names=best_selector_names,
    weights=best_weights,
    mode=best_mode,
)

final_compare = pd.DataFrame([
    {
        "model": "previous_best",
        "selectors": "70 + 65 + 75",
        "weights": (0.85, 0.10, 0.05),
        "mode": "proba",
        "test_auroc_mean": 0.8983,
        "test_auroc_std": 0.0161,
        "params": BEST_BASE_PARAMS,
    },
    {
        "model": "joint_optuna_final",
        "selectors": " + ".join(best_selector_names),
        "weights": best_weights,
        "mode": best_mode,
        "test_auroc_mean": final_mean_auc,
        "test_auroc_std": final_std_auc,
        "params": best_final_params,
    },
]).sort_values("test_auroc_mean", ascending=False)

display(final_compare)

print(
    f"FINAL BEST:\n"
    f"selectors={best_selector_names}\n"
    f"weights={best_weights}\n"
    f"mode={best_mode}\n"
    f"AUROC={final_mean_auc:.4f} ± {final_std_auc:.4f}"
)

,model,selectors,weights,mode,test_auroc_mean,test_auroc_std,params
1,joint_optuna_final,70 + 60 + 80,"(0.8354590676081969, 0.1255100571342469, 0.131...",proba,0.903205,0.009597,"{'iterations': 600, 'depth': 4, 'learning_rate..."
0,previous_best,70 + 65 + 75,"(0.85, 0.1, 0.05)",proba,0.898300,0.016100,"{'iterations': 400, 'depth': 4, 'learning_rate..."


FINAL BEST:
selectors=['70', '60', '80']
weights=(0.8354590676081969, 0.1255100571342469, 0.1312176699914651)
mode=proba
AUROC=0.9032 ± 0.0096


## More tuning

In [233]:
def objective_final_local(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 500, 700, step=100),
        "depth": trial.suggest_categorical("depth", [4]),
        "learning_rate": trial.suggest_float("learning_rate", 0.045, 0.065),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 7.0, 13.0),
        "random_strength": trial.suggest_float("random_strength", 4.0, 8.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 1.0, 1.7),
        "border_count": trial.suggest_int("border_count", 130, 180),
        "auto_class_weights": trial.suggest_categorical(
            "auto_class_weights",
            ["Balanced", "SqrtBalanced"],
        ),
    }

    w1 = trial.suggest_float("w1", 0.78, 0.90)
    w2 = trial.suggest_float("w2", 0.08, 0.17)
    w3 = trial.suggest_float("w3", 0.08, 0.17)

    mean_auc, std_auc = evaluate_joint_rfe_catboost(
        params=params,
        selector_names=["70", "60", "80"],
        weights=(w1, w2, w3),
        mode="proba",
    )

    trial.set_user_attr("mean_auc", mean_auc)
    trial.set_user_attr("std_auc", std_auc)

    return mean_auc - 0.10 * std_auc

In [234]:
study_final_local = optuna.create_study(direction="maximize")

study_final_local.optimize(
    objective_final_local,
    n_trials=40,
)

print("Best objective:", study_final_local.best_value)
print("Best params:")
print(study_final_local.best_params)
print("Best attrs:")
print(study_final_local.best_trial.user_attrs)

[I 2026-05-12 18:41:26,472] A new study created in memory with name: no-name-892fb408-c48b-47e3-9149-9cd5abe4e424
[I 2026-05-12 18:41:41,322] Trial 0 finished with value: 0.8946668345194533 and parameters: {'iterations': 700, 'depth': 4, 'learning_rate': 0.05712107802397897, 'l2_leaf_reg': 11.887871007102671, 'random_strength': 5.784700675438601, 'bagging_temperature': 1.378991601278084, 'border_count': 133, 'auto_class_weights': 'SqrtBalanced', 'w1': 0.8140325666368149, 'w2': 0.15357257101053387, 'w3': 0.15786779677018445}. Best is trial 0 with value: 0.8946668345194533.
[I 2026-05-12 18:41:59,184] Trial 1 finished with value: 0.8939237562535745 and parameters: {'iterations': 600, 'depth': 4, 'learning_rate': 0.049501673646468736, 'l2_leaf_reg': 7.994761021626989, 'random_strength': 5.635767736439509, 'bagging_temperature': 1.045559049956403, 'border_count': 171, 'auto_class_weights': 'SqrtBalanced', 'w1': 0.7983558238779461, 'w2': 0.10318555189545658, 'w3': 0.11895974660943838}. Best

Best objective: 0.9027766371453133
Best params:
{'iterations': 600, 'depth': 4, 'learning_rate': 0.0628238389168676, 'l2_leaf_reg': 9.703703315819581, 'random_strength': 6.728629794179622, 'bagging_temperature': 1.3001972097295067, 'border_count': 161, 'auto_class_weights': 'Balanced', 'w1': 0.8326522912543401, 'w2': 0.13906799872596168, 'w3': 0.14228215132046573}
Best attrs:
{'mean_auc': 0.9036572519167754, 'std_auc': 0.008806147714621033}


In [236]:
bp = study_final_local.best_params

local_best_params = {
    "iterations": bp["iterations"],
    "depth": bp["depth"],
    "learning_rate": bp["learning_rate"],
    "l2_leaf_reg": bp["l2_leaf_reg"],
    "random_strength": bp["random_strength"],
    "bagging_temperature": bp["bagging_temperature"],
    "border_count": bp["border_count"],
    "auto_class_weights": bp["auto_class_weights"],
}

local_best_weights = (
    bp["w1"],
    bp["w2"],
    bp["w3"],
)

local_mean_auc, local_std_auc = evaluate_joint_rfe_catboost(
    params=local_best_params,
    selector_names=["70", "60", "80"],
    weights=local_best_weights,
    mode="proba",
)

local_compare = pd.DataFrame([
    {
        "model": "previous_joint_best",
        "selectors": "70 + 60 + 80",
        "weights": (0.8354590676081969, 0.1255100571342469, 0.1312176699914651),
        "test_auroc_mean": 0.9032,
        "test_auroc_std": 0.0096,
    },
    {
        "model": "local_optuna_best",
        "selectors": "70 + 60 + 80",
        "weights": local_best_weights,
        "test_auroc_mean": local_mean_auc,
        "test_auroc_std": local_std_auc,
    },
]).sort_values("test_auroc_mean", ascending=False)

display(local_compare)

print(
    f"LOCAL FINAL BEST:\n"
    f"weights={local_best_weights}\n"
    f"AUROC={local_mean_auc:.4f} ± {local_std_auc:.4f}\n"
    f"params={local_best_params}"
)

,model,selectors,weights,test_auroc_mean,test_auroc_std
1,local_optuna_best,70 + 60 + 80,"(0.8326522912543401, 0.13906799872596168, 0.14...",0.903657,0.008806
0,previous_joint_best,70 + 60 + 80,"(0.8354590676081969, 0.1255100571342469, 0.131...",0.903200,0.009600


LOCAL FINAL BEST:
weights=(0.8326522912543401, 0.13906799872596168, 0.14228215132046573)
AUROC=0.9037 ± 0.0088
params={'iterations': 600, 'depth': 4, 'learning_rate': 0.0628238389168676, 'l2_leaf_reg': 9.703703315819581, 'random_strength': 6.728629794179622, 'bagging_temperature': 1.3001972097295067, 'border_count': 161, 'auto_class_weights': 'Balanced'}
